In [ ]:
# ============================================================
# CELL 2 — GOOGLE DRIVE CONFIGURATION
# ============================================================

from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount("/content/drive")

# Google Drive root
DRIVE_ROOT = Path("/content/drive/MyDrive")

# Dataset and result folders
DATASET_DIR = DRIVE_ROOT / "DNA Adaptive DATASET"
RESULT_DIR = DRIVE_ROOT / "DNA Adaptive RESULT"

# Display paths
print("Google Drive mounted successfully.")
print(f"Drive Root  : {DRIVE_ROOT}")
print(f"Dataset Path: {DATASET_DIR}")
print(f"Result Path : {RESULT_DIR}")

# Check dataset folder
if DATASET_DIR.exists():
    print("Dataset folder found.")
else:
    print("WARNING: Dataset folder not found.")
    print(f"Please create/upload the dataset at:\n{DATASET_DIR}")

Mounted at /content/drive
Google Drive mounted successfully.
Drive Root  : /content/drive/MyDrive
Dataset Path: /content/drive/MyDrive/DNA Adaptive DATASET
Result Path : /content/drive/MyDrive/DNA Adaptive RESULT
Dataset folder found.


In [ ]:
# ============================================================
# CELL 3 — CREATE RESULT DIRECTORY STRUCTURE
# ============================================================

RESULT_SUBDIRS = [
    "01_INPUT",

    "02_KEYS",

    "03_CHAOTIC",

    "04_DNA",

    "05_PERMUTATION",

    "06_ENCRYPTION",
    "06_ENCRYPTION/cipher_images",
    "06_ENCRYPTION/encryption_arrays",

    "07_DECRYPTION",
    "07_DECRYPTION/decrypted_images",
    "07_DECRYPTION/decryption_arrays",

    "08_METRICS",

    "09_SECURITY",
    "09_SECURITY/histogram",
    "09_SECURITY/correlation",
    "09_SECURITY/npcr",
    "09_SECURITY/uaci",
    "09_SECURITY/key_sensitivity",
    "09_SECURITY/plaintext_sensitivity",
    "09_SECURITY/chosen_plaintext",
    "09_SECURITY/noise",

    "10_ABLATION",

    "11_SUMMARY",
]

# Create all directories
for subdir in RESULT_SUBDIRS:
    (RESULT_DIR / subdir).mkdir(parents=True, exist_ok=True)

print("Result directory structure created successfully.")
print(f"\nMain result directory:\n{RESULT_DIR}")

print("\nCreated folders:")
for subdir in RESULT_SUBDIRS:
    print(f"  ✓ {subdir}")

Result directory structure created successfully.

Main result directory:
/content/drive/MyDrive/DNA Adaptive RESULT

Created folders:
  ✓ 01_INPUT
  ✓ 02_KEYS
  ✓ 03_CHAOTIC
  ✓ 04_DNA
  ✓ 05_PERMUTATION
  ✓ 06_ENCRYPTION
  ✓ 06_ENCRYPTION/cipher_images
  ✓ 06_ENCRYPTION/encryption_arrays
  ✓ 07_DECRYPTION
  ✓ 07_DECRYPTION/decrypted_images
  ✓ 07_DECRYPTION/decryption_arrays
  ✓ 08_METRICS
  ✓ 09_SECURITY
  ✓ 09_SECURITY/histogram
  ✓ 09_SECURITY/correlation
  ✓ 09_SECURITY/npcr
  ✓ 09_SECURITY/uaci
  ✓ 09_SECURITY/key_sensitivity
  ✓ 09_SECURITY/plaintext_sensitivity
  ✓ 09_SECURITY/chosen_plaintext
  ✓ 09_SECURITY/noise
  ✓ 10_ABLATION
  ✓ 11_SUMMARY


In [ ]:
# ============================================================
# CELL 4 — IMPORTS
# ============================================================

import os
import sys
import time
import json
import hashlib
import platform
import random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from PIL import Image

import scipy
from scipy import stats

import psutil

print("Libraries imported successfully.")

print("\nEnvironment:")
print(f"Python       : {sys.version.split()[0]}")
print(f"NumPy        : {np.__version__}")
print(f"OpenCV       : {cv2.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"SciPy        : {scipy.__version__}")
print(f"Platform     : {platform.platform()}")

Libraries imported successfully.

Environment:
Python       : 3.13.15
NumPy        : 2.1.3
OpenCV       : 4.14.0
Pandas       : 2.2.3
SciPy        : 1.16.3
Platform     : Linux-6.6.122+-x86_64-with-glibc2.39


In [ ]:
# ============================================================
# CELL 5 — MASTER CONFIGURATION
# ============================================================

CONFIG = {

    # --------------------------------------------------------
    # PROJECT
    # --------------------------------------------------------
    "PROJECT_NAME": "Adaptive Plaintext-Dependent DNA Encryption",
    "METHOD_NAME": "State-Coupled Adaptive DNA Encryption",

    # --------------------------------------------------------
    # DATASET
    # --------------------------------------------------------
    "DATASET_NAME": "DNA Adaptive DATASET",
    "RESULT_NAME": "DNA Adaptive RESULT",

    "DATASET_DIR": str(DATASET_DIR),
    "RESULT_DIR": str(RESULT_DIR),

    # Process every valid image recursively
    "PROCESS_ALL_IMAGES": True,

    # --------------------------------------------------------
    # IMAGE
    # --------------------------------------------------------
    "IMAGE_MODE": "GRAYSCALE",
    "PIXEL_DEPTH": 8,

    # Expected experimental image size
    "EXPECTED_HEIGHT": 256,
    "EXPECTED_WIDTH": 256,

    # --------------------------------------------------------
    # BLOCK PROCESSING
    # --------------------------------------------------------
    "BLOCK_SIZE": 16,

    # --------------------------------------------------------
    # DNA
    # --------------------------------------------------------
    "DNA_SYMBOLS": ("A", "T", "C", "G"),
    "DNA_RULE_COUNT": 8,

    # --------------------------------------------------------
    # HASH
    # --------------------------------------------------------
    "HASH_ALGORITHM": "SHA-256",
    "HASH_BITS": 256,
    "HASH_BYTES": 32,

    # --------------------------------------------------------
    # CHAOTIC SYSTEM
    # --------------------------------------------------------
    "CHAOTIC_MAP": "LOGISTIC",

    # Logistic map:
    # x_(n+1) = r * x_n * (1 - x_n)
    "LOGISTIC_R_MIN": 3.900000,
    "LOGISTIC_R_MAX": 3.999999,

    # Number of transient iterations discarded
    "CHAOTIC_TRANSIENT": 1000,

    # --------------------------------------------------------
    # KEY / STATE
    # --------------------------------------------------------
    "KEY_LENGTH_BYTES": 32,

    # --------------------------------------------------------
    # DNA OPERATION
    # --------------------------------------------------------
    "DNA_OPERATION": "DNA-XOR",

    # --------------------------------------------------------
    # FEEDBACK
    # --------------------------------------------------------
    "FEEDBACK_ENABLED": True,
    "FEEDBACK_TYPE": "PREVIOUS_CIPHERTEXT",

    # State equation:
    #
    # S_(i+1) =
    # SHA256(S_i || C_i || H_i)
    #
    # where:
    # S_i = current state
    # C_i = current cipher block
    # H_i = plaintext block hash

    # --------------------------------------------------------
    # REPRODUCIBILITY
    # --------------------------------------------------------
    "RANDOM_SEED": 42,

    # --------------------------------------------------------
    # SECURITY TESTS
    # --------------------------------------------------------
    "RUN_ENTROPY": True,
    "RUN_HISTOGRAM": True,
    "RUN_CORRELATION": True,
    "RUN_NPCR": True,
    "RUN_UACI": True,
    "RUN_KEY_SENSITIVITY": True,
    "RUN_PLAINTEXT_SENSITIVITY": True,
    "RUN_CHOSEN_PLAINTEXT": True,
    "RUN_NOISE_ATTACK": True,
    "RUN_DATA_LOSS_ATTACK": True,

    # --------------------------------------------------------
    # ABLATION
    # --------------------------------------------------------
    "RUN_ABLATION": True,

    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------
    "SAVE_OUTPUTS": True,

    # --------------------------------------------------------
    # SECURITY TEST DATA
    # --------------------------------------------------------
    "SECURITY_TEST_IMAGES": "ALL",

    # --------------------------------------------------------
    # NUMERICAL
    # --------------------------------------------------------
    "FLOAT_DTYPE": "float64",
    "IMAGE_DTYPE": "uint8",
}

# ------------------------------------------------------------
# Set reproducible seeds
# ------------------------------------------------------------
SEED = CONFIG["RANDOM_SEED"]

random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------
print("=" * 70)
print("MASTER CONFIGURATION")
print("=" * 70)

for key, value in CONFIG.items():
    print(f"{key:30s}: {value}")

print("=" * 70)

# ------------------------------------------------------------
# Configuration validation
# ------------------------------------------------------------

assert CONFIG["IMAGE_MODE"] == "GRAYSCALE"
assert CONFIG["PIXEL_DEPTH"] == 8
assert CONFIG["BLOCK_SIZE"] > 0
assert CONFIG["DNA_RULE_COUNT"] == 8
assert CONFIG["HASH_BITS"] == 256
assert CONFIG["HASH_BYTES"] == 32
assert CONFIG["CHAOTIC_MAP"] == "LOGISTIC"
assert CONFIG["FEEDBACK_ENABLED"] is True
assert CONFIG["PROCESS_ALL_IMAGES"] is True

print("\nConfiguration validation: PASSED")

MASTER CONFIGURATION
PROJECT_NAME                  : Adaptive Plaintext-Dependent DNA Encryption
METHOD_NAME                   : State-Coupled Adaptive DNA Encryption
DATASET_NAME                  : DNA Adaptive DATASET
RESULT_NAME                   : DNA Adaptive RESULT
DATASET_DIR                   : /content/drive/MyDrive/DNA Adaptive DATASET
RESULT_DIR                    : /content/drive/MyDrive/DNA Adaptive RESULT
PROCESS_ALL_IMAGES            : True
IMAGE_MODE                    : GRAYSCALE
PIXEL_DEPTH                   : 8
EXPECTED_HEIGHT               : 256
EXPECTED_WIDTH                : 256
BLOCK_SIZE                    : 16
DNA_SYMBOLS                   : ('A', 'T', 'C', 'G')
DNA_RULE_COUNT                : 8
HASH_ALGORITHM                : SHA-256
HASH_BITS                     : 256
HASH_BYTES                    : 32
CHAOTIC_MAP                   : LOGISTIC
LOGISTIC_R_MIN                : 3.9
LOGISTIC_R_MAX                : 3.999999
CHAOTIC_TRANSIENT             : 1000
KEY_

In [ ]:
# ============================================================
# CELL 7 — DATASET VALIDATION
# ============================================================

SUPPORTED_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"
}

validation_records = []
valid_dataset_files = []

# Recursively scan the complete dataset
all_image_paths = sorted([
    p for p in DATASET_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
])

print("=" * 80)
print("DATASET VALIDATION")
print("=" * 80)
print(f"Images discovered: {len(all_image_paths)}")
print()

for index, image_path in enumerate(all_image_paths, start=1):

    record = {
        "Index": index,
        "Image": image_path.name,
        "Relative_Path": str(image_path.relative_to(DATASET_DIR)),
        "Path": str(image_path),
        "Readable": False,
        "Non_Empty": False,
        "Grayscale": False,
        "8_Bit": False,
        "Valid_Dimensions": False,
        "Height": None,
        "Width": None,
        "Channels": None,
        "Dtype": None,
        "Status": "INVALID",
        "Reason": ""
    }

    try:
        # Read image without forcing a color conversion
        image = cv2.imread(
            str(image_path),
            cv2.IMREAD_UNCHANGED
        )

        # ----------------------------------------------------
        # Readability
        # ----------------------------------------------------
        if image is None:
            record["Reason"] = "Image could not be read"
            validation_records.append(record)
            continue

        record["Readable"] = True

        # ----------------------------------------------------
        # Non-empty check
        # ----------------------------------------------------
        if image.size == 0:
            record["Reason"] = "Image is empty"
            validation_records.append(record)
            continue

        record["Non_Empty"] = True

        # ----------------------------------------------------
        # Dimensions and channels
        # ----------------------------------------------------
        if image.ndim == 2:
            height, width = image.shape
            channels = 1

        elif image.ndim == 3:
            height, width, channels = image.shape

        else:
            record["Reason"] = f"Unsupported number of dimensions: {image.ndim}"
            validation_records.append(record)
            continue

        record["Height"] = height
        record["Width"] = width
        record["Channels"] = channels
        record["Dtype"] = str(image.dtype)

        # ----------------------------------------------------
        # Grayscale validation
        # ----------------------------------------------------
        if image.ndim == 2:
            is_grayscale = True

        elif image.ndim == 3 and channels == 1:
            is_grayscale = True

        elif image.ndim == 3 and channels == 3:
            # Check whether a 3-channel image is actually grayscale
            is_grayscale = bool(
                np.array_equal(image[:, :, 0], image[:, :, 1])
                and
                np.array_equal(image[:, :, 1], image[:, :, 2])
            )

        else:
            is_grayscale = False

        record["Grayscale"] = is_grayscale

        # ----------------------------------------------------
        # 8-bit validation
        # ----------------------------------------------------
        is_8bit = image.dtype == np.uint8
        record["8_Bit"] = is_8bit

        # ----------------------------------------------------
        # Dimension validation
        # ----------------------------------------------------
        expected_h = CONFIG["EXPECTED_HEIGHT"]
        expected_w = CONFIG["EXPECTED_WIDTH"]

        valid_dimensions = (
            height == expected_h and
            width == expected_w
        )

        record["Valid_Dimensions"] = valid_dimensions

        # ----------------------------------------------------
        # Final validation
        # ----------------------------------------------------
        if not is_grayscale:
            record["Reason"] = "Image is not grayscale"

        elif not is_8bit:
            record["Reason"] = "Image is not 8-bit uint8"

        elif not valid_dimensions:
            record["Reason"] = (
                f"Expected {expected_h}x{expected_w}, "
                f"got {height}x{width}"
            )

        else:
            record["Status"] = "VALID"
            record["Reason"] = "All validation checks passed"
            valid_dataset_files.append(image_path)

    except Exception as e:
        record["Reason"] = f"Validation error: {str(e)}"

    validation_records.append(record)

# Convert validation information to DataFrame
validation_df = pd.DataFrame(validation_records)

# Save validation report
validation_report_path = (
    RESULT_DIR / "01_INPUT" / "dataset_validation.csv"
)

validation_df.to_csv(
    validation_report_path,
    index=False
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

valid_count = len(valid_dataset_files)
invalid_count = len(all_image_paths) - valid_count

print(f"Valid images  : {valid_count}")
print(f"Invalid images: {invalid_count}")
print()

if invalid_count > 0:
    print("Invalid images:")
    print(
        validation_df[
            validation_df["Status"] == "INVALID"
        ][
            ["Image", "Relative_Path", "Reason"]
        ].to_string(index=False)
    )
else:
    print("All discovered images passed validation.")

print()
print(f"Validation report saved to:")
print(validation_report_path)

# ------------------------------------------------------------
# Strict validation for the proposed experiment
# ------------------------------------------------------------

if valid_count == 0:
    raise RuntimeError(
        "No valid 256x256 grayscale 8-bit images were found."
    )

print()
print("=" * 80)
print("DATASET VALIDATION COMPLETED")
print("=" * 80)

DATASET VALIDATION
Images discovered: 49

Valid images  : 49
Invalid images: 0

All discovered images passed validation.

Validation report saved to:
/content/drive/MyDrive/DNA Adaptive RESULT/01_INPUT/dataset_validation.csv

DATASET VALIDATION COMPLETED


In [ ]:
# ============================================================
# CELL 8 — INPUT METADATA
# ============================================================

metadata_records = []

for image_path in valid_dataset_files:

    image = cv2.imread(
        str(image_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        continue

    height, width = image.shape

    metadata_records.append({
        "Image": image_path.name,
        "Relative_Path": str(image_path.relative_to(DATASET_DIR)),
        "Height": int(height),
        "Width": int(width),
        "Channels": 1,
        "Dtype": str(image.dtype),
        "Min": int(np.min(image)),
        "Max": int(np.max(image)),
        "Mean": float(np.mean(image)),
        "Std": float(np.std(image))
    })

input_metadata_df = pd.DataFrame(metadata_records)

# Save metadata
input_metadata_path = (
    RESULT_DIR / "01_INPUT" / "input_metadata.csv"
)

input_metadata_df.to_csv(
    input_metadata_path,
    index=False
)

print("=" * 80)
print("INPUT METADATA")
print("=" * 80)

print(f"Total valid images : {len(input_metadata_df)}")
print(f"Metadata file      : {input_metadata_path}")
print()

display(input_metadata_df)

print()
print("Dataset dimensions:")
print(
    input_metadata_df[
        ["Height", "Width"]
    ].drop_duplicates().to_string(index=False)
)

INPUT METADATA
Total valid images : 49
Metadata file      : /content/drive/MyDrive/DNA Adaptive RESULT/01_INPUT/input_metadata.csv



,Image,Relative_Path,Height,Width,Channels,Dtype,Min,Max,Mean,Std
0,Copy of Copy of image_0050.png,Copy of Copy of image_0050.png,256,256,1,uint8,5,255,89.383728,41.439759
1,Copy of Copy of image_0051.png,Copy of Copy of image_0051.png,256,256,1,uint8,5,238,56.392181,63.220905
2,Copy of Copy of image_0052.png,Copy of Copy of image_0052.png,256,256,1,uint8,5,238,56.392181,63.220905
3,Copy of Copy of image_0053.png,Copy of Copy of image_0053.png,256,256,1,uint8,5,238,56.392181,63.220905
4,Copy of Copy of image_0054.png,Copy of Copy of image_0054.png,256,256,1,uint8,5,238,56.392181,63.220905
5,Copy of Copy of image_0055.png,Copy of Copy of image_0055.png,256,256,1,uint8,2,252,95.453430,47.342302
6,Copy of Copy of image_0056.png,Copy of Copy of image_0056.png,256,256,1,uint8,2,252,95.453430,47.342302
7,Copy of Copy of image_0057.png,Copy of Copy of image_0057.png,256,256,1,uint8,2,252,95.453430,47.342302
8,Copy of Copy of image_0058.png,Copy of Copy of image_0058.png,256,256,1,uint8,2,252,95.453430,47.342302
9,Copy of Copy of image_0059.png,Copy of Copy of image_0059.png,256,256,1,uint8,8,255,107.672195,49.434859



Dataset dimensions:
 Height  Width
    256    256


In [ ]:
# ============================================================
# CELL 9 — IMAGE LOADING
# ============================================================

def load_grayscale_image(image_path):
    """
    Load one image as an 8-bit grayscale uint8 NumPy array.

    Mathematical representation:
        P ∈ [0,255]^(H×W)

    For the current experiment:
        H = 256
        W = 256
        C = 1
        n = H × W = 65536
    """

    image = cv2.imread(
        str(image_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        raise ValueError(
            f"Unable to load image: {image_path}"
        )

    if image.dtype != np.uint8:
        raise TypeError(
            f"Image must be uint8, got {image.dtype}: {image_path}"
        )

    if image.ndim != 2:
        raise ValueError(
            f"Image must be 2-D grayscale, got shape {image.shape}: "
            f"{image_path}"
        )

    if image.size == 0:
        raise ValueError(
            f"Image is empty: {image_path}"
        )

    return image


# ------------------------------------------------------------
# Test the loader on the first valid image
# ------------------------------------------------------------

test_image_path = valid_dataset_files[0]
test_image = load_grayscale_image(test_image_path)

H, W = test_image.shape
C = 1
N = H * W

print("=" * 80)
print("IMAGE LOADING TEST")
print("=" * 80)

print(f"Image          : {test_image_path.name}")
print(f"Shape          : {test_image.shape}")
print(f"Height (H)     : {H}")
print(f"Width (W)      : {W}")
print(f"Channels (C)   : {C}")
print(f"Dtype          : {test_image.dtype}")
print(f"Minimum pixel  : {test_image.min()}")
print(f"Maximum pixel  : {test_image.max()}")
print(f"Pixel count N  : {N}")

assert test_image.dtype == np.uint8
assert test_image.ndim == 2
assert test_image.size == H * W

print()
print("Image loader test: PASSED")

IMAGE LOADING TEST
Image          : Copy of Copy of image_0050.png
Shape          : (256, 256)
Height (H)     : 256
Width (W)      : 256
Channels (C)   : 1
Dtype          : uint8
Minimum pixel  : 5
Maximum pixel  : 255
Pixel count N  : 65536

Image loader test: PASSED


In [ ]:
# ============================================================
# CELL 10 — SHA-256 PLAINTEXT HASH
# ============================================================

def sha256_bytes(data):
    """
    Compute SHA-256 digest.

    Input:
        bytes

    Output:
        32-byte SHA-256 digest
        = 256 bits
    """

    return hashlib.sha256(data).digest()


def sha256_image(image):
    """
    Compute SHA-256 hash of the raw uint8 image bytes.

    Mathematical form:

        H0 = SHA256(P)

    where P is the byte representation of the plaintext image.

    Output:
        32-byte digest
    """

    if not isinstance(image, np.ndarray):
        raise TypeError("Image must be a NumPy array.")

    if image.dtype != np.uint8:
        raise TypeError("Image must have dtype uint8.")

    # Ensure deterministic contiguous byte representation
    image_bytes = np.ascontiguousarray(image).tobytes()

    return sha256_bytes(image_bytes)


def digest_to_hex(digest):
    """
    Convert a SHA-256 digest to hexadecimal representation.
    """

    return digest.hex()


def digest_to_bits(digest):
    """
    Convert a 32-byte digest into a 256-bit NumPy array.
    """

    return np.unpackbits(
        np.frombuffer(digest, dtype=np.uint8)
    )


# ------------------------------------------------------------
# Test SHA-256 on first image
# ------------------------------------------------------------

H0 = sha256_image(test_image)

H0_bits = digest_to_bits(H0)
H0_hex = digest_to_hex(H0)

print("=" * 80)
print("SHA-256 PLAINTEXT HASH")
print("=" * 80)

print(f"Image           : {test_image_path.name}")
print(f"Hash algorithm  : {CONFIG['HASH_ALGORITHM']}")
print(f"Hash size       : {len(H0) * 8} bits")
print(f"Hash bytes      : {len(H0)}")
print()
print(f"H0 (hex):")
print(H0_hex)
print()
print(f"H0 bit length: {len(H0_bits)}")

assert len(H0) == 32
assert len(H0_bits) == 256

print()
print("SHA-256 hash test: PASSED")

SHA-256 PLAINTEXT HASH
Image           : Copy of Copy of image_0050.png
Hash algorithm  : SHA-256
Hash size       : 256 bits
Hash bytes      : 32

H0 (hex):
33c6e035a6955e9290978b7f5cead80c002e62ba76f889ad15dd728d247e3d67

H0 bit length: 256

SHA-256 hash test: PASSED


In [ ]:
# ============================================================
# CELL 11 — CRYPTOGRAPHIC KEY SCHEDULE + CONTENT DESCRIPTOR
# ============================================================

import hashlib
import hmac
import json
import struct
import secrets


# ------------------------------------------------------------
# EXPERIMENT CONFIGURATION
# ------------------------------------------------------------

CONFIG["METHOD_NAME"] = (
    "Content-Adaptive DNA Encryption "
    "with Cryptographic Key Scheduling"
)

CONFIG["VERSION"] = "DNA-ADAPTIVE-V2"

CONFIG["KDF_ITERATIONS"] = 100000

CONFIG["SALT_BYTES"] = 16

CONFIG["NONCE_BYTES"] = 16

CONFIG["AUTH_TAG_BYTES"] = 32


# ------------------------------------------------------------
# SECRET KEY CONVERSION
# ------------------------------------------------------------

def secret_key_to_bytes(secret_key):

    if isinstance(secret_key, str):

        return secret_key.encode("utf-8")

    if isinstance(secret_key, bytes):

        return secret_key

    raise TypeError(
        "secret_key must be str or bytes."
    )


# ------------------------------------------------------------
# HKDF-EXTRACT
# ------------------------------------------------------------

def hkdf_extract(
    salt,
    input_key_material
):

    return hmac.new(
        salt,
        input_key_material,
        hashlib.sha256
    ).digest()


# ------------------------------------------------------------
# HKDF-EXPAND
# ------------------------------------------------------------

def hkdf_expand(
    prk,
    info,
    length=32
):

    output = b""

    previous = b""

    counter = 1

    while len(output) < length:

        previous = hmac.new(
            prk,
            previous +
            info +
            bytes([counter]),
            hashlib.sha256
        ).digest()

        output += previous

        counter += 1

    return output[:length]


# ------------------------------------------------------------
# CONTENT DESCRIPTOR
# ------------------------------------------------------------

def extract_content_descriptor(
    image
):

    image = np.asarray(
        image,
        dtype=np.uint8
    )

    if image.ndim != 2:

        raise ValueError(
            "Content descriptor requires "
            "a grayscale 2-D image."
        )

    pixels = image.astype(
        np.float64
    )

    # --------------------------------------------------------
    # Mean
    # --------------------------------------------------------

    mean_value = float(
        np.mean(pixels)
    )

    # --------------------------------------------------------
    # Standard deviation
    # --------------------------------------------------------

    std_value = float(
        np.std(pixels)
    )

    # --------------------------------------------------------
    # Shannon entropy
    # --------------------------------------------------------

    histogram = np.bincount(
        image.reshape(-1),
        minlength=256
    ).astype(np.float64)

    probabilities = (
        histogram /
        histogram.sum()
    )

    probabilities = probabilities[
        probabilities > 0
    ]

    entropy_value = float(
        -np.sum(
            probabilities *
            np.log2(probabilities)
        )
    )

    # --------------------------------------------------------
    # Edge density
    # --------------------------------------------------------

    edges = cv2.Canny(
        image,
        50,
        150
    )

    edge_density = float(
        np.mean(
            edges > 0
        )
    )

    # --------------------------------------------------------
    # Quantize to deterministic uint16 values
    # --------------------------------------------------------

    mean_q = int(
        np.clip(
            round(
                mean_value * 256.0
            ),
            0,
            65535
        )
    )

    std_q = int(
        np.clip(
            round(
                std_value * 256.0
            ),
            0,
            65535
        )
    )

    entropy_q = int(
        np.clip(
            round(
                entropy_value * 4096.0
            ),
            0,
            65535
        )
    )

    edge_q = int(
        np.clip(
            round(
                edge_density * 65535.0
            ),
            0,
            65535
        )
    )

    descriptor = struct.pack(
        ">HHHH",
        mean_q,
        std_q,
        entropy_q,
        edge_q
    )

    descriptor_digest = hashlib.sha256(
        descriptor
    ).digest()

    feature_values = {

        "Mean":
            mean_value,

        "Std":
            std_value,

        "Entropy":
            entropy_value,

        "Edge_Density":
            edge_density
    }

    return (
        descriptor,
        descriptor_digest,
        feature_values
    )


# ------------------------------------------------------------
# CRYPTOGRAPHIC KEY SCHEDULE
# ------------------------------------------------------------

def derive_key_schedule(
    secret_key,
    salt,
    nonce,
    descriptor_digest
):

    secret_bytes = (
        secret_key_to_bytes(
            secret_key
        )
    )

    # --------------------------------------------------------
    # Password/passphrase -> root key
    # --------------------------------------------------------

    root_key = hashlib.pbkdf2_hmac(
        "sha256",
        secret_bytes,
        salt,
        int(
            CONFIG["KDF_ITERATIONS"]
        ),
        dklen=32
    )

    # --------------------------------------------------------
    # HKDF extract
    # --------------------------------------------------------

    prk = hkdf_extract(
        nonce,
        root_key
    )

    context = (
        b"DNA-ADAPTIVE-V2|" +
        descriptor_digest
    )

    # --------------------------------------------------------
    # Independent subkeys
    # --------------------------------------------------------

    keys = {

        "K_DNA":
            hkdf_expand(
                prk,
                context + b"|DNA",
                32
            ),

        "K_PERM":
            hkdf_expand(
                prk,
                context + b"|PERMUTATION",
                32
            ),

        "K_DIFF":
            hkdf_expand(
                prk,
                context + b"|DIFFUSION",
                32
            ),

        "K_CHAOS":
            hkdf_expand(
                prk,
                context + b"|CHAOS",
                32
            ),

        "K_AUTH":
            hkdf_expand(
                prk,
                context + b"|AUTHENTICATION",
                32
            )
    }

    return keys


# ------------------------------------------------------------
# CANONICAL HEADER
# ------------------------------------------------------------

def canonical_header_bytes(
    metadata
):

    metadata_without_tag = {
        key: value
        for key, value in metadata.items()
        if key != "authentication_tag"
    }

    return json.dumps(
        metadata_without_tag,
        sort_keys=True,
        separators=(",", ":")
    ).encode("utf-8")


print("=" * 80)
print("CELL 11 — CRYPTOGRAPHIC KEY SCHEDULE")
print("=" * 80)

print()
print("PBKDF2-HMAC-SHA256 : ENABLED")
print(
    f"KDF iterations     : "
    f"{CONFIG['KDF_ITERATIONS']}"
)

print(
    "HKDF subkeys       : "
    "K_DNA / K_PERM / K_DIFF / "
    "K_CHAOS / K_AUTH"
)

print(
    "Content descriptor : "
    "entropy + variance + mean + edge density"
)

print()
print("CELL 11 COMPLETE")

CELL 11 — CRYPTOGRAPHIC KEY SCHEDULE

PBKDF2-HMAC-SHA256 : ENABLED
KDF iterations     : 100000
HKDF subkeys       : K_DNA / K_PERM / K_DIFF / K_CHAOS / K_AUTH
Content descriptor : entropy + variance + mean + edge density

CELL 11 COMPLETE


In [ ]:
# ============================================================
# CELL 12 — STATE-TO-CHAOTIC-PARAMETER CONVERSION — V2
# ============================================================

"""
PURPOSE:
    Convert a 256-bit internal state into Logistic-map
    parameters.

UPDATED V2 ARCHITECTURE:

    The old global S0 is NOT used here.

    Instead, CELL 26 / CELL 28 generate a block-specific
    256-bit chaos state:

        chaos_state_i = HMAC-SHA256(
            K_CHAOS,
            state_i
            || block_index
            || feature_digest
            || b"CHAOS_STATE"
        )

    This 32-byte state is passed to:

        state_to_chaotic_parameters()

    The function returns:

        x0
        r
        u
        v

IMPORTANT:
    Do NOT use:

        S0

    in this cell.

    S0 was part of the old plaintext-dependent architecture.
"""


# ============================================================
# REQUIRED IMPORT
# ============================================================

import numpy as np


# ============================================================
# STATE → CHAOTIC PARAMETERS
# ============================================================

def state_to_chaotic_parameters(
    state
):
    """
    Derive Logistic-map parameters from a 256-bit state.

    Parameters
    ----------
    state : bytes
        Exactly 32 bytes.

    Returns
    -------
    x0 : float
        Logistic-map initial condition in (0, 1).

    r : float
        Logistic-map control parameter in [3.9, 3.999].

    u : int
        Unsigned 32-bit value from state[0:4].

    v : int
        Unsigned 32-bit value from state[4:8].

    Formula
    -------
        x0 = (u + 1) / (2^32 + 1)

        r  = 3.9 + 0.099 *
             v / (2^32 - 1)
    """


    # ========================================================
    # VALIDATE STATE
    # ========================================================

    if not isinstance(
        state,
        (bytes, bytearray, np.ndarray)
    ):

        raise TypeError(
            "state must be bytes, bytearray, "
            "or a NumPy uint8 array."
        )


    state = bytes(
        state
    )


    if len(state) != 32:

        raise ValueError(

            "State must contain exactly "
            "32 bytes (256 bits)."

        )


    # ========================================================
    # EXTRACT u
    # ========================================================

    u = int.from_bytes(

        state[0:4],

        byteorder="big",

        signed=False

    )


    # ========================================================
    # EXTRACT v
    # ========================================================

    v = int.from_bytes(

        state[4:8],

        byteorder="big",

        signed=False

    )


    # ========================================================
    # CALCULATE x0
    # ========================================================

    x0 = (

        (u + 1.0)

        /

        (2**32 + 1.0)

    )


    # ========================================================
    # CALCULATE LOGISTIC r
    # ========================================================

    r = (

        3.9

        +

        0.099

        *

        (
            v
            /
            (2**32 - 1.0)
        )

    )


    # ========================================================
    # NUMERICAL SAFETY
    # ========================================================

    x0 = min(

        max(

            x0,

            np.finfo(
                np.float64
            ).eps

        ),

        1.0
        -
        np.finfo(
            np.float64
        ).eps

    )


    r = min(

        max(

            r,

            3.9

        ),

        3.999999

    )


    return (

        x0,

        r,

        u,

        v

    )


# ============================================================
# STANDALONE FUNCTION VALIDATION
# ============================================================
#
# This is ONLY a function test.
#
# It does NOT create S0 and does NOT become the encryption
# state.
# ============================================================

_test_state_cell12 = bytes.fromhex(

    "00112233445566778899aabbccddeeff"
    "102132435465768798a9bacbdcedfe0f"

)


(
    test_x0,
    test_logistic_r,
    test_u,
    test_v
) = state_to_chaotic_parameters(

    _test_state_cell12

)


# ============================================================
# VALIDATE PARAMETERS
# ============================================================

assert (
    0.0
    <
    test_x0
    <
    1.0
)


assert (
    3.9
    <=
    test_logistic_r
    <
    4.0
)


assert isinstance(
    test_u,
    int
)


assert isinstance(
    test_v,
    int
)


# ============================================================
# DISPLAY
# ============================================================

print("=" * 80)

print(
    "CELL 12 — STATE-TO-CHAOTIC-PARAMETER CONVERSION — V2"
)

print("=" * 80)

print()

print(
    "Test state length : "
    f"{len(_test_state_cell12)} bytes"
)

print(
    f"u                 : {test_u}"
)

print(
    f"v                 : {test_v}"
)

print(
    f"x0                : {test_x0:.17f}"
)

print(
    f"Logistic r        : {test_logistic_r:.17f}"
)

print()

print(
    "Chaotic parameter generation: PASSED"
)

print()

print(
    "V2 architecture check:"
)

print(
    "Global S0 usage: REMOVED"
)

print(
    "Per-block chaos state: SUPPORTED"
)

print()

print(
    "CELL 12 COMPLETE"
)

CELL 12 — STATE-TO-CHAOTIC-PARAMETER CONVERSION — V2

Test state length : 32 bytes
u                 : 1122867
v                 : 1146447479
x0                : 0.00026143807911746
Logistic r        : 3.92642588234679435

Chaotic parameter generation: PASSED

V2 architecture check:
Global S0 usage: REMOVED
Per-block chaos state: SUPPORTED

CELL 12 COMPLETE


In [ ]:
# ============================================================
# CELL 13 — LIGHTWEIGHT LOGISTIC CHAOTIC SEQUENCE — V2
# ============================================================

"""
PURPOSE:
    Validate the Logistic-map generator independently.

V2 ARCHITECTURE:
    - No global S0
    - No global x0 / logistic_r
    - No dependency on test_image
    - Actual encryption will generate a separate chaotic
      sequence for each block using a block-specific state.
"""

def generate_logistic_sequence(
    x0,
    r,
    sequence_length,
    transient=1000
):
    """
    Generate Logistic-map sequence.

        x_(n+1) = r * x_n * (1 - x_n)

    The first 'transient' values are discarded.
    """

    if not (0.0 < x0 < 1.0):
        raise ValueError(
            "x0 must satisfy 0 < x0 < 1."
        )

    if not (3.9 <= r < 4.0):
        raise ValueError(
            "r must satisfy 3.9 <= r < 4.0."
        )

    if sequence_length <= 0:
        raise ValueError(
            "sequence_length must be positive."
        )

    if transient < 0:
        raise ValueError(
            "transient must be non-negative."
        )

    total_iterations = (
        transient + sequence_length
    )

    sequence = np.empty(
        total_iterations,
        dtype=np.float64
    )

    x = np.float64(x0)

    for i in range(total_iterations):

        # Logistic map
        x = (
            r *
            x *
            (1.0 - x)
        )

        # Numerical protection
        if x <= 0.0:
            x = np.finfo(np.float64).eps

        elif x >= 1.0:
            x = 1.0 - np.finfo(np.float64).eps

        sequence[i] = x

    # Remove transient values
    usable_sequence = sequence[transient:]

    return usable_sequence


# ------------------------------------------------------------
# Independent deterministic test state
# ------------------------------------------------------------

_test_state_cell13 = bytes.fromhex(
    "00112233445566778899aabbccddeeff"
    "102132435465768798a9bacbdcedfe0f"
)

_test_x0, _test_logistic_r, _test_u, _test_v = (
    state_to_chaotic_parameters(
        _test_state_cell13
    )
)


# ------------------------------------------------------------
# Test sequence length
# ------------------------------------------------------------

TEST_SEQUENCE_LENGTH = 256 * 256

chaotic_sequence = generate_logistic_sequence(
    x0=_test_x0,
    r=_test_logistic_r,
    sequence_length=TEST_SEQUENCE_LENGTH,
    transient=CONFIG["CHAOTIC_TRANSIENT"]
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 80)
print("CELL 13 — LIGHTWEIGHT LOGISTIC CHAOTIC SEQUENCE — V2")
print("=" * 80)

print(
    f"Test state length      : "
    f"{len(_test_state_cell13)} bytes"
)

print(
    f"x0                     : "
    f"{_test_x0:.17f}"
)

print(
    f"Logistic r             : "
    f"{_test_logistic_r:.17f}"
)

print(
    f"Required values        : "
    f"{TEST_SEQUENCE_LENGTH}"
)

print(
    f"Transient iterations   : "
    f"{CONFIG['CHAOTIC_TRANSIENT']}"
)

print(
    f"Generated values       : "
    f"{len(chaotic_sequence)}"
)

print()

print(
    f"Minimum value          : "
    f"{np.min(chaotic_sequence):.15f}"
)

print(
    f"Maximum value          : "
    f"{np.max(chaotic_sequence):.15f}"
)

print(
    f"Mean                   : "
    f"{np.mean(chaotic_sequence):.15f}"
)

print(
    f"Standard deviation     : "
    f"{np.std(chaotic_sequence):.15f}"
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(chaotic_sequence) == TEST_SEQUENCE_LENGTH

assert np.all(
    chaotic_sequence > 0.0
)

assert np.all(
    chaotic_sequence < 1.0
)

assert np.all(
    np.isfinite(chaotic_sequence)
)

print()
print("Chaotic sequence generation: PASSED")
print()
print("V2 architecture check:")
print("Global S0 dependency       : REMOVED")
print("Global x0 dependency      : REMOVED")
print("Global logistic_r dependency: REMOVED")
print("Independent test state    : PASSED")
print()
print("CELL 13 COMPLETE")

CELL 13 — LIGHTWEIGHT LOGISTIC CHAOTIC SEQUENCE — V2
Test state length      : 32 bytes
x0                     : 0.00026143807911746
Logistic r             : 3.92642588234679435
Required values        : 65536
Transient iterations   : 1000
Generated values       : 65536

Minimum value          : 0.070892434533436
Maximum value          : 0.981606470444606
Mean                   : 0.597041007767264
Standard deviation     : 0.297533825996882

Chaotic sequence generation: PASSED

V2 architecture check:
Global S0 dependency       : REMOVED
Global x0 dependency      : REMOVED
Global logistic_r dependency: REMOVED
Independent test state    : PASSED

CELL 13 COMPLETE


In [ ]:
# ============================================================
# CELL 14 — CHAOTIC SEQUENCE TO BYTE KEYSTREAM
# ============================================================

def chaotic_to_bytes(chaotic_sequence):
    """
    Convert chaotic floating-point values to uint8 bytes.

        k_i = floor(10^14 * x_i) mod 256

    where:
        0 < x_i < 1

    Returns:
        uint8 keystream
    """

    chaotic_sequence = np.asarray(
        chaotic_sequence,
        dtype=np.float64
    )

    if chaotic_sequence.size == 0:
        raise ValueError(
            "Chaotic sequence is empty."
        )

    if np.any(chaotic_sequence <= 0.0):
        raise ValueError(
            "Chaotic sequence contains values <= 0."
        )

    if np.any(chaotic_sequence >= 1.0):
        raise ValueError(
            "Chaotic sequence contains values >= 1."
        )

    chaotic_bytes = (
        np.floor(
            chaotic_sequence * 10**14
        ).astype(np.uint64) % 256
    ).astype(np.uint8)

    return chaotic_bytes


# Generate chaotic byte keystream
chaotic_bytes = chaotic_to_bytes(
    chaotic_sequence
)

print("=" * 80)
print("CHAOTIC SEQUENCE → BYTE KEYSTREAM")
print("=" * 80)

print(f"Chaotic values : {len(chaotic_sequence)}")
print(f"Key bytes      : {len(chaotic_bytes)}")
print(f"Dtype          : {chaotic_bytes.dtype}")
print()
print(
    "First 32 chaotic bytes:"
)
print(
    chaotic_bytes[:32]
)

# Basic repetition check
unique_values, counts = np.unique(
    chaotic_bytes,
    return_counts=True
)

max_frequency = int(np.max(counts))
unique_count = int(len(unique_values))

print()
print(f"Unique byte values : {unique_count}/256")
print(f"Maximum frequency  : {max_frequency}")

assert len(chaotic_bytes) == N
assert chaotic_bytes.dtype == np.uint8

print()
print("Chaotic byte conversion: PASSED")

CHAOTIC SEQUENCE → BYTE KEYSTREAM
Chaotic values : 65536
Key bytes      : 65536
Dtype          : uint8

First 32 chaotic bytes:
[ 70  12  87  17 183  54 254  20  32 117 225  25 231 152 242 188  14 139
 196 131  24  59  49 208 124   5 147 222  51 169 119 198]

Unique byte values : 256/256
Maximum frequency  : 299

Chaotic byte conversion: PASSED


In [ ]:
# ============================================================
# CELL 15 — DIVIDE IMAGE INTO 16×16 BLOCKS
# ============================================================

BLOCK_SIZE = CONFIG["BLOCK_SIZE"]

def divide_into_blocks(image, block_size=16):
    """
    Divide a grayscale image into non-overlapping square blocks.

    For the current experiment:

        Image = 256 × 256
        Block = 16 × 16

        Number of blocks per dimension:
            256 / 16 = 16

        Total blocks:
            16 × 16 = 256
    """

    if image.ndim != 2:
        raise ValueError(
            "Input image must be grayscale (2-D)."
        )

    H, W = image.shape

    if H % block_size != 0:
        raise ValueError(
            f"Height {H} is not divisible by "
            f"block size {block_size}."
        )

    if W % block_size != 0:
        raise ValueError(
            f"Width {W} is not divisible by "
            f"block size {block_size}."
        )

    blocks = []

    blocks_per_row = W // block_size
    blocks_per_col = H // block_size

    for row in range(blocks_per_col):

        for col in range(blocks_per_row):

            y0 = row * block_size
            y1 = y0 + block_size

            x0 = col * block_size
            x1 = x0 + block_size

            block = image[
                y0:y1,
                x0:x1
            ].copy()

            blocks.append({
                "index": len(blocks),
                "row": row,
                "col": col,
                "y0": y0,
                "y1": y1,
                "x0": x0,
                "x1": x1,
                "data": block
            })

    return blocks


# ------------------------------------------------------------
# Divide test image
# ------------------------------------------------------------

image_blocks = divide_into_blocks(
    test_image,
    block_size=BLOCK_SIZE
)

blocks_per_row = W // BLOCK_SIZE
blocks_per_col = H // BLOCK_SIZE
total_blocks = len(image_blocks)

print("=" * 80)
print("IMAGE BLOCK DIVISION")
print("=" * 80)

print(f"Image dimensions       : {H} × {W}")
print(f"Block dimensions       : {BLOCK_SIZE} × {BLOCK_SIZE}")
print(f"Blocks per row         : {blocks_per_row}")
print(f"Blocks per column      : {blocks_per_col}")
print(f"Total blocks           : {total_blocks}")
print()

print(
    "First block shape      :",
    image_blocks[0]["data"].shape
)

print(
    "Last block shape       :",
    image_blocks[-1]["data"].shape
)

assert blocks_per_row == 16
assert blocks_per_col == 16
assert total_blocks == 256

# Verify complete image reconstruction from blocks
reconstructed_test = np.zeros_like(
    test_image
)

for block_info in image_blocks:

    reconstructed_test[
        block_info["y0"]:block_info["y1"],
        block_info["x0"]:block_info["x1"]
    ] = block_info["data"]

assert np.array_equal(
    test_image,
    reconstructed_test
)

print()
print("Block division and reconstruction test: PASSED")

IMAGE BLOCK DIVISION
Image dimensions       : 256 × 256
Block dimensions       : 16 × 16
Blocks per row         : 16
Blocks per column      : 16
Total blocks           : 256

First block shape      : (16, 16)
Last block shape       : (16, 16)

Block division and reconstruction test: PASSED


In [ ]:
# ============================================================
# CELL 16 — DNA RULE TABLE
# ============================================================

DNA_RULES = {
    1: "ATCG",
    2: "ACGT",
    3: "TGCA",
    4: "TCGA",
    5: "CTAG",
    6: "CATG",
    7: "GTAC",
    8: "GATC"
}

# Validate that exactly eight rules are defined
assert len(DNA_RULES) == 8

# Validate each rule
for rule_id, rule in DNA_RULES.items():

    assert rule_id in range(1, 9)

    assert len(rule) == 4

    assert set(rule) == {
        "A", "T", "C", "G"
    }

# Reverse lookup:
# rule -> rule number
DNA_RULE_TO_ID = {
    rule: rule_id
    for rule_id, rule in DNA_RULES.items()
}

print("=" * 80)
print("DNA ENCODING RULE TABLE")
print("=" * 80)

for rule_id in sorted(DNA_RULES):
    print(
        f"R_{rule_id} = {DNA_RULES[rule_id]}"
    )

print()
print("DNA rule validation: PASSED")

DNA ENCODING RULE TABLE
R_1 = ATCG
R_2 = ACGT
R_3 = TGCA
R_4 = TCGA
R_5 = CTAG
R_6 = CATG
R_7 = GTAC
R_8 = GATC

DNA rule validation: PASSED


In [ ]:
# ============================================================
# CELL 17 — STATE-DEPENDENT BLOCK CONTROL STATE — V2
# ============================================================

"""
PURPOSE:
    Generate the adaptive control state for block i.

V2 ARCHITECTURE:
    Z_i = SHA256(S_i || i || ADAPTIVE_CONTROL)

    The plaintext block is NOT used.

    current_state is supplied by the encryption/decryption
    pipeline and is evolved deterministically from the
    ciphertext blocks.

    No global S0 is used.
"""


def compute_adaptive_state(
    current_state,
    block_index
):
    """
    Generate the adaptive control state for block i.

        Z_i = SHA256(S_i || i || ADAPTIVE_CONTROL)

    Parameters
    ----------
    current_state : bytes
        Current 256-bit internal state.

    block_index : int
        Zero-based block index.

    Returns
    -------
    bytes
        256-bit adaptive control state Z_i.
    """

    if not isinstance(
        current_state,
        (bytes, bytearray)
    ):
        raise TypeError(
            "current_state must be bytes or bytearray."
        )

    current_state = bytes(current_state)

    if len(current_state) != 32:
        raise ValueError(
            "current_state must contain exactly 32 bytes."
        )

    if not isinstance(
        block_index,
        (int, np.integer)
    ):
        raise TypeError(
            "block_index must be an integer."
        )

    if block_index < 0:
        raise ValueError(
            "block_index must be non-negative."
        )

    index_bytes = int(block_index).to_bytes(
        8,
        byteorder="big",
        signed=False
    )

    Z_i = hashlib.sha256(
        current_state
        + index_bytes
        + b"ADAPTIVE_CONTROL"
    ).digest()

    return Z_i


# ------------------------------------------------------------
# Deterministic standalone test
# ------------------------------------------------------------

_test_state_cell17 = bytes.fromhex(
    "00112233445566778899aabbccddeeff"
    "102132435465768798a9bacbdcedfe0f"
)

_test_Z_i = compute_adaptive_state(
    current_state=_test_state_cell17,
    block_index=0
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 80)
print("CELL 17 — STATE-DEPENDENT BLOCK CONTROL STATE — V2")
print("=" * 80)

print("Block index : 0")

print(
    "Formula     : "
    "SHA256(S_i || i || ADAPTIVE_CONTROL)"
)

print(
    f"State size  : "
    f"{len(_test_state_cell17)} bytes"
)

print(
    f"Z_i         : "
    f"{_test_Z_i.hex()}"
)

print(
    f"Z_i size    : "
    f"{len(_test_Z_i) * 8} bits"
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(_test_state_cell17) == 32

assert isinstance(
    _test_Z_i,
    bytes
)

assert len(_test_Z_i) == 32


# ------------------------------------------------------------
# Determinism test
# ------------------------------------------------------------

_test_Z_i_repeat = compute_adaptive_state(
    current_state=_test_state_cell17,
    block_index=0
)

assert _test_Z_i == _test_Z_i_repeat

print()
print("Adaptive control-state generation : PASSED")
print("Deterministic repeatability       : PASSED")

print()
print("V2 architecture check:")
print("Global S0 dependency             : REMOVED")
print("Plaintext dependency             : REMOVED")
print("Block-index dependency           : SUPPORTED")
print("256-bit adaptive state           : SUPPORTED")

print()
print("CELL 17 COMPLETE")

CELL 17 — STATE-DEPENDENT BLOCK CONTROL STATE — V2
Block index : 0
Formula     : SHA256(S_i || i || ADAPTIVE_CONTROL)
State size  : 32 bytes
Z_i         : 0001d5d74b60e6bff3feb202cc5630f38f546ea1a0ee93b0155fad59ccf28f24
Z_i size    : 256 bits

Adaptive control-state generation : PASSED
Deterministic repeatability       : PASSED

V2 architecture check:
Global S0 dependency             : REMOVED
Plaintext dependency             : REMOVED
Block-index dependency           : SUPPORTED
256-bit adaptive state           : SUPPORTED

CELL 17 COMPLETE


In [ ]:
# ============================================================
# CELL 18 — ADAPTIVE DNA-RULE SELECTION — V2
# ============================================================

"""
PURPOSE:
    Select one of the eight DNA encoding rules from the
    adaptive control state Z_i.

    q_i = uint32(Z_i[0:4])

    r_i = (q_i mod 8) + 1

    DNA Rule_i = R_(r_i)

V2 ARCHITECTURE:
    Z_i is supplied explicitly to this function.

    No global Z_i is used.

    In the actual encryption/decryption pipeline,
    Z_i is generated from the current block state.
"""


def select_adaptive_dna_rule(Z_i):
    """
    Select the DNA rule from the state-dependent
    adaptive control state.

    Parameters
    ----------
    Z_i : bytes
        256-bit adaptive control state.

    Returns
    -------
    q_i : int
        32-bit integer extracted from Z_i.

    r_i : int
        DNA rule index in the range 1..8.

    dna_rule
        Corresponding DNA encoding rule.
    """

    if not isinstance(
        Z_i,
        (bytes, bytearray)
    ):
        raise TypeError(
            "Z_i must be bytes or bytearray."
        )

    Z_i = bytes(Z_i)

    if len(Z_i) != 32:
        raise ValueError(
            "Z_i must contain exactly 32 bytes."
        )

    q_i = int.from_bytes(
        Z_i[0:4],
        byteorder="big",
        signed=False
    )

    r_i = (
        q_i % 8
    ) + 1

    if r_i not in DNA_RULES:
        raise KeyError(
            f"DNA rule R_{r_i} is not available in DNA_RULES."
        )

    dna_rule = DNA_RULES[r_i]

    return q_i, r_i, dna_rule


# ------------------------------------------------------------
# Deterministic standalone test
# ------------------------------------------------------------

_test_state_cell18 = bytes.fromhex(
    "00112233445566778899aabbccddeeff"
    "102132435465768798a9bacbdcedfe0f"
)

_test_Z_i_cell18 = compute_adaptive_state(
    current_state=_test_state_cell18,
    block_index=0
)


# ------------------------------------------------------------
# Select DNA rule
# ------------------------------------------------------------

_test_q_i, _test_r_i, _test_selected_rule = (
    select_adaptive_dna_rule(
        _test_Z_i_cell18
    )
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 80)
print("CELL 18 — ADAPTIVE DNA-RULE SELECTION — V2")
print("=" * 80)

print(
    f"Z_i         : "
    f"{_test_Z_i_cell18.hex()}"
)

print(
    f"q_i         : "
    f"{_test_q_i}"
)

print(
    f"Rule index  : "
    f"{_test_r_i}"
)

print(
    f"Selected rule : "
    f"R_{_test_r_i}"
)

print(
    f"DNA rule    : "
    f"{_test_selected_rule}"
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(_test_Z_i_cell18) == 32

assert isinstance(
    _test_q_i,
    int
)

assert 1 <= _test_r_i <= 8

assert _test_selected_rule == DNA_RULES[_test_r_i]


# ------------------------------------------------------------
# Determinism test
# ------------------------------------------------------------

_test_q_repeat, _test_r_repeat, _test_rule_repeat = (
    select_adaptive_dna_rule(
        _test_Z_i_cell18
    )
)

assert _test_q_i == _test_q_repeat
assert _test_r_i == _test_r_repeat
assert _test_selected_rule == _test_rule_repeat


print()
print("Adaptive DNA-rule selection : PASSED")
print("Deterministic repeatability : PASSED")

print()
print("V2 architecture check:")
print("Global Z_i dependency       : REMOVED")
print("Plaintext dependency       : REMOVED")
print("Eight DNA rules supported  : YES")

print()
print("CELL 18 COMPLETE")

CELL 18 — ADAPTIVE DNA-RULE SELECTION — V2
Z_i         : 0001d5d74b60e6bff3feb202cc5630f38f546ea1a0ee93b0155fad59ccf28f24
q_i         : 120279
Rule index  : 8
Selected rule : R_8
DNA rule    : GATC

Adaptive DNA-rule selection : PASSED
Deterministic repeatability : PASSED

V2 architecture check:
Global Z_i dependency       : REMOVED
Plaintext dependency       : REMOVED
Eight DNA rules supported  : YES

CELL 18 COMPLETE


In [ ]:
# ============================================================
# CELL 19 — STATE-DEPENDENT ADAPTIVE PERMUTATION — V2
# ============================================================

"""
PURPOSE:
    Generate a deterministic reversible permutation for
    each image block.

V2 ARCHITECTURE:

    The permutation does NOT depend on the plaintext block.

    The actual encryption pipeline derives a dedicated
    permutation state from K_PERM before calling this
    function.

    Therefore:

        permutation_state_i
            |
            v
        generate_block_permutation()
            |
            +---- permutation_i
            +---- inverse_permutation_i

    No global S0 is used.
"""


# ============================================================
# PERMUTATION GENERATION
# ============================================================

def generate_block_permutation(
    S_i,
    block_index,
    block_size=16
):
    """
    Generate a deterministic reversible permutation.

        P_i = SHA256(
                S_i ||
                block_index ||
                "PERMUTATION"
              )

    Parameters
    ----------
    S_i : bytes
        256-bit permutation state.

    block_index : int
        Zero-based block index.

    block_size : int
        Width/height of the square block.

    Returns
    -------
    permutation : numpy.ndarray
        Forward permutation.

    inverse_permutation : numpy.ndarray
        Inverse permutation.

    permutation_seed : int
        64-bit seed used by NumPy RNG.
    """

    # --------------------------------------------------------
    # Validate state
    # --------------------------------------------------------

    if not isinstance(
        S_i,
        (bytes, bytearray)
    ):
        raise TypeError(
            "S_i must be bytes or bytearray."
        )

    S_i = bytes(S_i)

    if len(S_i) != 32:
        raise ValueError(
            "S_i must contain exactly 32 bytes."
        )

    # --------------------------------------------------------
    # Validate block index
    # --------------------------------------------------------

    if not isinstance(
        block_index,
        (int, np.integer)
    ):
        raise TypeError(
            "block_index must be an integer."
        )

    if block_index < 0:
        raise ValueError(
            "block_index must be non-negative."
        )

    # --------------------------------------------------------
    # Validate block size
    # --------------------------------------------------------

    if not isinstance(
        block_size,
        (int, np.integer)
    ):
        raise TypeError(
            "block_size must be an integer."
        )

    if block_size <= 0:
        raise ValueError(
            "block_size must be positive."
        )

    number_of_positions = (
        int(block_size) *
        int(block_size)
    )

    # --------------------------------------------------------
    # Block index encoding
    # --------------------------------------------------------

    index_bytes = int(
        block_index
    ).to_bytes(
        8,
        byteorder="big",
        signed=False
    )

    # --------------------------------------------------------
    # Derive deterministic permutation digest
    # --------------------------------------------------------

    permutation_digest = hashlib.sha256(
        S_i +
        index_bytes +
        b"PERMUTATION"
    ).digest()

    # --------------------------------------------------------
    # Convert first 64 bits to RNG seed
    # --------------------------------------------------------

    permutation_seed = int.from_bytes(
        permutation_digest[:8],
        byteorder="big",
        signed=False
    )

    # --------------------------------------------------------
    # Generate permutation
    # --------------------------------------------------------

    rng = np.random.default_rng(
        permutation_seed
    )

    permutation = np.arange(
        number_of_positions,
        dtype=np.int64
    )

    rng.shuffle(
        permutation
    )

    # --------------------------------------------------------
    # Generate inverse permutation
    # --------------------------------------------------------

    inverse_permutation = np.empty_like(
        permutation
    )

    inverse_permutation[
        permutation
    ] = np.arange(
        number_of_positions,
        dtype=np.int64
    )

    return (
        permutation,
        inverse_permutation,
        permutation_seed
    )


# ============================================================
# FORWARD PERMUTATION
# ============================================================

def permute_block(
    block,
    permutation
):
    """
    Forward permutation.

        B'_i = Permute(B_i, pi_i)
    """

    block = np.asarray(
        block
    )

    flat_block = np.ascontiguousarray(
        block
    ).reshape(-1)

    permutation = np.asarray(
        permutation
    )

    if len(permutation) != len(flat_block):
        raise ValueError(
            "Permutation length does not match block."
        )

    return flat_block[
        permutation
    ].reshape(
        block.shape
    ).copy()


# ============================================================
# INVERSE PERMUTATION
# ============================================================

def inverse_permute_block(
    permuted_block,
    inverse_permutation
):
    """
    Inverse permutation.

        B_i = InversePermute(B'_i, pi_i)
    """

    permuted_block = np.asarray(
        permuted_block
    )

    flat_block = np.ascontiguousarray(
        permuted_block
    ).reshape(-1)

    inverse_permutation = np.asarray(
        inverse_permutation
    )

    if len(inverse_permutation) != len(flat_block):
        raise ValueError(
            "Inverse permutation length does not match block."
        )

    return flat_block[
        inverse_permutation
    ].reshape(
        permuted_block.shape
    ).copy()


# ============================================================
# STANDALONE DETERMINISTIC TEST
# ============================================================

"""
Do NOT use S0 here.

This is only a function-level test. The actual encryption
pipeline will provide its own K_PERM-derived permutation
state.
"""

_test_state_cell19 = bytes.fromhex(
    "00112233445566778899aabbccddeeff"
    "102132435465768798a9bacbdcedfe0f"
)


# ------------------------------------------------------------
# Use the existing notebook test image
# ------------------------------------------------------------

_test_block_cell19 = np.asarray(
    test_image[
        :BLOCK_SIZE,
        :BLOCK_SIZE
    ],
    dtype=np.uint8
).copy()


# ------------------------------------------------------------
# Generate permutation
# ------------------------------------------------------------

(
    _test_permutation_cell19,
    _test_inverse_permutation_cell19,
    _test_permutation_seed_cell19
) = generate_block_permutation(
    S_i=_test_state_cell19,
    block_index=0,
    block_size=BLOCK_SIZE
)


# ------------------------------------------------------------
# Forward permutation
# ------------------------------------------------------------

_test_permuted_block_cell19 = permute_block(
    _test_block_cell19,
    _test_permutation_cell19
)


# ------------------------------------------------------------
# Reverse permutation
# ------------------------------------------------------------

_test_recovered_block_cell19 = inverse_permute_block(
    _test_permuted_block_cell19,
    _test_inverse_permutation_cell19
)


# ============================================================
# VALIDATION
# ============================================================

print("=" * 80)
print("CELL 19 — STATE-DEPENDENT ADAPTIVE PERMUTATION — V2")
print("=" * 80)

print(
    f"Block size       : "
    f"{BLOCK_SIZE} × {BLOCK_SIZE}"
)

print(
    f"Positions        : "
    f"{BLOCK_SIZE * BLOCK_SIZE}"
)

print(
    f"Permutation seed : "
    f"{_test_permutation_seed_cell19}"
)

print()
print("First 20 permutation indices:")

print(
    _test_permutation_cell19[:20]
)

print()

print(
    "Permutation reversible :",
    np.array_equal(
        _test_block_cell19,
        _test_recovered_block_cell19
    )
)


# ============================================================
# ASSERTIONS
# ============================================================

assert len(
    _test_permutation_cell19
) == BLOCK_SIZE * BLOCK_SIZE

assert len(
    _test_inverse_permutation_cell19
) == BLOCK_SIZE * BLOCK_SIZE


# Forward permutation must contain every position exactly once
assert np.array_equal(
    np.sort(
        _test_permutation_cell19
    ),
    np.arange(
        BLOCK_SIZE * BLOCK_SIZE
    )
)


# Inverse must correctly reverse the permutation
assert np.array_equal(
    _test_block_cell19,
    _test_recovered_block_cell19
)


# Explicit inverse relationship
assert np.array_equal(
    _test_permutation_cell19[
        _test_inverse_permutation_cell19
    ],
    np.arange(
        BLOCK_SIZE * BLOCK_SIZE
    )
)


# ------------------------------------------------------------
# Determinism test
# ------------------------------------------------------------

(
    _test_perm_repeat,
    _test_inv_repeat,
    _test_seed_repeat
) = generate_block_permutation(
    S_i=_test_state_cell19,
    block_index=0,
    block_size=BLOCK_SIZE
)

assert np.array_equal(
    _test_permutation_cell19,
    _test_perm_repeat
)

assert np.array_equal(
    _test_inverse_permutation_cell19,
    _test_inv_repeat
)

assert (
    _test_permutation_seed_cell19
    ==
    _test_seed_repeat
)


# ============================================================
# FINAL STATUS
# ============================================================

print()
print("Permutation generation      : PASSED")
print("Permutation reversibility   : PASSED")
print("Permutation determinism     : PASSED")

print()
print("V2 architecture check:")
print("Global S0 dependency        : REMOVED")
print("Plaintext dependency        : REMOVED")
print("256-bit state input         : SUPPORTED")
print("Block-index dependency      : SUPPORTED")

print()
print("CELL 19 COMPLETE")

CELL 19 — STATE-DEPENDENT ADAPTIVE PERMUTATION — V2
Block size       : 16 × 16
Positions        : 256
Permutation seed : 11181160551623332960

First 20 permutation indices:
[224 130 159   3 189  64 211  92  13  39 175 162   0 123 145  90  60  36
  22 116]

Permutation reversible : True

Permutation generation      : PASSED
Permutation reversibility   : PASSED
Permutation determinism     : PASSED

V2 architecture check:
Global S0 dependency        : REMOVED
Plaintext dependency        : REMOVED
256-bit state input         : SUPPORTED
Block-index dependency      : SUPPORTED

CELL 19 COMPLETE


In [ ]:
# ============================================================
# CELL 20 — DNA ENCODING
# ============================================================

# 2-bit → DNA base index
DNA_BASES = np.array(
    ["A", "T", "C", "G"],
    dtype="<U1"
)

# Base index corresponding to each 2-bit value
BIT_PAIR_TO_INDEX = {
    0b00: 0,
    0b01: 1,
    0b10: 2,
    0b11: 3
}


def dna_encode_byte(value, rule):
    """
    Encode one 8-bit pixel/byte into four DNA bases.

        p = b7 b6 b5 b4 b3 b2 b1 b0

        (b7,b6), (b5,b4), (b3,b2), (b1,b0)

    The selected DNA rule determines the base assigned
    to each 2-bit value.

    Example:
        rule = ATCG

        00 -> A
        01 -> T
        10 -> C
        11 -> G
    """

    value = int(value)

    if not (0 <= value <= 255):
        raise ValueError(
            "Pixel/byte value must be in [0,255]."
        )

    if rule not in DNA_RULE_TO_ID:
        raise ValueError(
            f"Invalid DNA rule: {rule}"
        )

    # Four 2-bit groups
    pair_1 = (value >> 6) & 0b11
    pair_2 = (value >> 4) & 0b11
    pair_3 = (value >> 2) & 0b11
    pair_4 = value & 0b11

    pairs = [
        pair_1,
        pair_2,
        pair_3,
        pair_4
    ]

    return "".join(
        rule[pair]
        for pair in pairs
    )


def dna_encode_array(values, rule):
    """
    Encode a uint8 array into DNA.

    Input:
        values : uint8 array

    Output:
        DNA array with shape (..., 4)
    """

    values = np.asarray(
        values,
        dtype=np.uint8
    )

    if rule not in DNA_RULE_TO_ID:
        raise ValueError(
            f"Invalid DNA rule: {rule}"
        )

    flat_values = values.reshape(-1)

    # Extract four 2-bit groups
    pair_1 = (flat_values >> 6) & 0b11
    pair_2 = (flat_values >> 4) & 0b11
    pair_3 = (flat_values >> 2) & 0b11
    pair_4 = flat_values & 0b11

    encoded = np.empty(
        (flat_values.size, 4),
        dtype="<U1"
    )

    encoded[:, 0] = [
        rule[int(x)]
        for x in pair_1
    ]

    encoded[:, 1] = [
        rule[int(x)]
        for x in pair_2
    ]

    encoded[:, 2] = [
        rule[int(x)]
        for x in pair_3
    ]

    encoded[:, 3] = [
        rule[int(x)]
        for x in pair_4
    ]

    return encoded.reshape(
        values.shape + (4,)
    )


# ------------------------------------------------------------
# Test DNA encoding
# ------------------------------------------------------------

test_value = 0b00011011
test_rule = DNA_RULES[1]

test_dna = dna_encode_byte(
    test_value,
    test_rule
)

print("=" * 80)
print("DNA ENCODING TEST")
print("=" * 80)

print(f"Pixel value : {test_value}")
print(f"Binary      : {format(test_value, '08b')}")
print(f"DNA rule    : R_1 = {test_rule}")
print(f"DNA output  : {test_dna}")

# Expected:
# 00 -> A
# 01 -> T
# 10 -> C
# 11 -> G
assert test_dna == "ATCG"

print()
print("DNA encoding test: PASSED")

DNA ENCODING TEST
Pixel value : 27
Binary      : 00011011
DNA rule    : R_1 = ATCG
DNA output  : ATCG

DNA encoding test: PASSED


In [ ]:
# ============================================================
# CELL 21 — STATE-DERIVED CHAOTIC DNA KEYSTREAM — V2
# ============================================================

"""
PURPOSE:
    Generate a deterministic chaotic DNA keystream from an
    explicitly supplied 256-bit state.

V2 ARCHITECTURE:
    - No global S0
    - No plaintext dependency
    - No global x0 / logistic_r
    - One chaotic byte per pixel
    - DNA encoding preserves block shape

The actual encryption pipeline will provide the appropriate
state derived from the V2 key schedule and block context.
"""


# ============================================================
# LOGISTIC PARAMETERS FROM STATE
# ============================================================

def derive_logistic_parameters_from_state(S_i):
    """
    Derive Logistic-map parameters from a 256-bit state.

        x0 = (u + 1) / (2^32 + 1)

        r  = 3.9 +
             0.099 * v / (2^32 - 1)
    """

    if not isinstance(
        S_i,
        (bytes, bytearray)
    ):
        raise TypeError(
            "S_i must be bytes or bytearray."
        )

    S_i = bytes(S_i)

    if len(S_i) != 32:
        raise ValueError(
            "S_i must contain exactly 32 bytes."
        )

    u = int.from_bytes(
        S_i[0:4],
        byteorder="big",
        signed=False
    )

    v = int.from_bytes(
        S_i[4:8],
        byteorder="big",
        signed=False
    )

    x0 = (
        (u + 1.0)
        /
        ((2 ** 32) + 1.0)
    )

    r = (
        3.9
        +
        0.099
        *
        v
        /
        ((2 ** 32) - 1.0)
    )

    # Numerical safety
    x0 = min(
        max(
            x0,
            np.finfo(np.float64).eps
        ),
        1.0 - np.finfo(np.float64).eps
    )

    r = min(
        max(r, 3.9),
        3.999999
    )

    return x0, r


# ============================================================
# GENERATE STATE-DERIVED CHAOTIC BYTES
# ============================================================

def generate_state_chaotic_bytes(
    S_i,
    number_of_bytes,
    transient=1000
):
    """
    Generate state-dependent Logistic chaotic bytes.

        x_(n+1) = r*x_n*(1-x_n)

        k_n = floor(10^14*x_n) mod 256

    The initial transient is discarded.
    """

    if not isinstance(
        S_i,
        (bytes, bytearray)
    ):
        raise TypeError(
            "S_i must be bytes or bytearray."
        )

    S_i = bytes(S_i)

    if len(S_i) != 32:
        raise ValueError(
            "S_i must contain exactly 32 bytes."
        )

    if not isinstance(
        number_of_bytes,
        (int, np.integer)
    ):
        raise TypeError(
            "number_of_bytes must be an integer."
        )

    if number_of_bytes <= 0:
        raise ValueError(
            "number_of_bytes must be positive."
        )

    if transient < 0:
        raise ValueError(
            "transient must be non-negative."
        )

    x, r = (
        derive_logistic_parameters_from_state(
            S_i
        )
    )

    total_iterations = (
        int(transient)
        +
        int(number_of_bytes)
    )

    values = np.empty(
        int(number_of_bytes),
        dtype=np.uint8
    )

    value_index = 0

    for iteration in range(
        total_iterations
    ):

        x = (
            r *
            x *
            (1.0 - x)
        )

        # Numerical protection
        if x <= 0.0:
            x = np.finfo(
                np.float64
            ).eps

        elif x >= 1.0:
            x = (
                1.0 -
                np.finfo(
                    np.float64
                ).eps
            )

        if iteration >= transient:

            chaotic_value = (
                int(
                    np.floor(
                        10 ** 14 * x
                    )
                )
                % 256
            )

            values[value_index] = (
                chaotic_value
            )

            value_index += 1

    return values


# ============================================================
# GENERATE DNA KEYSTREAM
# ============================================================

def generate_dna_keystream(
    S_i,
    rule,
    block_shape,
    transient=1000
):
    """
    Generate a state-derived chaotic DNA keystream.

    For a 16x16 block:

        chaotic bytes
            -> (16,16)

        DNA encoding
            -> (16,16,4)
    """

    if not isinstance(
        block_shape,
        (tuple, list)
    ):
        raise TypeError(
            "block_shape must be a tuple or list."
        )

    if len(block_shape) != 2:
        raise ValueError(
            "block_shape must contain "
            "(height, width)."
        )

    block_height = int(
        block_shape[0]
    )

    block_width = int(
        block_shape[1]
    )

    if (
        block_height <= 0
        or
        block_width <= 0
    ):
        raise ValueError(
            "Block dimensions must be positive."
        )

    number_of_pixels = (
        block_height *
        block_width
    )

    # --------------------------------------------------------
    # Generate one chaotic byte per pixel
    # --------------------------------------------------------

    chaotic_bytes_flat = (
        generate_state_chaotic_bytes(
            S_i=S_i,
            number_of_bytes=number_of_pixels,
            transient=transient
        )
    )

    # --------------------------------------------------------
    # Restore spatial block shape
    # --------------------------------------------------------

    chaotic_bytes = (
        chaotic_bytes_flat.reshape(
            block_height,
            block_width
        )
    )

    # --------------------------------------------------------
    # DNA encode
    # --------------------------------------------------------

    dna_stream = (
        dna_encode_array(
            chaotic_bytes,
            rule
        )
    )

    return (
        chaotic_bytes,
        dna_stream
    )


# ============================================================
# STANDALONE DETERMINISTIC TEST
# ============================================================

"""
Do NOT use S0 here.

This state exists only to test the functions in CELL 21.
The production encryption routine supplies its own state.
"""

_test_state_cell21 = bytes.fromhex(
    "00112233445566778899aabbccddeeff"
    "102132435465768798a9bacbdcedfe0f"
)


# ------------------------------------------------------------
# Use an existing DNA rule explicitly
# ------------------------------------------------------------

_test_rule_cell21 = DNA_RULES[1]


# ------------------------------------------------------------
# Test block dimensions
# ------------------------------------------------------------

_test_block_shape_cell21 = (
    BLOCK_SIZE,
    BLOCK_SIZE
)

block_pixel_count = (
    BLOCK_SIZE *
    BLOCK_SIZE
)


# ============================================================
# GENERATE TEST KEYSTREAM
# ============================================================

test_chaotic_bytes, test_dna_key = (
    generate_dna_keystream(
        S_i=_test_state_cell21,
        rule=_test_rule_cell21,
        block_shape=_test_block_shape_cell21,
        transient=CONFIG["CHAOTIC_TRANSIENT"]
    )
)


# ============================================================
# TEST PLAINTEXT DNA REPRESENTATION
# ============================================================

test_plain_block_cell21 = np.asarray(
    test_image[
        :BLOCK_SIZE,
        :BLOCK_SIZE
    ],
    dtype=np.uint8
).copy()

test_plain_dna = (
    dna_encode_array(
        test_plain_block_cell21,
        _test_rule_cell21
    )
)


# ============================================================
# DISPLAY
# ============================================================

print("=" * 80)
print("CELL 21 — STATE-DERIVED CHAOTIC DNA KEYSTREAM — V2")
print("=" * 80)

x0_test, r_test = (
    derive_logistic_parameters_from_state(
        _test_state_cell21
    )
)

print(
    f"x0                 : "
    f"{x0_test:.16f}"
)

print(
    f"r                  : "
    f"{r_test:.16f}"
)

print(
    f"Transient          : "
    f"{CONFIG['CHAOTIC_TRANSIENT']}"
)

print(
    f"Block shape        : "
    f"{BLOCK_SIZE} × {BLOCK_SIZE}"
)

print(
    f"Block pixels       : "
    f"{block_pixel_count}"
)

print()

print(
    f"Chaotic bytes shape: "
    f"{test_chaotic_bytes.shape}"
)

print(
    f"DNA key shape      : "
    f"{test_dna_key.shape}"
)

print(
    f"Plain DNA shape    : "
    f"{test_plain_dna.shape}"
)

print()

print("First chaotic bytes:")
print(
    test_chaotic_bytes.reshape(-1)[:16]
)

print()

print("First plaintext DNA symbols:")
print(
    test_plain_dna.reshape(-1)[:32]
)

print()

print("First keystream DNA symbols:")
print(
    test_dna_key.reshape(-1)[:32]
)


# ============================================================
# ASSERTIONS
# ============================================================

assert test_chaotic_bytes.shape == (
    BLOCK_SIZE,
    BLOCK_SIZE
)

assert test_dna_key.shape == (
    BLOCK_SIZE,
    BLOCK_SIZE,
    4
)

assert test_plain_dna.shape == (
    BLOCK_SIZE,
    BLOCK_SIZE,
    4
)

assert test_chaotic_bytes.dtype == np.uint8

assert np.all(
    np.isfinite(
        test_chaotic_bytes
    )
)

assert np.all(
    np.isin(
        test_dna_key,
        ["A", "T", "C", "G"]
    )
)


# ============================================================
# DETERMINISM TEST
# ============================================================

repeat_chaotic_bytes, repeat_dna_key = (
    generate_dna_keystream(
        S_i=_test_state_cell21,
        rule=_test_rule_cell21,
        block_shape=_test_block_shape_cell21,
        transient=CONFIG["CHAOTIC_TRANSIENT"]
    )
)

assert np.array_equal(
    test_chaotic_bytes,
    repeat_chaotic_bytes
)

assert np.array_equal(
    test_dna_key,
    repeat_dna_key
)


# ============================================================
# FINAL STATUS
# ============================================================

print()
print("State-derived chaotic DNA keystream : PASSED")
print("Deterministic repeatability         : PASSED")

print()
print("V2 architecture check:")
print("Global S0 dependency               : REMOVED")
print("Plaintext dependency               : REMOVED")
print("Block-shaped DNA keystream         : SUPPORTED")
print("Deterministic state input          : SUPPORTED")

print()
print("CELL 21 COMPLETE")

CELL 21 — STATE-DERIVED CHAOTIC DNA KEYSTREAM — V2
x0                 : 0.0002614380791175
r                  : 3.9264258823467943
Transient          : 1000
Block shape        : 16 × 16
Block pixels       : 256

Chaotic bytes shape: (16, 16)
DNA key shape      : (16, 16, 4)
Plain DNA shape    : (16, 16, 4)

First chaotic bytes:
[ 70  12  87  17 183  54 254  20  32 117 225  25 231 152 242 188]

First plaintext DNA symbols:
['A' 'A' 'C' 'C' 'A' 'A' 'C' 'C' 'A' 'A' 'C' 'G' 'A' 'G' 'T' 'G' 'A' 'G'
 'G' 'T' 'A' 'G' 'G' 'T' 'A' 'G' 'G' 'G' 'T' 'T' 'A' 'T']

First keystream DNA symbols:
['T' 'A' 'T' 'C' 'A' 'A' 'G' 'A' 'T' 'T' 'T' 'G' 'A' 'T' 'A' 'T' 'C' 'G'
 'T' 'G' 'A' 'G' 'T' 'C' 'G' 'G' 'G' 'C' 'A' 'T' 'T' 'A']

State-derived chaotic DNA keystream : PASSED
Deterministic repeatability         : PASSED

V2 architecture check:
Global S0 dependency               : REMOVED
Plaintext dependency               : REMOVED
Block-shaped DNA keystream         : SUPPORTED
Deterministic state input     

In [ ]:
# ============================================================
# CELL 22 — DNA-XOR
# ============================================================

# ------------------------------------------------------------
# Fixed DNA-XOR operation
#
#        ⊕DNA
#
#        A T C G
#    A   A T C G
#    T   T A G C
#    C   C G A T
#    G   G C T A
# ------------------------------------------------------------

DNA_XOR_TABLE = {
    "A": {
        "A": "A",
        "T": "T",
        "C": "C",
        "G": "G"
    },

    "T": {
        "A": "T",
        "T": "A",
        "C": "G",
        "G": "C"
    },

    "C": {
        "A": "C",
        "T": "G",
        "C": "A",
        "G": "T"
    },

    "G": {
        "A": "G",
        "T": "C",
        "C": "T",
        "G": "A"
    }
}


def dna_xor_symbol(a, b):
    """
    Perform one DNA-XOR operation.
    """

    if a not in DNA_XOR_TABLE:
        raise ValueError(
            f"Invalid DNA symbol: {a}"
        )

    if b not in DNA_XOR_TABLE:
        raise ValueError(
            f"Invalid DNA symbol: {b}"
        )

    return DNA_XOR_TABLE[a][b]


def dna_xor(
    dna_a,
    dna_b
):
    """
    Element-wise DNA-XOR.

        X_i =
        DNA(B_i') ⊕_DNA D_i
    """

    dna_a = np.asarray(
        dna_a,
        dtype="<U1"
    )

    dna_b = np.asarray(
        dna_b,
        dtype="<U1"
    )

    if dna_a.shape != dna_b.shape:
        raise ValueError(
            "DNA arrays must have identical shapes."
        )

    result = np.empty_like(
        dna_a
    )

    for symbol_a in ["A", "T", "C", "G"]:
        for symbol_b in ["A", "T", "C", "G"]:

            mask = (
                (dna_a == symbol_a) &
                (dna_b == symbol_b)
            )

            result[mask] = DNA_XOR_TABLE[
                symbol_a
            ][symbol_b]

    return result


# ------------------------------------------------------------
# Apply first DNA-XOR layer
# ------------------------------------------------------------

X_i = dna_xor(
    test_plain_dna,
    test_dna_key
)

print("=" * 80)
print("DNA-XOR")
print("=" * 80)

print("DNA-XOR table:")
print()
print("      A T C G")
print("A     A T C G")
print("T     T A G C")
print("C     C G A T")
print("G     G C T A")

print()
print(f"Input DNA shape : {test_plain_dna.shape}")
print(f"Key DNA shape   : {test_dna_key.shape}")
print(f"Output X_i      : {X_i.shape}")

# Verify all outputs are valid DNA symbols
assert np.all(
    np.isin(
        X_i,
        ["A", "T", "C", "G"]
    )
)

print()
print("DNA-XOR test: PASSED")

DNA-XOR
DNA-XOR table:

      A T C G
A     A T C G
T     T A G C
C     C G A T
G     G C T A

Input DNA shape : (16, 16, 4)
Key DNA shape   : (16, 16, 4)
Output X_i      : (16, 16, 4)

DNA-XOR test: PASSED


In [ ]:
# ============================================================
# CELL 23 — FULL-BLOCK CIPHERTEXT FEEDBACK DIFFUSION
# ============================================================

def generate_feedback_mask(
    current_state,
    feedback_bytes,
    block_pixel_count
):
    """
    Generate a complete block-sized feedback mask using
    counter-mode SHA-256.

        M_j = SHA256(
            S_i || F_i || counter_j || FEEDBACK_MASK
        )

    Multiple SHA-256 digests are concatenated when the block
    contains more than 32 pixels.
    """

    if isinstance(current_state, bytearray):
        current_state = bytes(current_state)

    if not isinstance(current_state, bytes):
        raise TypeError(
            "current_state must be bytes or bytearray."
        )

    if len(current_state) != 32:
        raise ValueError(
            "current_state must contain exactly 32 bytes."
        )

    feedback_bytes = np.asarray(
        feedback_bytes,
        dtype=np.uint8
    )

    if feedback_bytes.size == 0:
        raise ValueError(
            "feedback_bytes cannot be empty."
        )

    required_bytes = int(block_pixel_count)

    if required_bytes <= 0:
        raise ValueError(
            "block_pixel_count must be positive."
        )

    feedback_bytes = np.ascontiguousarray(
        feedback_bytes
    ).tobytes()

    generated = bytearray()

    counter = 0

    while len(generated) < required_bytes:

        counter_bytes = int(counter).to_bytes(
            8,
            byteorder="big",
            signed=False
        )

        digest = hashlib.sha256(
            current_state
            + feedback_bytes
            + counter_bytes
            + b"FEEDBACK_MASK"
        ).digest()

        generated.extend(digest)

        counter += 1

    return np.frombuffer(
        bytes(generated[:required_bytes]),
        dtype=np.uint8
    ).copy()


def apply_feedback_diffusion(
    X_i,
    current_state,
    feedback_bytes,
    rule
):
    """
    Apply the full-block DNA feedback diffusion layer.

        M_i = SHA256-counter stream(S_i || F_i)

        D_i = DNAEncode(M_i, R_i)

        Y_i = X_i XOR_DNA D_i

    F_0 is supplied as a zero block during the first-block test.

    For subsequent encryption blocks, feedback_bytes should be
    the previous ciphertext block.
    """

    X_i = np.asarray(X_i)

    if X_i.ndim != 3:
        raise ValueError(
            "X_i must have shape (H, W, 4)."
        )

    if X_i.shape[2] != 4:
        raise ValueError(
            "X_i must contain 4 DNA bases per pixel."
        )

    block_pixel_count = (
        int(X_i.shape[0]) *
        int(X_i.shape[1])
    )

    M_i = generate_feedback_mask(
        current_state=current_state,
        feedback_bytes=feedback_bytes,
        block_pixel_count=block_pixel_count
    )

    feedback_dna = dna_encode_array(
        M_i,
        rule
    )

    feedback_dna = feedback_dna.reshape(
        X_i.shape
    )

    Y_i = dna_xor(
        X_i,
        feedback_dna
    )

    return (
        Y_i,
        M_i,
        feedback_dna
    )


# ============================================================
# STANDALONE TEST
# Do NOT use the old global S0.
# ============================================================

_test_state_cell23 = bytes.fromhex(
    "00112233445566778899aabbccddeeff"
    "102132435465768798a9bacbdcedfe0f"
)

_test_rule_cell23 = DNA_RULES[1]

_test_block_cell23 = np.asarray(
    test_image[:BLOCK_SIZE, :BLOCK_SIZE],
    dtype=np.uint8
).copy()

_test_X_i_cell23 = dna_encode_array(
    _test_block_cell23,
    _test_rule_cell23
)

_test_block_pixel_count_cell23 = (
    BLOCK_SIZE * BLOCK_SIZE
)

_test_F0_cell23 = np.zeros(
    _test_block_pixel_count_cell23,
    dtype=np.uint8
)

Y_i, M_i, feedback_dna = apply_feedback_diffusion(
    X_i=_test_X_i_cell23,
    current_state=_test_state_cell23,
    feedback_bytes=_test_F0_cell23,
    rule=_test_rule_cell23
)

print("=" * 80)
print("FULL-BLOCK CIPHERTEXT FEEDBACK DIFFUSION")
print("=" * 80)

print("First block feedback:")
print("F_0 = 0")

print()
print(
    f"Feedback mask bytes : {len(M_i)}"
)

print(
    f"Feedback mask bits  : {len(M_i) * 8}"
)

print(
    f"Feedback DNA shape  : {feedback_dna.shape}"
)

print(
    f"Y_i shape           : {Y_i.shape}"
)

print()
print("First 32 feedback bytes:")
print(M_i[:32])

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert len(M_i) == _test_block_pixel_count_cell23

assert feedback_dna.shape == (
    BLOCK_SIZE,
    BLOCK_SIZE,
    4
)

assert Y_i.shape == _test_X_i_cell23.shape

assert np.all(
    np.isin(
        feedback_dna,
        ["A", "T", "C", "G"]
    )
)

assert np.all(
    np.isin(
        Y_i,
        ["A", "T", "C", "G"]
    )
)

# ------------------------------------------------------------
# Determinism test
# ------------------------------------------------------------

Y_i_repeat, M_i_repeat, feedback_dna_repeat = (
    apply_feedback_diffusion(
        X_i=_test_X_i_cell23,
        current_state=_test_state_cell23,
        feedback_bytes=_test_F0_cell23,
        rule=_test_rule_cell23
    )
)

assert np.array_equal(
    M_i,
    M_i_repeat
)

assert np.array_equal(
    feedback_dna,
    feedback_dna_repeat
)

assert np.array_equal(
    Y_i,
    Y_i_repeat
)

print()
print("Deterministic feedback generation: PASSED")
print("Full-block feedback diffusion: PASSED")

FULL-BLOCK CIPHERTEXT FEEDBACK DIFFUSION
First block feedback:
F_0 = 0

Feedback mask bytes : 256
Feedback mask bits  : 2048
Feedback DNA shape  : (16, 16, 4)
Y_i shape           : (16, 16, 4)

First 32 feedback bytes:
[115 192 143 102 228 162 191 155  67 216 168  86  75 189 116 125 135   2
  43  98 190 134 219 221 189  62  14 183  44  51 200 108]

Deterministic feedback generation: PASSED
Full-block feedback diffusion: PASSED


In [ ]:
# ============================================================
# CELL 24 — DNA DECODING + CIPHER BLOCK RECONSTRUCTION
# ============================================================

def dna_decode_array(
    dna_array,
    rule
):
    """
    Decode four DNA bases into one 8-bit byte.

        DNA → four 2-bit groups → uint8

    The inverse mapping is determined by the
    selected DNA rule.
    """

    dna_array = np.asarray(
        dna_array,
        dtype="<U1"
    )

    if dna_array.shape[-1] != 4:
        raise ValueError(
            "Last DNA dimension must contain 4 bases."
        )

    if rule not in DNA_RULE_TO_ID:
        raise ValueError(
            f"Invalid DNA rule: {rule}"
        )

    # Build inverse rule:
    #
    # base -> 2-bit value
    inverse_rule = {
        base: index
        for index, base in enumerate(rule)
    }

    flat_dna = dna_array.reshape(
        -1,
        4
    )

    decoded = np.empty(
        flat_dna.shape[0],
        dtype=np.uint8
    )

    for i in range(flat_dna.shape[0]):

        b0 = inverse_rule[
            flat_dna[i, 0]
        ]

        b1 = inverse_rule[
            flat_dna[i, 1]
        ]

        b2 = inverse_rule[
            flat_dna[i, 2]
        ]

        b3 = inverse_rule[
            flat_dna[i, 3]
        ]

        value = (
            (b0 << 6) |
            (b1 << 4) |
            (b2 << 2) |
            b3
        )

        decoded[i] = value

    output_shape = dna_array.shape[:-1]

    return decoded.reshape(
        output_shape
    )


# ------------------------------------------------------------
# Decode final DNA block
# ------------------------------------------------------------

cipher_block = dna_decode_array(
    Y_i,
    test_rule
)

print("=" * 80)
print("DNA DECODING + CIPHER BLOCK")
print("=" * 80)

print(f"DNA input shape : {Y_i.shape}")
print(f"Cipher block    : {cipher_block.shape}")
print(f"Dtype           : {cipher_block.dtype}")
print(f"Minimum value   : {cipher_block.min()}")
print(f"Maximum value   : {cipher_block.max()}")

assert cipher_block.shape == (
    BLOCK_SIZE,
    BLOCK_SIZE
)

assert cipher_block.dtype == np.uint8

assert np.all(
    (cipher_block >= 0) &
    (cipher_block <= 255)
)

print()
print("DNA decoding test: PASSED")

DNA DECODING + CIPHER BLOCK
DNA input shape : (16, 16, 4)
Cipher block    : (16, 16)
Dtype           : uint8
Minimum value   : 0
Maximum value   : 255

DNA decoding test: PASSED


In [ ]:
# ============================================================
# CELL 25 — CIPHERTEXT-ONLY STATE EVOLUTION
#              + BIDIRECTIONAL CIPHERTEXT DIFFUSION
# ============================================================

import hashlib
import hmac
import numpy as np


# ============================================================
# FORWARD CIPHERTEXT-ONLY STATE EVOLUTION
# ============================================================

def evolve_state_v2(
    current_state,
    cipher_block,
    block_index,
    k_diff
):

    if not isinstance(current_state, bytes):
        raise TypeError(
            "current_state must be bytes."
        )

    if len(current_state) != 32:
        raise ValueError(
            "current_state must contain 32 bytes."
        )

    cipher_block = np.asarray(
        cipher_block,
        dtype=np.uint8
    ).reshape(-1)

    index_bytes = int(
        block_index
    ).to_bytes(
        8,
        byteorder="big",
        signed=False
    )

    message = (
        current_state
        + index_bytes
        + np.ascontiguousarray(
            cipher_block
        ).tobytes()
        + b"STATE_UPDATE"
    )

    return hmac.new(
        k_diff,
        message,
        hashlib.sha256
    ).digest()


# ============================================================
# REVERSE-DIFFUSION MASK
# ============================================================

def _generate_reverse_diffusion_mask(
    k_diff,
    next_block,
    block_index,
    block_pixel_count
):
    """
    Generate a deterministic mask for backward ciphertext
    diffusion.

    The mask depends only on:
        K_DIFF
        next final ciphertext block
        block index

    No plaintext is used.
    """

    if not isinstance(k_diff, bytes):
        raise TypeError(
            "k_diff must be bytes."
        )

    if len(k_diff) != 32:
        raise ValueError(
            "k_diff must contain 32 bytes."
        )

    next_block = np.asarray(
        next_block,
        dtype=np.uint8
    ).reshape(-1)

    if next_block.size != block_pixel_count:
        raise ValueError(
            "next_block has incorrect size."
        )

    index_bytes = int(
        block_index
    ).to_bytes(
        8,
        byteorder="big",
        signed=False
    )

    mask = bytearray()

    counter = 0

    while len(mask) < block_pixel_count:

        counter_bytes = int(
            counter
        ).to_bytes(
            4,
            byteorder="big",
            signed=False
        )

        message = (
            b"REVERSE_DIFFUSION_V1"
            + index_bytes
            + next_block.tobytes()
            + counter_bytes
        )

        digest = hmac.new(
            k_diff,
            message,
            hashlib.sha256
        ).digest()

        mask.extend(digest)

        counter += 1

    return np.frombuffer(
        bytes(
            mask[:block_pixel_count]
        ),
        dtype=np.uint8
    )


# ============================================================
# FORWARD APPLICATION OF BACKWARD CIPHERTEXT DIFFUSION
# ============================================================

def apply_reverse_ciphertext_diffusion(
    cipher_image,
    k_diff,
    block_size
):
    """
    Apply reversible backward ciphertext diffusion.

    Existing encryption produces:

        P -> F

    This function produces:

        F -> C

    by processing blocks from the final block toward
    the first block.

    For block i:

        C_i = F_i XOR M_i

    where M_i depends on C_(i+1).

    Therefore a plaintext difference that changes a
    forward ciphertext block can propagate backward
    through the complete ciphertext.
    """

    cipher_image = np.asarray(
        cipher_image,
        dtype=np.uint8
    )

    if cipher_image.ndim != 2:
        raise ValueError(
            "cipher_image must be a 2-D grayscale image."
        )

    H, W = cipher_image.shape

    B = int(block_size)

    if B <= 0:
        raise ValueError(
            "block_size must be positive."
        )

    if H % B != 0 or W % B != 0:
        raise ValueError(
            "Image dimensions must be divisible "
            "by block_size."
        )

    output = np.zeros_like(
        cipher_image,
        dtype=np.uint8
    )

    blocks_per_row = W // B
    total_blocks = (
        (H // B) *
        blocks_per_row
    )

    # There is no next block after the final block.
    next_final_block = np.zeros(
        B * B,
        dtype=np.uint8
    )

    # --------------------------------------------------------
    # IMPORTANT:
    # Process from LAST block to FIRST block.
    # --------------------------------------------------------

    for block_index in range(
        total_blocks - 1,
        -1,
        -1
    ):

        block_row = (
            block_index //
            blocks_per_row
        )

        block_col = (
            block_index %
            blocks_per_row
        )

        y0 = block_row * B
        y1 = y0 + B

        x0 = block_col * B
        x1 = x0 + B

        forward_block = np.asarray(
            cipher_image[
                y0:y1,
                x0:x1
            ],
            dtype=np.uint8
        ).reshape(-1)

        mask = _generate_reverse_diffusion_mask(
            k_diff=k_diff,
            next_block=next_final_block,
            block_index=block_index,
            block_pixel_count=B * B
        )

        final_block = np.bitwise_xor(
            forward_block,
            mask
        ).astype(np.uint8)

        output[
            y0:y1,
            x0:x1
        ] = final_block.reshape(
            B,
            B
        )

        # The FINAL ciphertext block is used by the
        # preceding block.
        next_final_block = (
            final_block.copy()
        )

    return output


# ============================================================
# INVERSE OF BACKWARD CIPHERTEXT DIFFUSION
# ============================================================

def undo_reverse_ciphertext_diffusion(
    final_cipher_image,
    k_diff,
    block_size
):
    """
    Recover the forward ciphertext from the final
    bidirectionally diffused ciphertext.

        C_i = F_i XOR M(C_(i+1))

    therefore:

        F_i = C_i XOR M(C_(i+1))
    """

    final_cipher_image = np.asarray(
        final_cipher_image,
        dtype=np.uint8
    )

    if final_cipher_image.ndim != 2:
        raise ValueError(
            "final_cipher_image must be 2-D."
        )

    H, W = final_cipher_image.shape

    B = int(block_size)

    if B <= 0:
        raise ValueError(
            "block_size must be positive."
        )

    if H % B != 0 or W % B != 0:
        raise ValueError(
            "Cipher dimensions must be divisible "
            "by block_size."
        )

    recovered = np.zeros_like(
        final_cipher_image,
        dtype=np.uint8
    )

    blocks_per_row = W // B

    total_blocks = (
        (H // B) *
        blocks_per_row
    )

    next_final_block = np.zeros(
        B * B,
        dtype=np.uint8
    )

    # Same order used during encryption.
    for block_index in range(
        total_blocks - 1,
        -1,
        -1
    ):

        block_row = (
            block_index //
            blocks_per_row
        )

        block_col = (
            block_index %
            blocks_per_row
        )

        y0 = block_row * B
        y1 = y0 + B

        x0 = block_col * B
        x1 = x0 + B

        final_block = np.asarray(
            final_cipher_image[
                y0:y1,
                x0:x1
            ],
            dtype=np.uint8
        ).reshape(-1)

        mask = _generate_reverse_diffusion_mask(
            k_diff=k_diff,
            next_block=next_final_block,
            block_index=block_index,
            block_pixel_count=B * B
        )

        forward_block = np.bitwise_xor(
            final_block,
            mask
        ).astype(np.uint8)

        recovered[
            y0:y1,
            x0:x1
        ] = forward_block.reshape(
            B,
            B
        )

        next_final_block = (
            final_block.copy()
        )

    return recovered


# ============================================================
# CELL 25 SELF-TEST
# ============================================================

_test_kdiff_cell25 = hashlib.sha256(
    b"CELL25-REVERSE-DIFFUSION-TEST"
).digest()

_test_image_cell25 = np.asarray(
    test_image,
    dtype=np.uint8
).copy()

_test_diffused_cell25 = (
    apply_reverse_ciphertext_diffusion(
        cipher_image=_test_image_cell25,
        k_diff=_test_kdiff_cell25,
        block_size=BLOCK_SIZE
    )
)

_test_recovered_cell25 = (
    undo_reverse_ciphertext_diffusion(
        final_cipher_image=_test_diffused_cell25,
        k_diff=_test_kdiff_cell25,
        block_size=BLOCK_SIZE
    )
)

if not np.array_equal(
    _test_image_cell25,
    _test_recovered_cell25
):
    raise RuntimeError(
        "CELL 25 reverse diffusion self-test FAILED."
    )


print("=" * 80)
print("CELL 25 — CIPHERTEXT-ONLY STATE + BIDIRECTIONAL DIFFUSION")
print("=" * 80)

print(
    "Forward ciphertext-only state evolution : ENABLED"
)

print(
    "Backward ciphertext diffusion            : ENABLED"
)

print(
    "Plaintext-dependent state update         : DISABLED"
)

print(
    "Reverse diffusion self-test              : PASSED"
)

print()
print("CELL 25 COMPLETE")

CELL 25 — CIPHERTEXT-ONLY STATE + BIDIRECTIONAL DIFFUSION
Forward ciphertext-only state evolution : ENABLED
Backward ciphertext diffusion            : ENABLED
Plaintext-dependent state update         : DISABLED
Reverse diffusion self-test              : PASSED

CELL 25 COMPLETE


In [ ]:
# ============================================================
# CELL 26 — COMPLETE V2 ENCRYPTION
# ============================================================

def encrypt_image(
    image,
    secret_key,
    save_internal=False,
    image_name="image",
    salt=None,
    nonce=None
):

    total_start = time.perf_counter()

    # --------------------------------------------------------
    # INPUT VALIDATION
    # --------------------------------------------------------

    image = np.asarray(
        image,
        dtype=np.uint8
    )

    if image.ndim != 2:

        raise ValueError(
            "Encryption requires "
            "a grayscale 2-D image."
        )

    H, W = image.shape

    B = int(
        CONFIG["BLOCK_SIZE"]
    )

    if B <= 0:

        raise ValueError(
            "CONFIG['BLOCK_SIZE'] must be positive."
        )

    if H % B != 0 or W % B != 0:

        raise ValueError(
            f"Image dimensions {H}x{W} "
            f"must be divisible by {B}."
        )

    # --------------------------------------------------------
    # CHAOS TRANSIENT
    #
    # IMPORTANT:
    # Older versions of the notebook did not contain
    # CONFIG["CHAOS_TRANSIENT"].
    #
    # Use CONFIG value when available.
    # Otherwise use the V2 default of 1000.
    # --------------------------------------------------------

    chaos_transient = int(
        CONFIG.get(
            "CHAOS_TRANSIENT",
            1000
        )
    )

    if chaos_transient < 0:

        raise ValueError(
            "CHAOS_TRANSIENT must be "
            "a non-negative integer."
        )

    # --------------------------------------------------------
    # SALT + NONCE
    # --------------------------------------------------------

    if salt is None:

        salt = secrets.token_bytes(
            int(
                CONFIG["SALT_BYTES"]
            )
        )

    else:

        salt = bytes(salt)

        if len(salt) != int(
            CONFIG["SALT_BYTES"]
        ):

            raise ValueError(
                "Invalid salt length: "
                f"expected "
                f"{CONFIG['SALT_BYTES']} bytes, "
                f"got {len(salt)}."
            )

    if nonce is None:

        nonce = secrets.token_bytes(
            int(
                CONFIG["NONCE_BYTES"]
            )
        )

    else:

        nonce = bytes(nonce)

        if len(nonce) != int(
            CONFIG["NONCE_BYTES"]
        ):

            raise ValueError(
                "Invalid nonce length: "
                f"expected "
                f"{CONFIG['NONCE_BYTES']} bytes, "
                f"got {len(nonce)}."
            )

    # --------------------------------------------------------
    # CONTENT DESCRIPTOR
    # --------------------------------------------------------

    (
        descriptor,
        descriptor_digest,
        feature_values
    ) = extract_content_descriptor(
        image
    )

    # --------------------------------------------------------
    # KEY SCHEDULE
    # --------------------------------------------------------

    keys = derive_key_schedule(
        secret_key=secret_key,
        salt=salt,
        nonce=nonce,
        descriptor_digest=descriptor_digest
    )

    required_keys = [
        "K_DNA",
        "K_PERM",
        "K_DIFF",
        "K_CHAOS",
        "K_AUTH"
    ]

    missing_keys = [
        key
        for key in required_keys
        if key not in keys
    ]

    if missing_keys:

        raise KeyError(
            "derive_key_schedule() did not return "
            "required keys: "
            + ", ".join(missing_keys)
        )

    K_DNA = keys["K_DNA"]
    K_PERM = keys["K_PERM"]
    K_DIFF = keys["K_DIFF"]
    K_CHAOS = keys["K_CHAOS"]
    K_AUTH = keys["K_AUTH"]

    # --------------------------------------------------------
    # INITIAL STATE
    #
    # IMPORTANT:
    #
    # This state is NOT derived from plaintext.
    #
    # It is derived from:
    #
    #     K_DIFF
    #     nonce
    #     descriptor_digest
    #
    # Therefore the old:
    #
    #     H0 = SHA256(P)
    #     S0 = SHA256(K || H0)
    #
    # construction is NOT used.
    # --------------------------------------------------------

    S_current = hmac.new(
        K_DIFF,
        nonce +
        descriptor_digest +
        b"INITIAL_STATE",
        hashlib.sha256
    ).digest()

    # Diagnostic alias only.
    #
    # This does NOT represent the old plaintext-derived S0.
    S0 = S_current

    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------

    cipher_image = np.zeros_like(
        image,
        dtype=np.uint8
    )

    block_states = []
    block_rules = []
    permutations = []

    # --------------------------------------------------------
    # INITIAL CIPHERTEXT FEEDBACK
    # --------------------------------------------------------

    previous_cipher_block = np.zeros(
        B * B,
        dtype=np.uint8
    )

    # --------------------------------------------------------
    # DIVIDE IMAGE INTO BLOCKS
    # --------------------------------------------------------

    blocks = divide_into_blocks(
        image,
        block_size=B
    )

    # --------------------------------------------------------
    # BLOCK PROCESSING
    # --------------------------------------------------------

    for block_info in blocks:

        block_index = int(
            block_info["index"]
        )

        B_i = np.asarray(
            block_info["data"],
            dtype=np.uint8
        )

        if B_i.shape != (B, B):

            raise ValueError(
                f"Invalid block shape at "
                f"block {block_index}: "
                f"{B_i.shape}. "
                f"Expected {(B, B)}."
            )

        # ----------------------------------------------------
        # BLOCK INDEX
        # ----------------------------------------------------

        index_bytes = int(
            block_index
        ).to_bytes(
            8,
            byteorder="big",
            signed=False
        )

        # ----------------------------------------------------
        # ADAPTIVE CONTROL STATE
        #
        # Depends on:
        #
        #   K_DNA
        #   current state
        #   previous ciphertext block
        #   content digest
        #   block index
        #
        # No plaintext is used here.
        # ----------------------------------------------------

        Z_i = hmac.new(
            K_DNA,
            S_current +
            previous_cipher_block.tobytes() +
            descriptor_digest +
            index_bytes +
            b"ADAPTIVE_CONTROL",
            hashlib.sha256
        ).digest()

        (
            q_i,
            r_i,
            rule_i
        ) = select_adaptive_dna_rule(
            Z_i
        )

        # ----------------------------------------------------
        # ADAPTIVE PERMUTATION
        # ----------------------------------------------------

        permutation_state = hmac.new(
            K_PERM,
            S_current +
            descriptor_digest +
            index_bytes +
            b"PERMUTATION_STATE",
            hashlib.sha256
        ).digest()

        (
            permutation_i,
            inverse_permutation_i,
            permutation_seed_i
        ) = generate_block_permutation(
            S_i=permutation_state,
            block_index=block_index,
            block_size=B
        )

        B_i_permuted = permute_block(
            B_i,
            permutation_i
        )

        B_i_permuted = np.asarray(
            B_i_permuted,
            dtype=np.uint8
        )

        # ----------------------------------------------------
        # BLOCK-SPECIFIC CHAOTIC STREAM
        # ----------------------------------------------------

        chaos_state = hmac.new(
            K_CHAOS,
            S_current +
            descriptor_digest +
            index_bytes +
            b"CHAOS",
            hashlib.sha256
        ).digest()

        (
            x0_i,
            r_i_chaos,
            _,
            _
        ) = state_to_chaotic_parameters(
            chaos_state
        )

        # ----------------------------------------------------
        # GENERATE LOGISTIC CHAOS SEQUENCE
        #
        # FIX:
        #
        # Do NOT use:
        #
        #     CONFIG["CHAOS_TRANSIENT"]
        #
        # directly because older CONFIG dictionaries do not
        # contain this entry.
        # ----------------------------------------------------

        chaos_sequence = generate_logistic_sequence(
            x0=x0_i,
            r=r_i_chaos,
            sequence_length=B * B,
            transient=chaos_transient
        )

        chaos_bytes = np.asarray(
            chaotic_to_bytes(
                chaos_sequence
            ),
            dtype=np.uint8
        ).reshape(-1)

        if chaos_bytes.size != B * B:

            raise ValueError(
                "Chaotic byte stream length mismatch: "
                f"expected {B * B}, "
                f"got {chaos_bytes.size}."
            )

        # ----------------------------------------------------
        # DNA ENCODING
        # ----------------------------------------------------

        plaintext_dna = dna_encode_array(
            B_i_permuted,
            rule_i
        )

        chaos_dna = dna_encode_array(
            chaos_bytes,
            rule_i
        )

        chaos_dna = chaos_dna.reshape(
            B,
            B,
            4
        )

        # ----------------------------------------------------
        # DNA XOR
        # ----------------------------------------------------

        X_i = dna_xor(
            plaintext_dna,
            chaos_dna
        )

        # ----------------------------------------------------
        # KEYED FEEDBACK STATE
        # ----------------------------------------------------

        feedback_state = hmac.new(
            K_DIFF,
            S_current +
            descriptor_digest +
            index_bytes +
            b"FEEDBACK_STATE",
            hashlib.sha256
        ).digest()

        (
            Y_i,
            M_i,
            feedback_dna
        ) = apply_feedback_diffusion(
            X_i=X_i,
            current_state=feedback_state,
            feedback_bytes=previous_cipher_block,
            rule=rule_i
        )

        # ----------------------------------------------------
        # DNA DECODE
        # ----------------------------------------------------

        C_i = dna_decode_array(
            Y_i,
            rule_i
        ).astype(
            np.uint8
        )

        C_i = C_i.reshape(
            B,
            B
        )

        # ----------------------------------------------------
        # STORE CIPHERTEXT BLOCK
        # --------------------------------------------------------

        cipher_image[
            block_info["y0"]:block_info["y1"],
            block_info["x0"]:block_info["x1"]
        ] = C_i

        # ----------------------------------------------------
        # CIPHERTEXT-ONLY STATE UPDATE
        #
        # IMPORTANT:
        #
        # The next state depends on C_i, not plaintext B_i.
        # ----------------------------------------------------

        S_next = evolve_state_v2(
            current_state=S_current,
            cipher_block=C_i,
            block_index=block_index,
            k_diff=K_DIFF
        )

        # ----------------------------------------------------
        # INTERNAL DIAGNOSTICS
        # ----------------------------------------------------

        if save_internal:

            block_states.append({

                "block_index":
                    block_index,

                "S_i":
                    S_current.hex(),

                "Z_i":
                    Z_i.hex(),

                "rule_index":
                    int(r_i),

                "rule":
                    rule_i,

                "permutation_seed":
                    int(
                        permutation_seed_i
                    ),

                "chaos_state":
                    chaos_state.hex(),

                "feedback_state":
                    feedback_state.hex(),

                "S_next":
                    S_next.hex()
            })

            permutations.append({

                "block_index":
                    block_index,

                "permutation":
                    permutation_i.copy(),

                "inverse_permutation":
                    inverse_permutation_i.copy()
            })

        # ----------------------------------------------------
        # STORE RULE
        # ----------------------------------------------------

        block_rules.append({

            "block_index":
                block_index,

            "rule_index":
                int(r_i),

            "rule":
                rule_i
        })

        # ----------------------------------------------------
        # CIPHERTEXT FEEDBACK FOR NEXT BLOCK
        # ----------------------------------------------------

        previous_cipher_block = (
            C_i.reshape(-1).copy()
        )

        # ----------------------------------------------------
        # ADVANCE STATE
        # ----------------------------------------------------

        S_current = S_next

            # ========================================================
    # FINAL BIDIRECTIONAL CIPHERTEXT DIFFUSION
    # ========================================================
    #
    # The existing block processing produces the forward
    # ciphertext:
    #
    #       plaintext -> forward ciphertext
    #
    # The second pass propagates ciphertext dependence
    # backward through earlier blocks.
    #
    # This does NOT use plaintext.
    # ========================================================

    forward_cipher_image = (
        cipher_image.copy()
    )

    cipher_image = (
        apply_reverse_ciphertext_diffusion(
            cipher_image=forward_cipher_image,
            k_diff=K_DIFF,
            block_size=B
        )
    )

    print(
        "Backward ciphertext diffusion applied."
    )

    # --------------------------------------------------------
    # METADATA
    # --------------------------------------------------------

    metadata = {

        "version":
            CONFIG["VERSION"],

        "image_name":
            str(image_name),

        "height":
            int(H),

        "width":
            int(W),
         "reverse_diffusion":
    "HMAC-SHA256 backward block diffusion v1",

        "block_size":
            int(B),

        "salt":
            salt.hex(),

        "nonce":
            nonce.hex(),

        "content_descriptor":
            descriptor.hex(),

        "content_digest":
            descriptor_digest.hex(),

        "kdf":
            "PBKDF2-HMAC-SHA256",

        "kdf_iterations":
            int(
                CONFIG["KDF_ITERATIONS"]
            ),

        "key_schedule":
            "HKDF-SHA256",

        # Store the actual transient value used so that
        # experiments are reproducible and auditable.
        "chaos_transient":
            int(chaos_transient),

        "feature_values":
            {
                key: float(value)
                for key, value
                in feature_values.items()
            }
    }

    # --------------------------------------------------------
    # AUTHENTICATION TAG
    # --------------------------------------------------------

    tag = hmac.new(
        K_AUTH,
        canonical_header_bytes(
            metadata
        ) +
        np.ascontiguousarray(
            cipher_image
        ).tobytes(),
        hashlib.sha256
    ).hexdigest()

    metadata[
        "authentication_tag"
    ] = tag

    # --------------------------------------------------------
    # TIMING
    # --------------------------------------------------------

    total_time = (
        time.perf_counter()
        -
        total_start
    )

    encryption_state = {

        # Diagnostic alias only.
        #
        # This is the V2 key-derived initial state and is
        # NOT plaintext-derived.
        "S0":
            S0,

        "final_state":
            S_current,

        "metadata":
            metadata,

        "feature_values":
            feature_values,

        "timing":
            {
                "total_encryption_time_seconds":
                    float(total_time),

                "pixels":
                    int(H * W),

                "blocks":
                    int(len(blocks))
            }
    }

    timing = (
        encryption_state[
            "timing"
        ]
    )

    # --------------------------------------------------------
    # RETURN
    # --------------------------------------------------------

    return (
        cipher_image,
        encryption_state,
        block_states,
        block_rules,
        permutations,
        timing
    )


# ============================================================
# CELL 26 STATUS
# ============================================================

print("=" * 80)
print("CELL 26 — V2 ENCRYPTION")
print("=" * 80)

print(
    "Plaintext-dependent bootstrap: REMOVED"
)

print(
    "Cryptographic key schedule: ENABLED"
)

print(
    "Content-adaptive control: ENABLED"
)

print(
    "Ciphertext-only state evolution: ENABLED"
)

print(
    "Authentication tag: ENABLED"
)

print(
    "Chaos transient:",
    int(
        CONFIG.get(
            "CHAOS_TRANSIENT",
            1000
        )
    )
)

print()
print("CELL 26 COMPLETE")

CELL 26 — V2 ENCRYPTION
Plaintext-dependent bootstrap: REMOVED
Cryptographic key schedule: ENABLED
Content-adaptive control: ENABLED
Ciphertext-only state evolution: ENABLED
Authentication tag: ENABLED
Chaos transient: 1000

CELL 26 COMPLETE


In [ ]:
# ============================================================
# CELL 27 — INVERSE FEEDBACK DIFFUSION / DECRYPT BLOCK
# ============================================================

def decrypt_block(
    cipher_block,
    current_state,
    rule,
    feedback_block
):
    """
    Inverse of apply_feedback_diffusion().

    Encryption in CELL 23:

        M_i =
            SHA256-counter(
                S_i || F_i || counter || FEEDBACK_MASK
            )

        D_i^feedback =
            DNAEncode(M_i, R_i)

        Y_i =
            X_i XOR_D D_i^feedback

    Then CELL 26 performs:

        C_i = DNADecode(Y_i, R_i)

    Therefore decryption performs:

        C_i
          ↓
        DNAEncode
          ↓
        Y_i
          ↓
        Generate EXACT SAME M_i
          ↓
        DNAEncode(M_i)
          ↓
        DNA-XOR
          ↓
        X_i

    IMPORTANT:
        The mask generation here MUST be identical to
        generate_feedback_mask() in CELL 23.

        Therefore:
            - no plaintext_block_hash is used
            - the counter is generated exactly as in CELL 23
            - the same FEEDBACK_MASK tag is used
    """

    # ========================================================
    # VALIDATE INPUTS
    # ========================================================

    cipher_block = np.asarray(
        cipher_block,
        dtype=np.uint8
    )

    feedback_block = np.asarray(
        feedback_block,
        dtype=np.uint8
    )

    B = int(
        CONFIG["BLOCK_SIZE"]
    )

    if cipher_block.shape != (
        B,
        B
    ):
        raise ValueError(
            "cipher_block must have shape "
            f"({B}, {B}). Got "
            f"{cipher_block.shape}"
        )

    if feedback_block.size != (
        B * B
    ):
        raise ValueError(
            "feedback_block must contain "
            f"{B * B} bytes. Got "
            f"{feedback_block.size}"
        )

    if not isinstance(
        current_state,
        bytes
    ):
        raise TypeError(
            "current_state must be bytes."
        )

    if len(current_state) != 32:
        raise ValueError(
            "current_state must contain 32 bytes."
        )

    # ========================================================
    # CIPHER BYTE BLOCK
    # ========================================================

    cipher_block = np.ascontiguousarray(
        cipher_block,
        dtype=np.uint8
    )

    feedback_block = np.ascontiguousarray(
        feedback_block.reshape(-1),
        dtype=np.uint8
    )

    # ========================================================
    # C_i → Y_i DNA
    #
    # CELL 26:
    #
    #     Y_i
    #      ↓
    #     DNA decode
    #      ↓
    #     C_i
    #
    # Therefore:
    #
    #     C_i
    #      ↓
    #     DNA encode
    #      ↓
    #     Y_i
    # ========================================================

    cipher_dna = dna_encode_array(
        cipher_block,
        rule
    )

    if cipher_dna.shape != (
        B,
        B,
        4
    ):
        raise RuntimeError(
            "Unexpected cipher DNA shape: "
            f"{cipher_dna.shape}"
        )

    # ========================================================
    # REGENERATE EXACT FEEDBACK MASK
    #
    # THIS MUST MATCH CELL 23 EXACTLY.
    #
    # CELL 23:
    #
    # generate_feedback_mask(
    #     current_state,
    #     feedback_bytes,
    #     block_pixel_count
    # )
    #
    # Internally:
    #
    # SHA256(
    #     current_state
    #     ||
    #     feedback_bytes
    #     ||
    #     counter_8_bytes
    #     ||
    #     b"FEEDBACK_MASK"
    # )
    #
    # ========================================================

    block_pixel_count = (
        B * B
    )

    M_i = generate_feedback_mask(
        current_state=current_state,
        feedback_bytes=feedback_block,
        block_pixel_count=block_pixel_count
    )

    M_i = np.asarray(
        M_i,
        dtype=np.uint8
    )

    if M_i.size != block_pixel_count:
        raise RuntimeError(
            "Feedback mask length mismatch: "
            f"{M_i.size} != "
            f"{block_pixel_count}"
        )

    # ========================================================
    # MASK → DNA
    # ========================================================

    mask_dna = dna_encode_array(
        M_i,
        rule
    )

    mask_dna = mask_dna.reshape(
        B,
        B,
        4
    )

    if mask_dna.shape != (
        B,
        B,
        4
    ):
        raise RuntimeError(
            "Unexpected feedback mask DNA shape: "
            f"{mask_dna.shape}"
        )

    # ========================================================
    # INVERSE DNA-XOR
    #
    # Encryption:
    #
    #     Y_i =
    #         X_i XOR_D mask_dna
    #
    # Since DNA-XOR is self-inverse:
    #
    #     X_i =
    #         Y_i XOR_D mask_dna
    #
    # ========================================================

    X_i = dna_xor(
        cipher_dna,
        mask_dna
    )

    # ========================================================
    # VALIDATE RESULT
    # ========================================================

    if X_i.shape != (
        B,
        B,
        4
    ):
        raise RuntimeError(
            "Unexpected decrypted DNA shape: "
            f"{X_i.shape}"
        )

    # ========================================================
    # RETURN
    # ========================================================

    return (
        X_i,
        M_i,
        mask_dna
    )


print("=" * 80)
print("INVERSE FEEDBACK DIFFUSION / DECRYPT BLOCK")
print("=" * 80)
print()
print("Mask generation source : CELL 23 generate_feedback_mask()")
print("Plaintext hash in mask : NO")
print("Feedback dependency    : YES")
print("Counter size           : 8 bytes")
print("Feedback tag           : FEEDBACK_MASK")
print()
print("CELL 27 definition: READY")


INVERSE FEEDBACK DIFFUSION / DECRYPT BLOCK

Mask generation source : CELL 23 generate_feedback_mask()
Plaintext hash in mask : NO
Feedback dependency    : YES
Counter size           : 8 bytes
Feedback tag           : FEEDBACK_MASK

CELL 27 definition: READY


In [ ]:
# ============================================================
# CELL 28 — NORMAL CIPHERTEXT-ONLY DECRYPTION
# ============================================================

def decrypt_image(
    cipher_image,
    metadata,
    secret_key,
    save_internal=False,
    image_name="image"
):

    total_start = time.perf_counter()

    # --------------------------------------------------------
    # INPUT VALIDATION
    # --------------------------------------------------------

    cipher_image = np.asarray(
        cipher_image,
        dtype=np.uint8
    )

    if cipher_image.ndim != 2:

        raise ValueError(
            "Cipher image must be grayscale 2-D."
        )

    if not isinstance(
        metadata,
        dict
    ):

        raise TypeError(
            "metadata must be a dictionary."
        )

    H, W = cipher_image.shape

    # --------------------------------------------------------
    # REQUIRED METADATA
    # --------------------------------------------------------

    required_metadata = [

        "version",
        "height",
        "width",
        "block_size",
        "salt",
        "nonce",
        "content_descriptor",
        "content_digest",
        "kdf",
        "kdf_iterations",
        "key_schedule",
        "authentication_tag"
    ]

    missing_metadata = [

        key
        for key in required_metadata
        if key not in metadata

    ]

    if missing_metadata:

        raise KeyError(
            "Required metadata fields are missing: "
            +
            ", ".join(
                missing_metadata
            )
        )

    # --------------------------------------------------------
    # IMAGE DIMENSIONS
    # --------------------------------------------------------

    B = int(
        metadata["block_size"]
    )

    metadata_height = int(
        metadata["height"]
    )

    metadata_width = int(
        metadata["width"]
    )

    if (
        H != metadata_height
        or
        W != metadata_width
    ):

        raise ValueError(
            "Cipher dimensions do not match metadata: "
            f"cipher={H}x{W}, "
            f"metadata={metadata_height}x{metadata_width}."
        )

    if B <= 0:

        raise ValueError(
            "Metadata block_size must be positive."
        )

    if H % B != 0 or W % B != 0:

        raise ValueError(
            f"Cipher dimensions {H}x{W} "
            f"must be divisible by block size {B}."
        )

    # --------------------------------------------------------
    # SALT
    # --------------------------------------------------------

    try:

        salt = bytes.fromhex(
            metadata["salt"]
        )

    except Exception as exc:

        raise ValueError(
            "Invalid hexadecimal salt in metadata."
        ) from exc

    expected_salt_length = int(
        CONFIG["SALT_BYTES"]
    )

    if len(salt) != expected_salt_length:

        raise ValueError(
            "Invalid salt length: "
            f"expected {expected_salt_length} bytes, "
            f"got {len(salt)}."
        )

    # --------------------------------------------------------
    # NONCE
    # --------------------------------------------------------

    try:

        nonce = bytes.fromhex(
            metadata["nonce"]
        )

    except Exception as exc:

        raise ValueError(
            "Invalid hexadecimal nonce in metadata."
        ) from exc

    expected_nonce_length = int(
        CONFIG["NONCE_BYTES"]
    )

    if len(nonce) != expected_nonce_length:

        raise ValueError(
            "Invalid nonce length: "
            f"expected {expected_nonce_length} bytes, "
            f"got {len(nonce)}."
        )

    # --------------------------------------------------------
    # CONTENT DIGEST
    # --------------------------------------------------------

    try:

        descriptor_digest = bytes.fromhex(
            metadata["content_digest"]
        )

    except Exception as exc:

        raise ValueError(
            "Invalid hexadecimal content_digest "
            "in metadata."
        ) from exc

    if len(descriptor_digest) != 32:

        raise ValueError(
            "content_digest must contain "
            "a 32-byte SHA-256 digest."
        )

    # --------------------------------------------------------
    # CHAOS TRANSIENT
    #
    # IMPORTANT:
    #
    # CELL 26 stores the actual transient value used
    # during encryption in authenticated metadata:
    #
    #     metadata["chaos_transient"]
    #
    # Therefore decryption uses that exact value.
    #
    # For compatibility with metadata produced before this
    # field was added, fall back to CONFIG and finally 1000.
    # --------------------------------------------------------

    chaos_transient = int(
        metadata.get(
            "chaos_transient",
            CONFIG.get(
                "CHAOS_TRANSIENT",
                1000
            )
        )
    )

    if chaos_transient < 0:

        raise ValueError(
            "chaos_transient must be "
            "a non-negative integer."
        )

    # --------------------------------------------------------
    # KEY SCHEDULE
    # --------------------------------------------------------

    keys = derive_key_schedule(
        secret_key=secret_key,
        salt=salt,
        nonce=nonce,
        descriptor_digest=descriptor_digest
    )

    required_keys = [

        "K_DNA",
        "K_PERM",
        "K_DIFF",
        "K_CHAOS",
        "K_AUTH"

    ]

    missing_keys = [

        key
        for key in required_keys
        if key not in keys

    ]

    if missing_keys:

        raise KeyError(
            "derive_key_schedule() did not return "
            "required keys: "
            +
            ", ".join(
                missing_keys
            )
        )

    K_DNA = keys["K_DNA"]
    K_PERM = keys["K_PERM"]
    K_DIFF = keys["K_DIFF"]
    K_CHAOS = keys["K_CHAOS"]
    K_AUTH = keys["K_AUTH"]

    # --------------------------------------------------------
    # AUTHENTICATION VERIFICATION
    #
    # IMPORTANT:
    #
    # The authentication tag covers:
    #
    #     canonical_header_bytes(metadata)
    #     ||
    #     ciphertext
    #
    # Therefore metadata is authenticated before any
    # ciphertext processing occurs.
    # --------------------------------------------------------

    expected_tag = hmac.new(
        K_AUTH,
        canonical_header_bytes(
            metadata
        )
        +
        np.ascontiguousarray(
            cipher_image
        ).tobytes(),
        hashlib.sha256
    ).hexdigest()

    stored_tag = str(
        metadata["authentication_tag"]
    )

    if not hmac.compare_digest(
        expected_tag,
        stored_tag
    ):

        raise ValueError(
            "Authentication verification failed. "
            "The ciphertext or metadata may have been "
            "modified, or the secret key is incorrect."
        )
        # ========================================================
    # REVERSE FINAL CIPHERTEXT DIFFUSION
    # ========================================================
    #
    # CELL 26 performs:
    #
    #     forward ciphertext
    #             ↓
    #     backward diffusion
    #             ↓
    #     final authenticated ciphertext
    #
    # Therefore CELL 28 must undo that operation BEFORE
    # normal block-by-block V2 decryption.
    # ========================================================

    reverse_diffusion_mode = metadata.get(
        "reverse_diffusion",
        None
    )

    if reverse_diffusion_mode != (
        "HMAC-SHA256 backward block diffusion v1"
    ):

        raise ValueError(
            "Unsupported or missing reverse diffusion mode. "
            "The ciphertext must be generated using the "
            "updated CELL 26."
        )

    cipher_image = (
        undo_reverse_ciphertext_diffusion(
            final_cipher_image=cipher_image,
            k_diff=K_DIFF,
            block_size=B
        )
    )
    # --------------------------------------------------------
    # INITIAL STATE
    #
    # Must exactly match CELL 26.
    #
    # No plaintext is required.
    # --------------------------------------------------------

    S_current = hmac.new(
        K_DIFF,
        nonce +
        descriptor_digest +
        b"INITIAL_STATE",
        hashlib.sha256
    ).digest()

    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------

    decrypted_image = np.zeros_like(
        cipher_image,
        dtype=np.uint8
    )

    # --------------------------------------------------------
    # INITIAL CIPHERTEXT FEEDBACK
    # --------------------------------------------------------

    previous_cipher_block = np.zeros(
        B * B,
        dtype=np.uint8
    )

    # --------------------------------------------------------
    # DIVIDE CIPHERTEXT INTO BLOCKS
    # --------------------------------------------------------

    cipher_blocks = divide_into_blocks(
        cipher_image,
        block_size=B
    )

    block_states = []

    # --------------------------------------------------------
    # PROCESS BLOCKS
    # --------------------------------------------------------

    for cipher_info in cipher_blocks:

        block_index = int(
            cipher_info["index"]
        )

        C_i = np.asarray(
            cipher_info["data"],
            dtype=np.uint8
        )

        if C_i.shape != (B, B):

            raise ValueError(
                f"Invalid ciphertext block shape "
                f"at block {block_index}: "
                f"{C_i.shape}. "
                f"Expected {(B, B)}."
            )

        # ----------------------------------------------------
        # BLOCK INDEX
        # ----------------------------------------------------

        index_bytes = int(
            block_index
        ).to_bytes(
            8,
            byteorder="big",
            signed=False
        )

        # ----------------------------------------------------
        # ADAPTIVE CONTROL
        #
        # Must exactly match CELL 26.
        # ----------------------------------------------------

        Z_i = hmac.new(
            K_DNA,
            S_current +
            previous_cipher_block.tobytes() +
            descriptor_digest +
            index_bytes +
            b"ADAPTIVE_CONTROL",
            hashlib.sha256
        ).digest()

        (
            q_i,
            r_i,
            rule_i
        ) = select_adaptive_dna_rule(
            Z_i
        )

        # ----------------------------------------------------
        # PERMUTATION
        # ----------------------------------------------------

        permutation_state = hmac.new(
            K_PERM,
            S_current +
            descriptor_digest +
            index_bytes +
            b"PERMUTATION_STATE",
            hashlib.sha256
        ).digest()

        (
            permutation_i,
            inverse_permutation_i,
            permutation_seed_i
        ) = generate_block_permutation(
            S_i=permutation_state,
            block_index=block_index,
            block_size=B
        )

        # ----------------------------------------------------
        # BLOCK-SPECIFIC CHAOTIC STREAM
        #
        # Must exactly match CELL 26.
        # ----------------------------------------------------

        chaos_state = hmac.new(
            K_CHAOS,
            S_current +
            descriptor_digest +
            index_bytes +
            b"CHAOS",
            hashlib.sha256
        ).digest()

        (
            x0_i,
            r_i_chaos,
            _,
            _
        ) = state_to_chaotic_parameters(
            chaos_state
        )

        chaos_sequence = generate_logistic_sequence(
            x0=x0_i,
            r=r_i_chaos,
            sequence_length=B * B,
            transient=chaos_transient
        )

        chaos_bytes = np.asarray(
            chaotic_to_bytes(
                chaos_sequence
            ),
            dtype=np.uint8
        ).reshape(-1)

        if chaos_bytes.size != B * B:

            raise ValueError(
                "Chaotic byte stream length mismatch: "
                f"expected {B * B}, "
                f"got {chaos_bytes.size}."
            )

        # ----------------------------------------------------
        # CHAOS DNA
        # ----------------------------------------------------

        chaos_dna = dna_encode_array(
            chaos_bytes,
            rule_i
        )

        chaos_dna = chaos_dna.reshape(
            B,
            B,
            4
        )

        # ----------------------------------------------------
        # FEEDBACK STATE
        #
        # Must exactly match CELL 26.
        # ----------------------------------------------------

        feedback_state = hmac.new(
            K_DIFF,
            S_current +
            descriptor_digest +
            index_bytes +
            b"FEEDBACK_STATE",
            hashlib.sha256
        ).digest()

        (
            X_i,
            _,
            _
        ) = decrypt_block(
            cipher_block=C_i,
            current_state=feedback_state,
            rule=rule_i,
            feedback_block=previous_cipher_block
        )

        # ----------------------------------------------------
        # REVERSE DNA XOR
        # ----------------------------------------------------

        permuted_dna = dna_xor(
            X_i,
            chaos_dna
        )

        # ----------------------------------------------------
        # DNA DECODE
        # ----------------------------------------------------

        B_i_permuted = dna_decode_array(
            permuted_dna,
            rule_i
        ).astype(
            np.uint8
        )

        B_i_permuted = (
            B_i_permuted.reshape(
                B,
                B
            )
        )

        # ----------------------------------------------------
        # REVERSE PERMUTATION
        # ----------------------------------------------------

        B_i = inverse_permute_block(
            B_i_permuted,
            inverse_permutation_i
        )

        B_i = np.asarray(
            B_i,
            dtype=np.uint8
        )

        # ----------------------------------------------------
        # STORE PLAINTEXT BLOCK
        # ----------------------------------------------------

        decrypted_image[
            cipher_info["y0"]:cipher_info["y1"],
            cipher_info["x0"]:cipher_info["x1"]
        ] = B_i

        # ----------------------------------------------------
        # CIPHERTEXT-ONLY STATE UPDATE
        #
        # IMPORTANT:
        #
        # The state evolves using C_i, exactly as in
        # CELL 26.
        # ----------------------------------------------------

        S_next = evolve_state_v2(
            current_state=S_current,
            cipher_block=C_i,
            block_index=block_index,
            k_diff=K_DIFF
        )

        # ----------------------------------------------------
        # INTERNAL DIAGNOSTICS
        # ----------------------------------------------------

        if save_internal:

            block_states.append({

                "block_index":
                    block_index,

                "S_i":
                    S_current.hex(),

                "Z_i":
                    Z_i.hex(),

                "rule_index":
                    int(r_i),

                "rule":
                    rule_i,

                "permutation_seed":
                    int(
                        permutation_seed_i
                    ),

                "chaos_state":
                    chaos_state.hex(),

                "feedback_state":
                    feedback_state.hex(),

                "S_next":
                    S_next.hex()
            })

        # ----------------------------------------------------
        # CIPHERTEXT FEEDBACK
        # ----------------------------------------------------

        previous_cipher_block = (
            C_i.reshape(-1).copy()
        )

        # ----------------------------------------------------
        # ADVANCE STATE
        # ----------------------------------------------------

        S_current = S_next

    # --------------------------------------------------------
    # TIMING
    # --------------------------------------------------------

    decryption_time = (
        time.perf_counter()
        -
        total_start
    )

    # --------------------------------------------------------
    # DECRYPTION STATE
    # --------------------------------------------------------

    decryption_state = {

        "final_state":
            S_current,

        "metadata":
            metadata,

        "timing":
            {
                "total_decryption_time_seconds":
                    float(decryption_time)
            }
    }

    # --------------------------------------------------------
    # RETURN
    # --------------------------------------------------------

    return (
        decrypted_image,
        decryption_state,
        block_states,
        {
            "total_decryption_time_seconds":
                float(decryption_time)
        }
    )


# ============================================================
# CELL 28 STATUS
# ============================================================

print("=" * 80)
print("CELL 28 — CIPHERTEXT-ONLY DECRYPTION")
print("=" * 80)

print(
    "Original plaintext required: NO"
)

print(
    "Metadata required: YES"
)

print(
    "Authentication verification: YES"
)

print(
    "Chaos transient source: "
    "authenticated metadata "
    "→ CONFIG fallback "
    "→ 1000"
)

print()
print("CELL 28 COMPLETE")

CELL 28 — CIPHERTEXT-ONLY DECRYPTION
Original plaintext required: NO
Metadata required: YES
Authentication verification: YES
Chaos transient source: authenticated metadata → CONFIG fallback → 1000

CELL 28 COMPLETE


In [ ]:
# ============================================================
# CELL 29 — EXACT RECONSTRUCTION VALIDATION
# ============================================================

# ------------------------------------------------------------
# Standalone deterministic test secret
# ------------------------------------------------------------

_TEST_SECRET_KEY_CELL29 = (
    "DNA-Adaptive-V2-Test-Key-2026"
)

# Use the existing test image.
_TEST_IMAGE_CELL29 = np.asarray(
    test_image,
    dtype=np.uint8
).copy()


# ------------------------------------------------------------
# Use the existing image path when available
# ------------------------------------------------------------

try:
    _TEST_IMAGE_NAME_CELL29 = (
        test_image_path.stem
    )
except NameError:
    _TEST_IMAGE_NAME_CELL29 = "test_image"


# ============================================================
# LOCAL METRIC HELPERS
# ============================================================

def _calculate_mse_cell29(
    original,
    recovered
):
    """
    Mean Squared Error.
    """

    original = np.asarray(
        original,
        dtype=np.float64
    )

    recovered = np.asarray(
        recovered,
        dtype=np.float64
    )

    if original.shape != recovered.shape:
        raise ValueError(
            "Original and recovered images must have "
            "the same shape."
        )

    mse_value = np.mean(
        (original - recovered) ** 2
    )

    return float(mse_value)


def _calculate_psnr_cell29(
    original,
    recovered,
    max_pixel_value=255.0
):
    """
    Peak Signal-to-Noise Ratio.

    Returns infinity when MSE = 0.
    """

    mse_value = _calculate_mse_cell29(
        original,
        recovered
    )

    if mse_value == 0.0:
        return float("inf")

    return float(
        10.0 *
        np.log10(
            (max_pixel_value ** 2) /
            mse_value
        )
    )


# ============================================================
# BASIC INPUT VALIDATION
# ============================================================

if _TEST_IMAGE_CELL29.ndim != 2:
    raise ValueError(
        "Cell 29 requires a grayscale 2-D image."
    )

if _TEST_IMAGE_CELL29.dtype != np.uint8:
    raise TypeError(
        "Test image must use dtype uint8."
    )


# ============================================================
# ENCRYPTION
# ============================================================

(
    test_cipher,
    test_encryption_state,
    test_block_states,
    test_block_rules,
    test_permutations,
    test_timing
) = encrypt_image(
    image=_TEST_IMAGE_CELL29,
    secret_key=_TEST_SECRET_KEY_CELL29,
    save_internal=True,
    image_name=_TEST_IMAGE_NAME_CELL29
)


# ============================================================
# EXTRACT AUTHENTICATED METADATA
# ============================================================

if not isinstance(
    test_encryption_state,
    dict
):
    raise TypeError(
        "encrypt_image() must return a dictionary "
        "as test_encryption_state."
    )

if "metadata" not in test_encryption_state:
    raise KeyError(
        "Encryption state does not contain 'metadata'."
    )

test_metadata = (
    test_encryption_state["metadata"]
)


# ============================================================
# CIPHERTEXT-ONLY DECRYPTION
# ============================================================

(
    test_decrypted,
    test_decryption_state,
    test_decryption_blocks,
    test_decryption_timing
) = decrypt_image(
    cipher_image=test_cipher,
    metadata=test_metadata,
    secret_key=_TEST_SECRET_KEY_CELL29,
    save_internal=True,
    image_name=_TEST_IMAGE_NAME_CELL29
)


# ============================================================
# NORMALIZE OUTPUT
# ============================================================

test_decrypted = np.asarray(
    test_decrypted,
    dtype=np.uint8
)


# ============================================================
# SHAPE VALIDATION
# ============================================================

if (
    test_decrypted.shape !=
    _TEST_IMAGE_CELL29.shape
):
    raise AssertionError(
        "Decrypted image shape does not match "
        "the original image shape."
    )


# ============================================================
# EXACT RECONSTRUCTION METRICS
# ============================================================

mse = _calculate_mse_cell29(
    _TEST_IMAGE_CELL29,
    test_decrypted
)

psnr = _calculate_psnr_cell29(
    _TEST_IMAGE_CELL29,
    test_decrypted
)

different_pixels = int(
    np.count_nonzero(
        _TEST_IMAGE_CELL29 != test_decrypted
    )
)

exact_reconstruction = bool(
    np.array_equal(
        _TEST_IMAGE_CELL29,
        test_decrypted
    )
)


# ============================================================
# REPORT
# ============================================================

print("=" * 80)
print("CELL 29 — EXACT RECONSTRUCTION")
print("=" * 80)

print(
    f"Image shape        : "
    f"{_TEST_IMAGE_CELL29.shape}"
)

print(
    f"MSE                : "
    f"{mse}"
)

print(
    f"PSNR               : "
    f"{psnr}"
)

print(
    f"Different pixels   : "
    f"{different_pixels}"
)

print(
    f"Exact reconstruction: "
    f"{exact_reconstruction}"
)


# ============================================================
# REQUIRED LOSSLESS CONDITIONS
# ============================================================

assert test_decrypted.shape == (
    _TEST_IMAGE_CELL29.shape
)

assert exact_reconstruction, (
    "Ciphertext-only decryption did not exactly "
    "reconstruct the original image."
)

assert different_pixels == 0, (
    f"Expected 0 different pixels, got "
    f"{different_pixels}."
)

assert mse == 0.0, (
    f"Expected MSE = 0.0, got {mse}."
)

assert np.isinf(psnr), (
    f"Expected infinite PSNR for exact recovery, "
    f"got {psnr}."
)


print()
print(
    "LOSSLESS CIPHERTEXT-ONLY RECOVERY: PASSED"
)

Backward ciphertext diffusion applied.
CELL 29 — EXACT RECONSTRUCTION
Image shape        : (256, 256)
MSE                : 0.0
PSNR               : inf
Different pixels   : 0
Exact reconstruction: True

LOSSLESS CIPHERTEXT-ONLY RECOVERY: PASSED


In [ ]:
# ============================================================
# CELL 29 — SAVE ENCRYPTED AND DECRYPTED IMAGES
# ============================================================

from pathlib import Path
import cv2
import numpy as np


# ============================================================
# RESULT DIRECTORIES
# ============================================================

RESULT_ROOT = Path(
    "/content/drive/MyDrive/DNA Adaptive RESULT"
)

ENCRYPTION_DIR = (
    RESULT_ROOT /
    "06_ENCRYPTION" /
    "cipher_images"
)

DECRYPTION_DIR = (
    RESULT_ROOT /
    "07_DECRYPTION" /
    "decrypted_images"
)

ENCRYPTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DECRYPTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# OUTPUT FILENAMES
# ============================================================

image_stem = test_image_path.stem

cipher_output_path = (
    ENCRYPTION_DIR /
    f"{image_stem}_cipher.png"
)

decrypted_output_path = (
    DECRYPTION_DIR /
    f"{image_stem}_decrypted.png"
)


# ============================================================
# SAVE CIPHER
# ============================================================

cipher_saved = cv2.imwrite(
    str(cipher_output_path),
    test_cipher
)

if not cipher_saved:
    raise IOError(
        f"Could not save encrypted image: "
        f"{cipher_output_path}"
    )


# ============================================================
# SAVE DECRYPTED IMAGE
# ============================================================

decrypted_saved = cv2.imwrite(
    str(decrypted_output_path),
    test_decrypted
)

if not decrypted_saved:
    raise IOError(
        f"Could not save decrypted image: "
        f"{decrypted_output_path}"
    )


# ============================================================
# EXACT RECOVERY METRICS
# ============================================================

absolute_difference = np.abs(
    test_image.astype(np.int16)
    -
    test_decrypted.astype(np.int16)
)

different_pixels = int(
    np.count_nonzero(
        absolute_difference
    )
)

max_difference = int(
    absolute_difference.max()
)

mse = float(
    np.mean(
        (
            test_image.astype(np.float64)
            -
            test_decrypted.astype(np.float64)
        ) ** 2
    )
)

exact_recovery = bool(
    np.array_equal(
        test_image,
        test_decrypted
    )
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 80)
print("ENCRYPTION / DECRYPTION OUTPUT")
print("=" * 80)

print(
    f"Encrypted image : "
    f"{cipher_output_path}"
)

print(
    f"Decrypted image : "
    f"{decrypted_output_path}"
)

print()
print(
    f"Different pixels : "
    f"{different_pixels}"
)

print(
    f"Maximum error    : "
    f"{max_difference}"
)

print(
    f"MSE              : "
    f"{mse:.12f}"
)

print(
    f"Exact recovery   : "
    f"{exact_recovery}"
)


# ============================================================
# FINAL ASSERTIONS
# ============================================================

assert different_pixels == 0
assert max_difference == 0
assert mse == 0.0
assert exact_recovery is True

print()
print("Encrypted image saved: PASSED")
print("Decrypted image saved: PASSED")
print("Original = Decrypted: PASSED")

ENCRYPTION / DECRYPTION OUTPUT
Encrypted image : /content/drive/MyDrive/DNA Adaptive RESULT/06_ENCRYPTION/cipher_images/Copy of Copy of image_0050_cipher.png
Decrypted image : /content/drive/MyDrive/DNA Adaptive RESULT/07_DECRYPTION/decrypted_images/Copy of Copy of image_0050_decrypted.png

Different pixels : 0
Maximum error    : 0
MSE              : 0.000000000000
Exact recovery   : True

Encrypted image saved: PASSED
Decrypted image saved: PASSED
Original = Decrypted: PASSED


In [ ]:
# ============================================================
# CELL 30 — ENCRYPTION / DECRYPTION TIMING — V2
# ============================================================

def measure_encryption_time(
    image,
    secret_key
):
    """
    Measure ONLY the actual V2 encryption operation.

    Excluded:
        - Drive mounting
        - Image loading
        - Image display
        - CSV writing
        - Plotting
    """

    start_time = time.perf_counter()

    (
        cipher_image,
        encryption_state,
        block_states,
        block_rules,
        permutations,
        _
    ) = encrypt_image(
        image=image,
        secret_key=secret_key,
        save_internal=False
    )

    end_time = time.perf_counter()

    encryption_time = (
        end_time - start_time
    )

    return (
        cipher_image,
        encryption_state,
        encryption_time
    )


def measure_decryption_time(
    cipher_image,
    metadata,
    secret_key
):
    """
    Measure ONLY the actual V2 ciphertext-only
    decryption operation.

    V2 decryption requires:
        - Ciphertext
        - Authenticated metadata
        - User secret key

    It does NOT require:
        - Original plaintext
        - Known plaintext bootstrap
        - S0 = SHA256(K_user || SHA256(P))

    Excluded:
        - Drive mounting
        - Image loading
        - Image display
        - CSV writing
        - Plotting
    """

    start_time = time.perf_counter()

    (
        decrypted_image,
        decryption_state,
        block_states,
        timing
    ) = decrypt_image(
        cipher_image=cipher_image,
        metadata=metadata,
        secret_key=secret_key,
        save_internal=False
    )

    end_time = time.perf_counter()

    decryption_time = (
        end_time - start_time
    )

    return (
        decrypted_image,
        decryption_state,
        decryption_time
    )


# ------------------------------------------------------------
# Timing test
# ------------------------------------------------------------

# Local deterministic test key.
# Do NOT depend on the old USER_SECRET global.
_TEST_SECRET_KEY_CELL30 = (
    "DNA-Adaptive-V2-Test-Key-2026"
)


# ------------------------------------------------------------
# Measure V2 encryption
# ------------------------------------------------------------

(
    timed_cipher,
    timed_encryption_state,
    T_enc
) = measure_encryption_time(
    image=test_image,
    secret_key=_TEST_SECRET_KEY_CELL30
)


# ------------------------------------------------------------
# Obtain authenticated V2 metadata
# ------------------------------------------------------------

timed_metadata = (
    timed_encryption_state["metadata"]
)


# ------------------------------------------------------------
# Measure V2 ciphertext-only decryption
# ------------------------------------------------------------

(
    timed_decrypted,
    timed_decryption_state,
    T_dec
) = measure_decryption_time(
    cipher_image=timed_cipher,
    metadata=timed_metadata,
    secret_key=_TEST_SECRET_KEY_CELL30
)


# ------------------------------------------------------------
# Verify exact reconstruction
# ------------------------------------------------------------

timed_decrypted = np.asarray(
    timed_decrypted,
    dtype=np.uint8
)

timed_original = np.asarray(
    test_image,
    dtype=np.uint8
)

timed_different_pixels = int(
    np.count_nonzero(
        timed_original != timed_decrypted
    )
)

timed_exact_recovery = bool(
    np.array_equal(
        timed_original,
        timed_decrypted
    )
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 80)
print("V2 ENCRYPTION / DECRYPTION TIMING")
print("=" * 80)

print(
    f"Image size       : "
    f"{test_image.shape[0]} × {test_image.shape[1]}"
)

print(
    f"Pixels           : "
    f"{test_image.size}"
)

print()

print(
    f"T_enc            : "
    f"{T_enc:.8f} seconds"
)

print(
    f"T_dec            : "
    f"{T_dec:.8f} seconds"
)

print()

print(
    f"Different pixels : "
    f"{timed_different_pixels}"
)

print(
    f"Exact recovery   : "
    f"{timed_exact_recovery}"
)

print()

print("Decryption mode  : Ciphertext-only V2")

print()

print("Timing excludes:")
print("  - Drive mounting")
print("  - Image loading")
print("  - Image display")
print("  - CSV writing")
print("  - Plotting")

Backward ciphertext diffusion applied.
V2 ENCRYPTION / DECRYPTION TIMING
Image size       : 256 × 256
Pixels           : 65536

T_enc            : 0.74441374 seconds
T_dec            : 0.76114654 seconds

Different pixels : 0
Exact recovery   : True

Decryption mode  : Ciphertext-only V2

Timing excludes:
  - Drive mounting
  - Image loading
  - Image display
  - CSV writing
  - Plotting


In [ ]:
# ============================================================
# CELL 31 — ENTROPY
# ============================================================

def calculate_entropy(image):
    """
    Shannon entropy:

        H(C) = -Σ p_i log2(p_i)

    where p_i is the probability of gray level i.

    Maximum entropy for an 8-bit grayscale image:

        H_max = 8 bits
    """

    image = np.asarray(
        image,
        dtype=np.uint8
    )

    if image.size == 0:
        raise ValueError(
            "Image cannot be empty."
        )

    histogram = np.bincount(
        image.reshape(-1),
        minlength=256
    ).astype(np.float64)

    probabilities = (
        histogram / image.size
    )

    nonzero_probabilities = (
        probabilities[
            probabilities > 0
        ]
    )

    entropy = -np.sum(
        nonzero_probabilities *
        np.log2(
            nonzero_probabilities
        )
    )

    return float(entropy)


# ------------------------------------------------------------
# Entropy test
# ------------------------------------------------------------

original_entropy = calculate_entropy(
    test_image
)

cipher_entropy = calculate_entropy(
    timed_cipher
)

decrypted_entropy = calculate_entropy(
    timed_decrypted
)

print("=" * 80)
print("SHANNON ENTROPY")
print("=" * 80)

print(
    f"Original entropy  : "
    f"{original_entropy:.10f} bits"
)

print(
    f"Cipher entropy    : "
    f"{cipher_entropy:.10f} bits"
)

print(
    f"Decrypted entropy : "
    f"{decrypted_entropy:.10f} bits"
)

print()
print("Maximum possible entropy for 8-bit grayscale: 8 bits")

SHANNON ENTROPY
Original entropy  : 6.9838301345 bits
Cipher entropy    : 7.9972930888 bits
Decrypted entropy : 6.9838301345 bits

Maximum possible entropy for 8-bit grayscale: 8 bits


In [ ]:
# ============================================================
# CELL 32 — HISTOGRAM
# ============================================================

def calculate_histogram(image):
    """
    Calculate the 256-level grayscale histogram.
    """

    image = np.asarray(
        image,
        dtype=np.uint8
    )

    histogram = np.bincount(
        image.reshape(-1),
        minlength=256
    )

    return histogram


def save_histogram_plot(
    original,
    cipher,
    decrypted,
    output_path,
    image_name="image"
):
    """
    Save original, cipher and decrypted histograms.

    This function is deliberately separate from the timing
    measurements so plotting is not included in T_enc/T_dec.
    """

    original_hist = calculate_histogram(
        original
    )

    cipher_hist = calculate_histogram(
        cipher
    )

    decrypted_hist = calculate_histogram(
        decrypted
    )

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(12, 10)
    )

    axes[0].bar(
        np.arange(256),
        original_hist,
        width=1.0
    )

    axes[0].set_title(
        f"{image_name} — Original Histogram"
    )

    axes[0].set_xlabel(
        "Gray Level"
    )

    axes[0].set_ylabel(
        "Frequency"
    )

    axes[1].bar(
        np.arange(256),
        cipher_hist,
        width=1.0
    )

    axes[1].set_title(
        f"{image_name} — Cipher Histogram"
    )

    axes[1].set_xlabel(
        "Gray Level"
    )

    axes[1].set_ylabel(
        "Frequency"
    )

    axes[2].bar(
        np.arange(256),
        decrypted_hist,
        width=1.0
    )

    axes[2].set_title(
        f"{image_name} — Decrypted Histogram"
    )

    axes[2].set_xlabel(
        "Gray Level"
    )

    axes[2].set_ylabel(
        "Frequency"
    )

    plt.tight_layout()

    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    return (
        original_hist,
        cipher_hist,
        decrypted_hist
    )


# ------------------------------------------------------------
# Histogram calculation
# ------------------------------------------------------------

original_histogram = calculate_histogram(
    test_image
)

cipher_histogram = calculate_histogram(
    timed_cipher
)

decrypted_histogram = calculate_histogram(
    timed_decrypted
)

print("=" * 80)
print("HISTOGRAM ANALYSIS")
print("=" * 80)

print(
    "Original histogram calculated :",
    original_histogram.shape
)

print(
    "Cipher histogram calculated   :",
    cipher_histogram.shape
)

print(
    "Decrypted histogram calculated:",
    decrypted_histogram.shape
)

assert len(original_histogram) == 256
assert len(cipher_histogram) == 256
assert len(decrypted_histogram) == 256

print()
print("Histogram calculation: PASSED")

HISTOGRAM ANALYSIS
Original histogram calculated : (256,)
Cipher histogram calculated   : (256,)
Decrypted histogram calculated: (256,)

Histogram calculation: PASSED


In [ ]:
# ============================================================
# CELL 33 — PIXEL CORRELATION
# ============================================================

def calculate_correlation(
    x,
    y
):
    """
    Pearson correlation coefficient:

        r = cov(x,y) / (σ_x σ_y)
    """

    x = np.asarray(
        x,
        dtype=np.float64
    ).reshape(-1)

    y = np.asarray(
        y,
        dtype=np.float64
    ).reshape(-1)

    if x.size != y.size:
        raise ValueError(
            "Input arrays must contain the same number of values."
        )

    if x.size < 2:
        return 0.0

    x_std = np.std(x)
    y_std = np.std(y)

    if x_std == 0 or y_std == 0:
        return 0.0

    return float(
        np.corrcoef(
            x,
            y
        )[0, 1]
    )


def calculate_directional_correlations(
    image
):
    """
    Calculate horizontal, vertical and diagonal
    adjacent-pixel correlations.

    Horizontal:
        (P[i,j], P[i,j+1])

    Vertical:
        (P[i,j], P[i+1,j])

    Diagonal:
        (P[i,j], P[i+1,j+1])
    """

    image = np.asarray(
        image,
        dtype=np.uint8
    )

    H, W = image.shape

    # --------------------------------------------------------
    # Horizontal
    # --------------------------------------------------------

    horizontal_x = image[:, :-1]
    horizontal_y = image[:, 1:]

    corr_h = calculate_correlation(
        horizontal_x,
        horizontal_y
    )

    # --------------------------------------------------------
    # Vertical
    # --------------------------------------------------------

    vertical_x = image[:-1, :]
    vertical_y = image[1:, :]

    corr_v = calculate_correlation(
        vertical_x,
        vertical_y
    )

    # --------------------------------------------------------
    # Diagonal
    # --------------------------------------------------------

    diagonal_x = image[:-1, :-1]
    diagonal_y = image[1:, 1:]

    corr_d = calculate_correlation(
        diagonal_x,
        diagonal_y
    )

    return {
        "Horizontal": corr_h,
        "Vertical": corr_v,
        "Diagonal": corr_d
    }


# ------------------------------------------------------------
# Calculate correlations
# ------------------------------------------------------------

original_correlation = (
    calculate_directional_correlations(
        test_image
    )
)

cipher_correlation = (
    calculate_directional_correlations(
        timed_cipher
    )
)

decrypted_correlation = (
    calculate_directional_correlations(
        timed_decrypted
    )
)

print("=" * 80)
print("PIXEL CORRELATION")
print("=" * 80)

print()
print("Original:")
print(
    f"  Horizontal : "
    f"{original_correlation['Horizontal']:.10f}"
)
print(
    f"  Vertical   : "
    f"{original_correlation['Vertical']:.10f}"
)
print(
    f"  Diagonal   : "
    f"{original_correlation['Diagonal']:.10f}"
)

print()
print("Cipher:")
print(
    f"  Horizontal : "
    f"{cipher_correlation['Horizontal']:.10f}"
)
print(
    f"  Vertical   : "
    f"{cipher_correlation['Vertical']:.10f}"
)
print(
    f"  Diagonal   : "
    f"{cipher_correlation['Diagonal']:.10f}"
)

print()
print("Decrypted:")
print(
    f"  Horizontal : "
    f"{decrypted_correlation['Horizontal']:.10f}"
)
print(
    f"  Vertical   : "
    f"{decrypted_correlation['Vertical']:.10f}"
)
print(
    f"  Diagonal   : "
    f"{decrypted_correlation['Diagonal']:.10f}"
)

PIXEL CORRELATION

Original:
  Horizontal : 0.9487960825
  Vertical   : 0.9589863595
  Diagonal   : 0.9370513582

Cipher:
  Horizontal : -0.0055385435
  Vertical   : 0.0037407344
  Diagonal   : 0.0051675689

Decrypted:
  Horizontal : 0.9487960825
  Vertical   : 0.9589863595
  Diagonal   : 0.9370513582


In [ ]:
# ============================================================
# CELL 34 — NPCR — V2
# ============================================================

def create_one_pixel_modified_image(
    image,
    row=None,
    col=None,
    new_value=None
):
    """
    Create P2 from P1 by modifying exactly one pixel.

    P2 differs from P1 at exactly one location.
    """

    image = np.asarray(
        image,
        dtype=np.uint8
    )

    if image.ndim != 2:
        raise ValueError(
            "NPCR requires a 2-D grayscale image."
        )

    modified = image.copy()

    H, W = image.shape

    if row is None:
        row = H // 2

    if col is None:
        col = W // 2

    if not (
        0 <= row < H and
        0 <= col < W
    ):
        raise ValueError(
            "Pixel coordinates are outside image dimensions."
        )

    original_value = int(
        modified[row, col]
    )

    if new_value is None:

        # Guarantee that the modified value differs
        # from the original.
        new_value = (
            original_value + 1
        ) % 256

    new_value = int(
        new_value
    )

    if not 0 <= new_value <= 255:
        raise ValueError(
            "new_value must be in [0,255]."
        )

    if new_value == original_value:
        raise ValueError(
            "Modified pixel must differ from original pixel."
        )

    modified[row, col] = np.uint8(
        new_value
    )

    return modified


def calculate_npcr(
    cipher_1,
    cipher_2
):
    """
    Number of Pixels Change Rate:

        D(i,j) =
            1, C1(i,j) != C2(i,j)
            0, otherwise

        NPCR =
            [Σ D(i,j) / (H×W)] × 100
    """

    cipher_1 = np.asarray(
        cipher_1,
        dtype=np.uint8
    )

    cipher_2 = np.asarray(
        cipher_2,
        dtype=np.uint8
    )

    if cipher_1.ndim != 2 or cipher_2.ndim != 2:
        raise ValueError(
            "NPCR requires 2-D grayscale cipher images."
        )

    if cipher_1.shape != cipher_2.shape:
        raise ValueError(
            "Cipher images must have identical dimensions."
        )

    difference_map = (
        cipher_1 != cipher_2
    )

    changed_pixels = int(
        np.count_nonzero(
            difference_map
        )
    )

    total_pixels = cipher_1.size

    if total_pixels == 0:
        raise ValueError(
            "Cipher image must contain at least one pixel."
        )

    npcr = (
        changed_pixels /
        total_pixels
    ) * 100.0

    return (
        float(npcr),
        changed_pixels,
        difference_map
    )


# ------------------------------------------------------------
# Local V2 test key
# ------------------------------------------------------------

# Do NOT depend on the removed USER_SECRET global.
_TEST_SECRET_KEY_CELL34 = (
    "DNA-Adaptive-V2-Test-Key-2026"
)


# ------------------------------------------------------------
# Create one-pixel plaintext modification
# ------------------------------------------------------------

plaintext_1 = np.asarray(
    test_image,
    dtype=np.uint8
).copy()

plaintext_2 = create_one_pixel_modified_image(
    plaintext_1
)


plaintext_difference = int(
    np.count_nonzero(
        plaintext_1 != plaintext_2
    )
)

assert plaintext_difference == 1


# ------------------------------------------------------------
# Encrypt both images using EXACTLY the same key
#
# Important:
# Each encrypt_image() call intentionally creates its
# own random salt and nonce in the production V2 path.
# Therefore this measures the complete production behavior
# when the plaintext changes by one pixel.
# ------------------------------------------------------------

start_npcr_1 = time.perf_counter()

(
    cipher_1,
    encryption_state_1,
    _,
    _,
    _,
    _
) = encrypt_image(
    image=plaintext_1,
    secret_key=_TEST_SECRET_KEY_CELL34,
    save_internal=False
)

end_npcr_1 = time.perf_counter()


start_npcr_2 = time.perf_counter()

(
    cipher_2,
    encryption_state_2,
    _,
    _,
    _,
    _
) = encrypt_image(
    image=plaintext_2,
    secret_key=_TEST_SECRET_KEY_CELL34,
    save_internal=False
)

end_npcr_2 = time.perf_counter()


# ------------------------------------------------------------
# Calculate NPCR
# ------------------------------------------------------------

(
    npcr_value,
    changed_pixels,
    npcr_map
) = calculate_npcr(
    cipher_1,
    cipher_2
)


# ------------------------------------------------------------
# Timing
# ------------------------------------------------------------

npcr_encryption_time_1 = (
    end_npcr_1 -
    start_npcr_1
)

npcr_encryption_time_2 = (
    end_npcr_2 -
    start_npcr_2
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 80)
print("NPCR — PLAINTEXT SENSITIVITY — V2")
print("=" * 80)

print(
    "Plaintext difference:",
    plaintext_difference,
    "pixel"
)

print(
    "Changed cipher pixels:",
    changed_pixels
)

print(
    "Total cipher pixels:",
    cipher_1.size
)

print(
    f"NPCR: {npcr_value:.10f}%"
)

print()

print(
    "Encryption time P1:",
    f"{npcr_encryption_time_1:.8f} seconds"
)

print(
    "Encryption time P2:",
    f"{npcr_encryption_time_2:.8f} seconds"
)

print()

print(
    "Reference value often used for 256×256, "
    "8-bit images:"
)

print(
    "≈ 99.6094%"
)

print()

print(
    "Measured NPCR is reported as obtained; "
    "it is not forced to a target value."
)

Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
NPCR — PLAINTEXT SENSITIVITY — V2
Plaintext difference: 1 pixel
Changed cipher pixels: 65299
Total cipher pixels: 65536
NPCR: 99.6383666992%

Encryption time P1: 0.76355082 seconds
Encryption time P2: 0.78839834 seconds

Reference value often used for 256×256, 8-bit images:
≈ 99.6094%

Measured NPCR is reported as obtained; it is not forced to a target value.


In [ ]:
# ============================================================
# CELL 35 — UACI
# ============================================================

def calculate_uaci(
    cipher_1,
    cipher_2
):
    """
    Unified Average Changing Intensity:

        UACI =
        [1/(H×W)] Σ
        |C1(i,j)-C2(i,j)| / 255 × 100
    """

    cipher_1 = np.asarray(
        cipher_1,
        dtype=np.float64
    )

    cipher_2 = np.asarray(
        cipher_2,
        dtype=np.float64
    )

    if cipher_1.shape != cipher_2.shape:
        raise ValueError(
            "Cipher images must have identical dimensions."
        )

    absolute_difference = np.abs(
        cipher_1 -
        cipher_2
    )

    uaci = (
        np.mean(
            absolute_difference
        ) /
        255.0
    ) * 100.0

    return float(
        uaci
    )


# ------------------------------------------------------------
# Calculate UACI using the two NPCR ciphers
# ------------------------------------------------------------

uaci_value = calculate_uaci(
    cipher_1,
    cipher_2
)

print("=" * 80)
print("UACI — PLAINTEXT SENSITIVITY")
print("=" * 80)

print(
    f"UACI: {uaci_value:.10f}%"
)

print()
print(
    "Reference value often used for 8-bit images:"
)
print(
    "≈ 33.46%"
)

print()
print(
    "Measured UACI is reported as obtained; "
    "it is not forced to a target value."
)

UACI — PLAINTEXT SENSITIVITY
UACI: 33.5089709712%

Reference value often used for 8-bit images:
≈ 33.46%

Measured UACI is reported as obtained; it is not forced to a target value.


In [ ]:
# ============================================================
# CELL 36 — NPCR/UACI SPATIAL PROPAGATION EXPERIMENT
# ============================================================

def block_index_to_position(
    block_index,
    image_shape,
    block_size
):
    """
    Convert 1-based block index to (row, col) block position.
    """

    H, W = image_shape

    blocks_per_row = W // block_size
    blocks_per_col = H // block_size

    total_blocks = (
        blocks_per_row *
        blocks_per_col
    )

    if not (
        1 <= block_index <= total_blocks
    ):
        raise ValueError(
            f"Block index must be between 1 and "
            f"{total_blocks}."
        )

    zero_index = block_index - 1

    block_row = (
        zero_index // blocks_per_row
    )

    block_col = (
        zero_index % blocks_per_row
    )

    y0 = block_row * block_size
    x0 = block_col * block_size

    return (
        block_row,
        block_col,
        y0,
        x0
    )


def modify_one_pixel_in_block(
    image,
    block_index,
    block_size=16,
    pixel_offset=(0, 0),
    delta=1
):
    """
    Modify exactly one pixel inside the selected block.

        P' = P + delta

    Values are wrapped within [0,255].
    """

    modified = np.asarray(
        image,
        dtype=np.uint8
    ).copy()

    H, W = modified.shape

    (
        block_row,
        block_col,
        y0,
        x0
    ) = block_index_to_position(
        block_index=block_index,
        image_shape=(H, W),
        block_size=block_size
    )

    local_y, local_x = pixel_offset

    if not (
        0 <= local_y < block_size and
        0 <= local_x < block_size
    ):
        raise ValueError(
            "pixel_offset must lie inside the selected block."
        )

    y = y0 + local_y
    x = x0 + local_x

    original_value = int(
        modified[y, x]
    )

    new_value = (
        original_value + delta
    ) % 256

    modified[y, x] = np.uint8(
        new_value
    )

    return modified


def calculate_block_difference_percentages(
    cipher_1,
    cipher_2,
    block_size=16
):
    """
    Calculate the percentage of changed ciphertext
    pixels separately for every ciphertext block.
    """

    cipher_1 = np.asarray(
        cipher_1,
        dtype=np.uint8
    )

    cipher_2 = np.asarray(
        cipher_2,
        dtype=np.uint8
    )

    if cipher_1.shape != cipher_2.shape:
        raise ValueError(
            "Cipher images must have identical dimensions."
        )

    if cipher_1.ndim != 2:
        raise ValueError(
            "Cipher images must be grayscale 2-D arrays."
        )

    H, W = cipher_1.shape

    if H % block_size != 0 or W % block_size != 0:
        raise ValueError(
            "Image dimensions must be divisible by block size."
        )

    blocks_per_row = W // block_size
    blocks_per_col = H // block_size

    records = []

    block_index = 1

    for br in range(blocks_per_col):

        for bc in range(blocks_per_row):

            y0 = br * block_size
            y1 = y0 + block_size

            x0 = bc * block_size
            x1 = x0 + block_size

            block_1 = cipher_1[
                y0:y1,
                x0:x1
            ]

            block_2 = cipher_2[
                y0:y1,
                x0:x1
            ]

            changed = (
                block_1 != block_2
            )

            changed_pixels = int(
                np.count_nonzero(
                    changed
                )
            )

            total_pixels = int(
                block_1.size
            )

            percentage = (
                changed_pixels /
                total_pixels
            ) * 100.0

            records.append({

                "Block":
                    block_index,

                "Block_Row":
                    br + 1,

                "Block_Column":
                    bc + 1,

                "Changed_Pixels":
                    changed_pixels,

                "Total_Pixels":
                    total_pixels,

                "Changed_Percentage":
                    percentage
            })

            block_index += 1

    return pd.DataFrame(
        records
    )


def run_spatial_propagation_experiment(
    image,
    secret_key,
    selected_blocks=(1, 50, 128, 256),
    block_size=16
):
    """
    Modify one pixel in selected plaintext blocks and
    measure ciphertext propagation block-by-block.

    For each selected plaintext block:

        1. Encrypt the original image.
        2. Modify exactly one plaintext pixel.
        3. Encrypt the modified image.
        4. Compare the two ciphertexts.
        5. Calculate changed-pixel percentage per block.

    NOTE:
    The current V2 encrypt_image() generates a fresh random
    salt and nonce for every encryption. Therefore this
    experiment measures both plaintext sensitivity and the
    effect of fresh randomization.
    """

    image = np.asarray(
        image,
        dtype=np.uint8
    )

    if image.ndim != 2:
        raise ValueError(
            "Input image must be a grayscale 2-D image."
        )

    H, W = image.shape

    if H % block_size != 0 or W % block_size != 0:
        raise ValueError(
            "Image dimensions must be divisible by block size."
        )

    total_blocks = (
        (H // block_size) *
        (W // block_size)
    )

    # --------------------------------------------------------
    # Validate selected blocks
    # --------------------------------------------------------

    for block_number in selected_blocks:

        if not (
            1 <= block_number <= total_blocks
        ):
            raise ValueError(
                f"Selected block {block_number} is invalid. "
                f"Valid range is 1 to {total_blocks}."
            )

    # --------------------------------------------------------
    # Reference ciphertext
    # --------------------------------------------------------

    (
        reference_cipher,
        _,
        _,
        _,
        _,
        _
    ) = encrypt_image(
        image=image,
        secret_key=secret_key,
        save_internal=False
    )

    all_results = []

    # --------------------------------------------------------
    # Test each selected plaintext block
    # --------------------------------------------------------

    for changed_block in selected_blocks:

        # ----------------------------------------------------
        # Modify exactly one plaintext pixel
        # ----------------------------------------------------

        modified_plaintext = (
            modify_one_pixel_in_block(
                image=image,
                block_index=changed_block,
                block_size=block_size,
                pixel_offset=(0, 0),
                delta=1
            )
        )

        # ----------------------------------------------------
        # Verify exactly one plaintext pixel changed
        # ----------------------------------------------------

        plaintext_difference = (
            image != modified_plaintext
        )

        changed_plaintext_pixels = int(
            np.count_nonzero(
                plaintext_difference
            )
        )

        if changed_plaintext_pixels != 1:
            raise RuntimeError(
                "Spatial propagation test failed: "
                f"expected exactly 1 changed plaintext pixel, "
                f"found {changed_plaintext_pixels}."
            )

        # ----------------------------------------------------
        # Encrypt modified plaintext
        # ----------------------------------------------------

        (
            modified_cipher,
            _,
            _,
            _,
            _,
            _
        ) = encrypt_image(
            image=modified_plaintext,
            secret_key=secret_key,
            save_internal=False
        )

        # ----------------------------------------------------
        # Compare ciphertexts block-by-block
        # ----------------------------------------------------

        block_results = (
            calculate_block_difference_percentages(
                cipher_1=reference_cipher,
                cipher_2=modified_cipher,
                block_size=block_size
            )
        )

        block_results.insert(
            0,
            "Changed_Plaintext_Block",
            changed_block
        )

        all_results.append(
            block_results
        )

    return pd.concat(
        all_results,
        ignore_index=True
    )


# ============================================================
# RUN EXPERIMENT
# ============================================================

# Local deterministic test key.
# This replaces the removed legacy USER_SECRET variable.
_TEST_SECRET_KEY_CELL36 = (
    "DNA-Adaptive-V2-Test-Key-2026"
)

_SELECTED_BLOCKS_CELL36 = (
    1,
    50,
    128,
    256
)

spatial_propagation_results = (
    run_spatial_propagation_experiment(
        image=test_image,
        secret_key=_TEST_SECRET_KEY_CELL36,
        selected_blocks=_SELECTED_BLOCKS_CELL36,
        block_size=CONFIG["BLOCK_SIZE"]
    )
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 80)
print("NPCR/UACI SPATIAL PROPAGATION EXPERIMENT")
print("=" * 80)

for block_number in _SELECTED_BLOCKS_CELL36:

    subset = (
        spatial_propagation_results[
            spatial_propagation_results[
                "Changed_Plaintext_Block"
            ] == block_number
        ]
    )

    # --------------------------------------------------------
    # Count blocks containing changed ciphertext pixels
    # --------------------------------------------------------

    changed_cipher_blocks = int(
        np.count_nonzero(
            subset[
                "Changed_Pixels"
            ].values
        )
    )

    total_cipher_blocks = int(
        len(subset)
    )

    # --------------------------------------------------------
    # Total changed ciphertext pixels
    # --------------------------------------------------------

    total_changed_pixels = int(
        subset[
            "Changed_Pixels"
        ].sum()
    )

    # --------------------------------------------------------
    # Overall changed-pixel percentage
    # --------------------------------------------------------

    total_cipher_pixels = int(
        subset[
            "Total_Pixels"
        ].sum()
    )

    overall_changed_percentage = (
        total_changed_pixels /
        total_cipher_pixels
    ) * 100.0

    print()

    print(
        f"Plaintext changed in block "
        f"{block_number}"
    )

    print(
        f"Blocks with changed ciphertext: "
        f"{changed_cipher_blocks} / "
        f"{total_cipher_blocks}"
    )

    print(
        f"Total changed ciphertext pixels: "
        f"{total_changed_pixels}"
    )

    print(
        f"Overall changed ciphertext percentage: "
        f"{overall_changed_percentage:.6f}%"
    )


# ============================================================
# SHOW FIRST 10 RECORDS
# ============================================================

print()

print(
    spatial_propagation_results
    .head(10)
    .to_string(index=False)
)

print()

print(
    "Spatial propagation experiment: PASSED"
)

Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
NPCR/UACI SPATIAL PROPAGATION EXPERIMENT

Plaintext changed in block 1
Blocks with changed ciphertext: 256 / 256
Total changed ciphertext pixels: 65284
Overall changed ciphertext percentage: 99.615479%

Plaintext changed in block 50
Blocks with changed ciphertext: 256 / 256
Total changed ciphertext pixels: 65268
Overall changed ciphertext percentage: 99.591064%

Plaintext changed in block 128
Blocks with changed ciphertext: 256 / 256
Total changed ciphertext pixels: 65273
Overall changed ciphertext percentage: 99.598694%

Plaintext changed in block 256
Blocks with changed ciphertext: 256 / 256
Total changed ciphertext pixels: 65293
Overall changed ciphertext percentage: 99.629211%

 Changed_Plaintext_Block  Block  Block_Row  Block_Column  Changed_Pixels  Total_Pixels  Changed_Percentage
       

In [ ]:
# ============================================================
# CELL 37 — KEY SENSITIVITY
# ============================================================

def flip_one_key_bit(
    secret_key,
    byte_index=0,
    bit_index=0
):
    """
    Modify exactly one bit of the user secret key.

        K' = K with one bit flipped
    """

    if isinstance(secret_key, str):
        key_bytes = bytearray(
            secret_key.encode("utf-8")
        )

    elif isinstance(secret_key, bytes):
        key_bytes = bytearray(
            secret_key
        )

    else:
        raise TypeError(
            "secret_key must be str or bytes."
        )

    if len(key_bytes) == 0:
        raise ValueError(
            "Secret key cannot be empty."
        )

    if not (
        0 <= byte_index < len(key_bytes)
    ):
        raise ValueError(
            "byte_index is outside the key."
        )

    if not (
        0 <= bit_index <= 7
    ):
        raise ValueError(
            "bit_index must be between 0 and 7."
        )

    key_bytes[byte_index] ^= (
        1 << bit_index
    )

    return bytes(key_bytes)


def calculate_key_sensitivity(
    plaintext,
    original_key,
    modified_key
):
    """
    Compare ciphertexts generated using keys that differ
    by exactly one bit.
    """

    # --------------------------------------------------------
    # Encrypt with original key
    # --------------------------------------------------------

    (
        cipher_original,
        _,
        _,
        _,
        _,
        _
    ) = encrypt_image(
        image=plaintext,
        secret_key=original_key,
        save_internal=False
    )

    # --------------------------------------------------------
    # Encrypt with one-bit-modified key
    # --------------------------------------------------------

    (
        cipher_modified,
        _,
        _,
        _,
        _,
        _
    ) = encrypt_image(
        image=plaintext,
        secret_key=modified_key,
        save_internal=False
    )

    # --------------------------------------------------------
    # NPCR
    # --------------------------------------------------------

    npcr, changed_pixels, difference_map = (
        calculate_npcr(
            cipher_original,
            cipher_modified
        )
    )

    # --------------------------------------------------------
    # UACI
    # --------------------------------------------------------

    uaci = calculate_uaci(
        cipher_original,
        cipher_modified
    )

    # --------------------------------------------------------
    # Ciphertext correlations
    # --------------------------------------------------------

    correlation_original = (
        calculate_directional_correlations(
            cipher_original
        )
    )

    correlation_modified = (
        calculate_directional_correlations(
            cipher_modified
        )
    )

    return {
        "Original_Cipher":
            cipher_original,

        "Modified_Key_Cipher":
            cipher_modified,

        "NPCR_Percent":
            npcr,

        "UACI_Percent":
            uaci,

        "Changed_Pixels":
            changed_pixels,

        "Difference_Map":
            difference_map,

        "Original_Correlation":
            correlation_original,

        "Modified_Correlation":
            correlation_modified
    }


# ============================================================
# LOCAL TEST KEY
# ============================================================

# Replaces the removed legacy USER_SECRET variable.
_TEST_SECRET_KEY_CELL37 = (
    "DNA-Adaptive-V2-Test-Key-2026"
)


# ============================================================
# GENERATE ONE-BIT-MODIFIED KEY
# ============================================================

modified_secret_key = flip_one_key_bit(
    _TEST_SECRET_KEY_CELL37,
    byte_index=0,
    bit_index=0
)


# ============================================================
# VERIFY EXACTLY ONE BIT WAS CHANGED
# ============================================================

original_key_bytes = (
    _TEST_SECRET_KEY_CELL37.encode("utf-8")
)

modified_key_bytes = bytes(
    modified_secret_key
)

if len(original_key_bytes) != len(modified_key_bytes):
    raise RuntimeError(
        "Modified key length changed unexpectedly."
    )

different_key_bytes = [
    (
        index,
        original_key_bytes[index],
        modified_key_bytes[index]
    )
    for index in range(
        len(original_key_bytes)
    )
    if (
        original_key_bytes[index]
        != modified_key_bytes[index]
    )
]

if len(different_key_bytes) != 1:
    raise RuntimeError(
        "Key-sensitivity setup failed: "
        "more than one key byte changed."
    )

original_byte = different_key_bytes[0][1]
modified_byte = different_key_bytes[0][2]

bit_difference = (
    original_byte ^
    modified_byte
)

if bit_difference.bit_count() != 1:
    raise RuntimeError(
        "Key-sensitivity setup failed: "
        "expected exactly one changed bit."
    )


# ============================================================
# RUN KEY SENSITIVITY EXPERIMENT
# ============================================================

key_sensitivity_result = (
    calculate_key_sensitivity(
        plaintext=test_image,
        original_key=_TEST_SECRET_KEY_CELL37,
        modified_key=modified_secret_key
    )
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 80)
print("KEY SENSITIVITY")
print("=" * 80)

print()

print(
    "Original key:"
    f" {_TEST_SECRET_KEY_CELL37}"
)

print(
    "Modified key differs by exactly one bit: TRUE"
)

print(
    "Modified byte index: 0"
)

print(
    "Modified bit index: 0"
)

print()

print(
    f"NPCR : "
    f"{key_sensitivity_result['NPCR_Percent']:.10f}%"
)

print(
    f"UACI : "
    f"{key_sensitivity_result['UACI_Percent']:.10f}%"
)

print(
    f"Changed pixels : "
    f"{key_sensitivity_result['Changed_Pixels']}"
)

print()

print("Original-key ciphertext correlations:")

print(
    key_sensitivity_result[
        "Original_Correlation"
    ]
)

print()

print("Modified-key ciphertext correlations:")

print(
    key_sensitivity_result[
        "Modified_Correlation"
    ]
)

print()

print(
    "Key sensitivity experiment: PASSED"
)

Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
KEY SENSITIVITY

Original key: DNA-Adaptive-V2-Test-Key-2026
Modified key differs by exactly one bit: TRUE
Modified byte index: 0
Modified bit index: 0

NPCR : 99.6215820312%
UACI : 33.6272056430%
Changed pixels : 65288

Original-key ciphertext correlations:
{'Horizontal': 0.00416165117642111, 'Vertical': -0.0018878382260185222, 'Diagonal': 0.0003870187525865799}

Modified-key ciphertext correlations:
{'Horizontal': 0.002523734534300215, 'Vertical': -0.003394438572354858, 'Diagonal': 0.003314143223145844}

Key sensitivity experiment: PASSED


In [ ]:
# ============================================================
# CELL 38 — PLAINTEXT SENSITIVITY
# ============================================================

def run_plaintext_sensitivity_test(
    image,
    secret_key,
    row=None,
    col=None,
    delta=1
):
    """
    Change exactly one plaintext pixel:

        P'(x,y) = P(x,y) ± 1

    Then compare:

        C  = E(P,K)
        C' = E(P',K)
    """

    image = np.asarray(
        image,
        dtype=np.uint8
    )

    if image.ndim != 2:
        raise ValueError(
            "Input image must be a grayscale 2-D image."
        )

    modified = image.copy()

    H, W = image.shape

    if row is None:
        row = H // 2

    if col is None:
        col = W // 2

    if not (
        0 <= row < H and
        0 <= col < W
    ):
        raise ValueError(
            "Pixel position is outside image."
        )

    if delta not in (-1, 1):
        raise ValueError(
            "delta must be either +1 or -1."
        )

    # --------------------------------------------------------
    # Modify exactly one plaintext pixel
    # --------------------------------------------------------

    original_value = int(
        modified[row, col]
    )

    new_value = (
        original_value + delta
    ) % 256

    modified[row, col] = np.uint8(
        new_value
    )

    # --------------------------------------------------------
    # Verify exactly one plaintext pixel changed
    # --------------------------------------------------------

    plaintext_difference = (
        image != modified
    )

    plaintext_changed_pixels = int(
        np.count_nonzero(
            plaintext_difference
        )
    )

    if plaintext_changed_pixels != 1:
        raise RuntimeError(
            "Plaintext sensitivity setup failed: "
            f"expected exactly 1 changed pixel, "
            f"found {plaintext_changed_pixels}."
        )

    # --------------------------------------------------------
    # Encrypt original plaintext
    # --------------------------------------------------------

    (
        cipher_original,
        _,
        _,
        _,
        _,
        _
    ) = encrypt_image(
        image=image,
        secret_key=secret_key,
        save_internal=False
    )

    # --------------------------------------------------------
    # Encrypt modified plaintext
    # --------------------------------------------------------

    (
        cipher_modified,
        _,
        _,
        _,
        _,
        _
    ) = encrypt_image(
        image=modified,
        secret_key=secret_key,
        save_internal=False
    )

    # --------------------------------------------------------
    # NPCR
    # --------------------------------------------------------

    (
        npcr,
        changed_pixels,
        difference_map
    ) = calculate_npcr(
        cipher_original,
        cipher_modified
    )

    # --------------------------------------------------------
    # UACI
    # --------------------------------------------------------

    uaci = calculate_uaci(
        cipher_original,
        cipher_modified
    )

    return {
        "Original_Plaintext":
            image,

        "Modified_Plaintext":
            modified,

        "Original_Cipher":
            cipher_original,

        "Modified_Cipher":
            cipher_modified,

        "Modified_Row":
            row,

        "Modified_Column":
            col,

        "Original_Pixel_Value":
            original_value,

        "Modified_Pixel_Value":
            new_value,

        "Plaintext_Changed_Pixels":
            plaintext_changed_pixels,

        "Cipher_Changed_Pixels":
            changed_pixels,

        "NPCR_Percent":
            npcr,

        "UACI_Percent":
            uaci,

        "Difference_Map":
            difference_map
    }


# ============================================================
# LOCAL TEST KEY
# ============================================================

# Replaces the removed legacy USER_SECRET variable.
_TEST_SECRET_KEY_CELL38 = (
    "DNA-Adaptive-V2-Test-Key-2026"
)


# ============================================================
# RUN PLAINTEXT SENSITIVITY
# ============================================================

plaintext_sensitivity_result = (
    run_plaintext_sensitivity_test(
        image=test_image,
        secret_key=_TEST_SECRET_KEY_CELL38,
        delta=1
    )
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 80)
print("PLAINTEXT SENSITIVITY")
print("=" * 80)

print()

print(
    f"Modified pixel position  : "
    f"("
    f"{plaintext_sensitivity_result['Modified_Row']}, "
    f"{plaintext_sensitivity_result['Modified_Column']}"
    f")"
)

print(
    f"Original pixel value     : "
    f"{plaintext_sensitivity_result['Original_Pixel_Value']}"
)

print(
    f"Modified pixel value     : "
    f"{plaintext_sensitivity_result['Modified_Pixel_Value']}"
)

print(
    f"Plaintext changed pixels : "
    f"{plaintext_sensitivity_result['Plaintext_Changed_Pixels']}"
)

print(
    f"Cipher changed pixels    : "
    f"{plaintext_sensitivity_result['Cipher_Changed_Pixels']}"
)

print(
    f"NPCR                     : "
    f"{plaintext_sensitivity_result['NPCR_Percent']:.10f}%"
)

print(
    f"UACI                     : "
    f"{plaintext_sensitivity_result['UACI_Percent']:.10f}%"
)

print()

print(
    "Plaintext sensitivity experiment: PASSED"
)

Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
PLAINTEXT SENSITIVITY

Modified pixel position  : (128, 128)
Original pixel value     : 192
Modified pixel value     : 193
Plaintext changed pixels : 1
Cipher changed pixels    : 65286
NPCR                     : 99.6185302734%
UACI                     : 33.4438488530%

Plaintext sensitivity experiment: PASSED


In [ ]:
# ============================================================
# CELL 39 — CHOSEN-PLAINTEXT TESTS
# ============================================================

def create_chosen_plaintext_images(
    height,
    width
):
    """
    Construct controlled plaintexts for security analysis.
    """

    # --------------------------------------------------------
    # Validate dimensions
    # --------------------------------------------------------

    height = int(height)
    width = int(width)

    if height <= 0 or width <= 0:
        raise ValueError(
            "Image height and width must be positive."
        )

    # --------------------------------------------------------
    # Test 1 — all zeros
    # --------------------------------------------------------

    all_zero = np.zeros(
        (height, width),
        dtype=np.uint8
    )

    # --------------------------------------------------------
    # Test 2 — all 255
    # --------------------------------------------------------

    all_255 = np.full(
        (height, width),
        255,
        dtype=np.uint8
    )

    # --------------------------------------------------------
    # Test 3 — constant image
    # --------------------------------------------------------

    constant = np.full(
        (height, width),
        128,
        dtype=np.uint8
    )

    # --------------------------------------------------------
    # Test 4 — checkerboard
    # --------------------------------------------------------

    checkerboard = (
        (
            np.indices(
                (height, width)
            ).sum(axis=0) % 2
        ) * 255
    ).astype(np.uint8)

    # --------------------------------------------------------
    # Test 5 — impulse
    # --------------------------------------------------------

    impulse = np.zeros(
        (height, width),
        dtype=np.uint8
    )

    impulse[
        height // 2,
        width // 2
    ] = 255

    # --------------------------------------------------------
    # Test 6 — deterministic random image
    # --------------------------------------------------------

    random_seed = int(
        CONFIG.get(
            "RANDOM_SEED",
            2026
        )
    )

    rng = np.random.default_rng(
        random_seed
    )

    random_image = rng.integers(
        low=0,
        high=256,
        size=(height, width),
        dtype=np.uint8
    )

    return {
        "All_Zero": all_zero,
        "All_255": all_255,
        "Constant_128": constant,
        "Checkerboard": checkerboard,
        "Impulse": impulse,
        "Random": random_image
    }


def analyze_chosen_plaintexts(
    plaintexts,
    secret_key
):
    """
    Encrypt each controlled plaintext and calculate:

        - entropy
        - horizontal correlation
        - vertical correlation
        - diagonal correlation
        - ciphertext histogram statistics
        - encryption time
    """

    if not isinstance(
        plaintexts,
        dict
    ):
        raise TypeError(
            "plaintexts must be a dictionary."
        )

    if secret_key is None:
        raise ValueError(
            "secret_key must not be None."
        )

    records = []
    cipher_images = {}

    for name, plaintext in plaintexts.items():

        plaintext = np.asarray(
            plaintext,
            dtype=np.uint8
        )

        # ----------------------------------------------------
        # Validate plaintext
        # ----------------------------------------------------

        if plaintext.ndim != 2:
            raise ValueError(
                f"{name}: plaintext must be a "
                "2-D grayscale image."
            )

        H, W = plaintext.shape

        if H == 0 or W == 0:
            raise ValueError(
                f"{name}: plaintext has invalid dimensions."
            )

        # ----------------------------------------------------
        # Encryption
        # ----------------------------------------------------

        start_time = time.perf_counter()

        encryption_result = encrypt_image(
            image=plaintext,
            secret_key=secret_key,
            save_internal=False
        )

        encryption_time = (
            time.perf_counter()
            - start_time
        )

        # ----------------------------------------------------
        # Validate encryption result
        # ----------------------------------------------------

        if not isinstance(
            encryption_result,
            tuple
        ):
            raise RuntimeError(
                "encrypt_image() did not return a tuple."
            )

        if len(encryption_result) < 1:
            raise RuntimeError(
                "encrypt_image() returned no ciphertext."
            )

        cipher = np.asarray(
            encryption_result[0],
            dtype=np.uint8
        )

        if cipher.shape != plaintext.shape:
            raise RuntimeError(
                f"{name}: ciphertext shape "
                f"{cipher.shape} does not match "
                f"plaintext shape {plaintext.shape}."
            )

        # ----------------------------------------------------
        # Ciphertext entropy
        # ----------------------------------------------------

        entropy = calculate_entropy(
            cipher
        )

        # ----------------------------------------------------
        # Directional correlations
        # ----------------------------------------------------

        correlations = (
            calculate_directional_correlations(
                cipher
            )
        )

        # ----------------------------------------------------
        # Histogram statistics
        # ----------------------------------------------------

        histogram = calculate_histogram(
            cipher
        )

        # ----------------------------------------------------
        # Store measurements
        # ----------------------------------------------------

        records.append({

            "Plaintext_Type":
                name,

            "Height":
                int(H),

            "Width":
                int(W),

            "Cipher_Entropy":
                float(entropy),

            "Correlation_H":
                float(
                    correlations[
                        "Horizontal"
                    ]
                ),

            "Correlation_V":
                float(
                    correlations[
                        "Vertical"
                    ]
                ),

            "Correlation_D":
                float(
                    correlations[
                        "Diagonal"
                    ]
                ),

            "Histogram_Min":
                int(
                    histogram.min()
                ),

            "Histogram_Max":
                int(
                    histogram.max()
                ),

            "Histogram_Mean":
                float(
                    histogram.mean()
                ),

            "Encryption_Time_Seconds":
                float(
                    encryption_time
                )
        })

        cipher_images[name] = cipher

    return (
        pd.DataFrame(records),
        cipher_images
    )


# ============================================================
# LOCAL TEST SECRET
# ============================================================
#
# Do NOT depend on the removed legacy USER_SECRET global.
# This is only a deterministic analysis key.
#
# Production encryption should receive the actual secret key
# explicitly.
# ============================================================

_TEST_SECRET_KEY_CELL39 = (
    "DNA-Adaptive-V2-Test-Key-2026"
)


# ============================================================
# CREATE CHOSEN PLAINTEXTS
# ============================================================

if (
    "test_image" not in globals()
    or test_image is None
):
    raise NameError(
        "test_image is not available. "
        "Run the test-image loading cell first."
    )


test_image_cell39 = np.asarray(
    test_image,
    dtype=np.uint8
)

if test_image_cell39.ndim != 2:
    raise ValueError(
        "test_image must be a 2-D grayscale image."
    )


chosen_plaintexts = (
    create_chosen_plaintext_images(
        height=test_image_cell39.shape[0],
        width=test_image_cell39.shape[1]
    )
)


# ============================================================
# ANALYZE
# ============================================================

chosen_plaintext_results, chosen_cipher_images = (
    analyze_chosen_plaintexts(
        plaintexts=chosen_plaintexts,
        secret_key=_TEST_SECRET_KEY_CELL39
    )
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 80)
print("CHOSEN-PLAINTEXT TESTS")
print("=" * 80)

print()

print(
    chosen_plaintext_results.to_string(
        index=False
    )
)

Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
CHOSEN-PLAINTEXT TESTS

Plaintext_Type  Height  Width  Cipher_Entropy  Correlation_H  Correlation_V  Correlation_D  Histogram_Min  Histogram_Max  Histogram_Mean  Encryption_Time_Seconds
      All_Zero     256    256        7.997565      -0.001018       0.005241       0.001405            215            310           256.0                 0.725304
       All_255     256    256        7.997521      -0.000887       0.004705       0.000496            212            307           256.0                 0.738789
  Constant_128     256    256        7.997409       0.005048      -0.004808       0.003238            221            300           256.0                 0.759083
  Checkerboard     256    256        7.997381       0.000652       0.000269       0.008766    

In [ ]:
# ============================================================
# CELL 40 — NOISE ATTACK — COMPLETE DATASET
# ============================================================

_TEST_SECRET_KEY_CELL40 = (
    "DNA-Adaptive-V2-Test-Key-2026"
)


def add_salt_pepper_noise(
    cipher_image,
    noise_percent,
    seed=42
):
    """
    Modify a percentage of ciphertext pixels using
    salt-and-pepper corruption.
    """

    if not 0 <= noise_percent <= 100:
        raise ValueError(
            "noise_percent must be between 0 and 100."
        )

    damaged = np.asarray(
        cipher_image,
        dtype=np.uint8
    ).copy()

    total_pixels = damaged.size

    number_changed = int(
        round(
            total_pixels *
            noise_percent /
            100.0
        )
    )

    if number_changed == 0:
        return damaged

    rng = np.random.default_rng(seed)

    indices = rng.choice(
        total_pixels,
        size=number_changed,
        replace=False
    )

    # Randomly choose salt or pepper
    salt_mask = (
        rng.random(number_changed) < 0.5
    )

    flat = damaged.reshape(-1)

    flat[indices[salt_mask]] = 255
    flat[indices[~salt_mask]] = 0

    return damaged


def add_gaussian_noise(
    cipher_image,
    sigma,
    seed=42
):
    """
    Add Gaussian noise to ciphertext.
    """

    rng = np.random.default_rng(seed)

    cipher_float = np.asarray(
        cipher_image,
        dtype=np.float64
    )

    noise = rng.normal(
        loc=0.0,
        scale=float(sigma),
        size=cipher_float.shape
    )

    noisy = (
        cipher_float +
        noise
    )

    noisy = np.clip(
        noisy,
        0,
        255
    ).astype(np.uint8)

    return noisy


def run_noise_attack_dataset(
    dataset_files,
    secret_key,
    noise_levels=(1, 5, 10)
):
    """
    Run ciphertext noise attacks over the COMPLETE
    validated dataset.

    Since CELL 28 authenticates the complete ciphertext
    before decryption, any modified ciphertext should
    be rejected.

    This experiment therefore measures:

        ciphertext modification
            ->
        authentication rejection

    rather than corrupted-image recovery.
    """

    records = []

    random_seed = int(
        CONFIG.get(
            "RANDOM_SEED",
            42
        )
    )

    for image_index, image_path in enumerate(
        dataset_files,
        start=1
    ):

        image_path = Path(image_path)

        original = cv2.imread(
            str(image_path),
            cv2.IMREAD_GRAYSCALE
        )

        if original is None:
            records.append({
                "Image": image_path.stem,
                "Image_Index": image_index,
                "Attack": "Image_Load",
                "Noise_Level": np.nan,
                "Modified_Cipher_Pixels": -1,
                "Authentication_Rejected": False,
                "Status": "Image could not be loaded"
            })
            continue

        original = np.asarray(
            original,
            dtype=np.uint8
        )

        # ----------------------------------------------------
        # Encrypt once
        # ----------------------------------------------------

        (
            cipher_image,
            encryption_state,
            _,
            _,
            _,
            _
        ) = encrypt_image(
            image=original,
            secret_key=secret_key,
            save_internal=False,
            image_name=image_path.stem
        )

        metadata = (
            encryption_state["metadata"]
        )

        total_pixels = int(
            cipher_image.size
        )

        # ----------------------------------------------------
        # ATTACK LEVELS
        # ----------------------------------------------------

        for noise_level in noise_levels:

            # =================================================
            # SALT AND PEPPER
            # =================================================

            damaged_sp = add_salt_pepper_noise(
                cipher_image,
                noise_percent=noise_level,
                seed=random_seed
            )

            modified_sp_pixels = int(
                np.count_nonzero(
                    cipher_image != damaged_sp
                )
            )

            try:

                decrypt_image(
                    cipher_image=damaged_sp,
                    metadata=metadata,
                    secret_key=secret_key,
                    save_internal=False
                )

                authentication_rejected = False

                status = (
                    "WARNING: modified ciphertext "
                    "was accepted"
                )

            except ValueError as exc:

                if "Authentication verification failed" in str(exc):

                    authentication_rejected = True

                    status = (
                        "Authentication rejected modified ciphertext"
                    )

                else:

                    authentication_rejected = False

                    status = (
                        f"Decryption error: "
                        f"{type(exc).__name__}: {exc}"
                    )

            except Exception as exc:

                authentication_rejected = False

                status = (
                    f"Unexpected error: "
                    f"{type(exc).__name__}: {exc}"
                )

            records.append({

                "Image":
                    image_path.stem,

                "Image_Index":
                    image_index,

                "Attack":
                    "Salt_Pepper",

                "Noise_Level":
                    float(noise_level),

                "Total_Pixels":
                    total_pixels,

                "Modified_Cipher_Pixels":
                    modified_sp_pixels,

                "Authentication_Rejected":
                    authentication_rejected,

                "Status":
                    status
            })

            # =================================================
            # GAUSSIAN
            # =================================================

            sigma = (
                255.0 *
                float(noise_level) /
                100.0
            )

            damaged_gaussian = add_gaussian_noise(
                cipher_image,
                sigma=sigma,
                seed=random_seed
            )

            modified_gaussian_pixels = int(
                np.count_nonzero(
                    cipher_image != damaged_gaussian
                )
            )

            try:

                decrypt_image(
                    cipher_image=damaged_gaussian,
                    metadata=metadata,
                    secret_key=secret_key,
                    save_internal=False
                )

                authentication_rejected = False

                status = (
                    "WARNING: modified ciphertext "
                    "was accepted"
                )

            except ValueError as exc:

                if "Authentication verification failed" in str(exc):

                    authentication_rejected = True

                    status = (
                        "Authentication rejected modified ciphertext"
                    )

                else:

                    authentication_rejected = False

                    status = (
                        f"Decryption error: "
                        f"{type(exc).__name__}: {exc}"
                    )

            except Exception as exc:

                authentication_rejected = False

                status = (
                    f"Unexpected error: "
                    f"{type(exc).__name__}: {exc}"
                )

            records.append({

                "Image":
                    image_path.stem,

                "Image_Index":
                    image_index,

                "Attack":
                    "Gaussian",

                "Noise_Level":
                    float(noise_level),

                "Total_Pixels":
                    total_pixels,

                "Modified_Cipher_Pixels":
                    modified_gaussian_pixels,

                "Authentication_Rejected":
                    authentication_rejected,

                "Status":
                    status
            })

        print(
            f"[CELL 40] Completed "
            f"{image_index}/{len(dataset_files)}: "
            f"{image_path.name}"
        )

    return pd.DataFrame(records)


# ============================================================
# RUN COMPLETE DATASET EXPERIMENT
# ============================================================

noise_attack_results = (
    run_noise_attack_dataset(
        dataset_files=valid_dataset_files,
        secret_key=_TEST_SECRET_KEY_CELL40,
        noise_levels=(1, 5, 10)
    )
)


# ============================================================
# SUMMARY
# ============================================================

total_tests = len(
    noise_attack_results
)

authentication_rejections = int(
    noise_attack_results[
        "Authentication_Rejected"
    ].sum()
)

print()
print("=" * 80)
print("CELL 40 — NOISE ATTACK — COMPLETE DATASET")
print("=" * 80)

print(
    f"Images processed : "
    f"{len(valid_dataset_files)}"
)

print(
    f"Attack cases     : "
    f"{total_tests}"
)

print(
    f"Authentication rejections : "
    f"{authentication_rejections}/{total_tests}"
)

print()

print(
    noise_attack_results.to_string(
        index=False
    )
)


# ============================================================
# SAVE RESULTS
# ============================================================

cell40_output_path = (
    RESULT_DIR /
    "09_SECURITY" /
    "cell40_noise_attack_all_images.csv"
)

cell40_output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

noise_attack_results.to_csv(
    cell40_output_path,
    index=False
)

print()
print(
    f"Results saved to:\n"
    f"{cell40_output_path}"
)

Backward ciphertext diffusion applied.
[CELL 40] Completed 1/49: Copy of Copy of image_0050.png
Backward ciphertext diffusion applied.
[CELL 40] Completed 2/49: Copy of Copy of image_0051.png
Backward ciphertext diffusion applied.
[CELL 40] Completed 3/49: Copy of Copy of image_0052.png
Backward ciphertext diffusion applied.
[CELL 40] Completed 4/49: Copy of Copy of image_0053.png
Backward ciphertext diffusion applied.
[CELL 40] Completed 5/49: Copy of Copy of image_0054.png
Backward ciphertext diffusion applied.
[CELL 40] Completed 6/49: Copy of Copy of image_0055.png
Backward ciphertext diffusion applied.
[CELL 40] Completed 7/49: Copy of Copy of image_0056.png
Backward ciphertext diffusion applied.
[CELL 40] Completed 8/49: Copy of Copy of image_0057.png
Backward ciphertext diffusion applied.
[CELL 40] Completed 9/49: Copy of Copy of image_0058.png
Backward ciphertext diffusion applied.
[CELL 40] Completed 10/49: Copy of Copy of image_0059.png
Backward ciphertext diffusion applied.


In [ ]:
# ============================================================
# CELL 41 — DATA-LOSS ATTACK — COMPLETE DATASET
# ============================================================

_TEST_SECRET_KEY_CELL41 = (
    "DNA-Adaptive-V2-Test-Key-2026"
)


def apply_ciphertext_data_loss(
    cipher_image,
    loss_percent,
    seed=42
):
    """
    Simulate ciphertext data loss by replacing a selected
    percentage of ciphertext pixels with zero.
    """

    if not 0 <= loss_percent <= 100:

        raise ValueError(
            "loss_percent must be between 0 and 100."
        )

    damaged = np.asarray(
        cipher_image,
        dtype=np.uint8
    ).copy()

    total_pixels = damaged.size

    number_lost = int(
        round(
            total_pixels *
            loss_percent /
            100.0
        )
    )

    if number_lost == 0:
        return damaged

    rng = np.random.default_rng(seed)

    indices = rng.choice(
        total_pixels,
        size=number_lost,
        replace=False
    )

    damaged.reshape(-1)[indices] = 0

    return damaged


def run_data_loss_dataset(
    dataset_files,
    secret_key,
    loss_levels=(1, 5, 10, 20)
):
    """
    Run ciphertext data-loss attacks over the complete
    validated dataset.

    The V2 authentication layer covers the complete
    ciphertext, so modified ciphertext should be rejected
    before plaintext reconstruction.
    """

    records = []

    random_seed = int(
        CONFIG.get(
            "RANDOM_SEED",
            42
        )
    )

    for image_index, image_path in enumerate(
        dataset_files,
        start=1
    ):

        image_path = Path(image_path)

        original = cv2.imread(
            str(image_path),
            cv2.IMREAD_GRAYSCALE
        )

        if original is None:

            records.append({

                "Image":
                    image_path.stem,

                "Image_Index":
                    image_index,

                "Data_Loss_Percent":
                    np.nan,

                "Lost_Cipher_Pixels":
                    -1,

                "Authentication_Rejected":
                    False,

                "Status":
                    "Image could not be loaded"
            })

            continue

        original = np.asarray(
            original,
            dtype=np.uint8
        )

        # ----------------------------------------------------
        # Encrypt once
        # ----------------------------------------------------

        (
            cipher_image,
            encryption_state,
            _,
            _,
            _,
            _
        ) = encrypt_image(
            image=original,
            secret_key=secret_key,
            save_internal=False,
            image_name=image_path.stem
        )

        metadata = (
            encryption_state["metadata"]
        )

        # ----------------------------------------------------
        # DATA LOSS LEVELS
        # ----------------------------------------------------

        for loss_level in loss_levels:

            damaged_cipher = (
                apply_ciphertext_data_loss(
                    cipher_image=cipher_image,
                    loss_percent=loss_level,
                    seed=random_seed
                )
            )

            lost_pixels = int(
                np.count_nonzero(
                    cipher_image != damaged_cipher
                )
            )

            try:

                decrypt_image(
                    cipher_image=damaged_cipher,
                    metadata=metadata,
                    secret_key=secret_key,
                    save_internal=False
                )

                authentication_rejected = False

                status = (
                    "WARNING: damaged ciphertext "
                    "was accepted"
                )

            except ValueError as exc:

                if "Authentication verification failed" in str(exc):

                    authentication_rejected = True

                    status = (
                        "Authentication rejected "
                        "damaged ciphertext"
                    )

                else:

                    authentication_rejected = False

                    status = (
                        f"Decryption error: "
                        f"{type(exc).__name__}: {exc}"
                    )

            except Exception as exc:

                authentication_rejected = False

                status = (
                    f"Unexpected error: "
                    f"{type(exc).__name__}: {exc}"
                )

            records.append({

                "Image":
                    image_path.stem,

                "Image_Index":
                    image_index,

                "Data_Loss_Percent":
                    float(loss_level),

                "Total_Pixels":
                    int(cipher_image.size),

                "Lost_Cipher_Pixels":
                    lost_pixels,

                "Authentication_Rejected":
                    authentication_rejected,

                "Status":
                    status
            })

        print(
            f"[CELL 41] Completed "
            f"{image_index}/{len(dataset_files)}: "
            f"{image_path.name}"
        )

    return pd.DataFrame(records)


# ============================================================
# RUN COMPLETE DATASET
# ============================================================

data_loss_results = (
    run_data_loss_dataset(
        dataset_files=valid_dataset_files,
        secret_key=_TEST_SECRET_KEY_CELL41,
        loss_levels=(1, 5, 10, 20)
    )
)


# ============================================================
# SUMMARY
# ============================================================

total_tests = len(
    data_loss_results
)

authentication_rejections = int(
    data_loss_results[
        "Authentication_Rejected"
    ].sum()
)

print()
print("=" * 80)
print("CELL 41 — DATA-LOSS ATTACK — COMPLETE DATASET")
print("=" * 80)

print(
    f"Images processed : "
    f"{len(valid_dataset_files)}"
)

print(
    f"Attack cases     : "
    f"{total_tests}"
)

print(
    f"Authentication rejections : "
    f"{authentication_rejections}/{total_tests}"
)

print()

print(
    data_loss_results.to_string(
        index=False
    )
)


# ============================================================
# SAVE RESULTS
# ============================================================

cell41_output_path = (
    RESULT_DIR /
    "09_SECURITY" /
    "cell41_data_loss_all_images.csv"
)

cell41_output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

data_loss_results.to_csv(
    cell41_output_path,
    index=False
)

print()
print(
    f"Results saved to:\n"
    f"{cell41_output_path}"
)

Backward ciphertext diffusion applied.
[CELL 41] Completed 1/49: Copy of Copy of image_0050.png
Backward ciphertext diffusion applied.
[CELL 41] Completed 2/49: Copy of Copy of image_0051.png
Backward ciphertext diffusion applied.
[CELL 41] Completed 3/49: Copy of Copy of image_0052.png
Backward ciphertext diffusion applied.
[CELL 41] Completed 4/49: Copy of Copy of image_0053.png
Backward ciphertext diffusion applied.
[CELL 41] Completed 5/49: Copy of Copy of image_0054.png
Backward ciphertext diffusion applied.
[CELL 41] Completed 6/49: Copy of Copy of image_0055.png
Backward ciphertext diffusion applied.
[CELL 41] Completed 7/49: Copy of Copy of image_0056.png
Backward ciphertext diffusion applied.
[CELL 41] Completed 8/49: Copy of Copy of image_0057.png
Backward ciphertext diffusion applied.
[CELL 41] Completed 9/49: Copy of Copy of image_0058.png
Backward ciphertext diffusion applied.
[CELL 41] Completed 10/49: Copy of Copy of image_0059.png
Backward ciphertext diffusion applied.


In [ ]:
# ============================================================
# CELL 42 — DIFFERENTIAL ATTACK SUMMARY — COMPLETE DATASET
# ============================================================

_TEST_SECRET_KEY_CELL42 = (
    "DNA-Adaptive-V2-Test-Key-2026"
)


# ============================================================
# METRIC SUMMARY
# ============================================================

def summarize_metric(values):

    values = np.asarray(
        values,
        dtype=np.float64
    )

    values = values[
        np.isfinite(values)
    ]

    if values.size == 0:

        return {
            "Mean": np.nan,
            "Std": np.nan,
            "Min": np.nan,
            "Max": np.nan
        }

    return {

        "Mean":
            float(
                np.mean(values)
            ),

        "Std":
            float(
                np.std(
                    values,
                    ddof=1
                )
                if values.size > 1
                else 0.0
            ),

        "Min":
            float(
                np.min(values)
            ),

        "Max":
            float(
                np.max(values)
            )
    }


# ============================================================
# CELL 42 — LOCAL NPCR
# ============================================================
#
# IMPORTANT:
# Do NOT use the notebook-wide calculate_npcr() here.
#
# CELL 42 requires:
#
#     changed_pixels = scalar integer
#     total_pixels   = scalar integer
#     npcr           = scalar float
#
# This local implementation guarantees that behavior.
# ============================================================

def calculate_npcr_cell42(
    cipher_original,
    cipher_modified
):

    cipher_original = np.asarray(
        cipher_original,
        dtype=np.uint8
    )

    cipher_modified = np.asarray(
        cipher_modified,
        dtype=np.uint8
    )

    # --------------------------------------------------------
    # Shape validation
    # --------------------------------------------------------

    if (
        cipher_original.shape
        !=
        cipher_modified.shape
    ):

        raise ValueError(
            "CELL 42 NPCR error: ciphertext "
            "images must have identical shapes. "
            f"Got "
            f"{cipher_original.shape} and "
            f"{cipher_modified.shape}."
        )

    # --------------------------------------------------------
    # Grayscale validation
    # --------------------------------------------------------

    if cipher_original.ndim != 2:

        raise ValueError(
            "CELL 42 NPCR error: ciphertext "
            "must be a 2-D grayscale image."
        )

    # --------------------------------------------------------
    # Total number of ciphertext pixels
    # --------------------------------------------------------

    total_pixels = int(
        cipher_original.size
    )

    if total_pixels <= 0:

        raise ValueError(
            "CELL 42 NPCR error: ciphertext "
            "contains zero pixels."
        )

    # --------------------------------------------------------
    # Changed pixels
    # --------------------------------------------------------

    changed_mask = (
        cipher_original
        !=
        cipher_modified
    )

    changed_pixels = int(
        np.count_nonzero(
            changed_mask
        )
    )

    # --------------------------------------------------------
    # NPCR
    # --------------------------------------------------------

    npcr = float(
        100.0
        *
        changed_pixels
        /
        total_pixels
    )

    return (
        npcr,
        changed_pixels,
        total_pixels
    )


# ============================================================
# CELL 42 — LOCAL UACI
# ============================================================

def calculate_uaci_cell42(
    cipher_original,
    cipher_modified
):

    cipher_original = np.asarray(
        cipher_original,
        dtype=np.float64
    )

    cipher_modified = np.asarray(
        cipher_modified,
        dtype=np.float64
    )

    # --------------------------------------------------------
    # Shape validation
    # --------------------------------------------------------

    if (
        cipher_original.shape
        !=
        cipher_modified.shape
    ):

        raise ValueError(
            "CELL 42 UACI error: ciphertext "
            "images must have identical shapes."
        )

    if cipher_original.ndim != 2:

        raise ValueError(
            "CELL 42 UACI error: ciphertext "
            "must be a 2-D grayscale image."
        )

    total_pixels = int(
        cipher_original.size
    )

    if total_pixels <= 0:

        raise ValueError(
            "CELL 42 UACI error: ciphertext "
            "contains zero pixels."
        )

    # --------------------------------------------------------
    # UACI
    # --------------------------------------------------------

    absolute_difference = np.abs(
        cipher_original
        -
        cipher_modified
    )

    uaci = float(
        100.0
        *
        np.mean(
            absolute_difference
            /
            255.0
        )
    )

    return uaci


# ============================================================
# DIFFERENTIAL EXPERIMENT
# ============================================================

def run_differential_summary(
    dataset_images,
    secret_key
):

    image_records = []

    salt_length = int(
        CONFIG["SALT_BYTES"]
    )

    nonce_length = int(
        CONFIG["NONCE_BYTES"]
    )

    total_dataset_images = len(
        dataset_images
    )

    if total_dataset_images == 0:

        raise RuntimeError(
            "CELL 42 received an empty dataset."
        )

    for image_index, (
        image_name,
        image
    ) in enumerate(
        dataset_images,
        start=1
    ):

        # ----------------------------------------------------
        # INPUT
        # ----------------------------------------------------

        image = np.asarray(
            image,
            dtype=np.uint8
        )

        if image.ndim != 2:

            raise ValueError(
                f"{image_name}: image must be "
                "a grayscale 2-D array."
            )

        H, W = image.shape

        B = int(
            CONFIG["BLOCK_SIZE"]
        )

        if (
            H % B != 0
            or
            W % B != 0
        ):

            raise ValueError(
                f"{image_name}: image dimensions "
                f"{H}x{W} are not divisible by "
                f"BLOCK_SIZE={B}."
            )

        # ----------------------------------------------------
        # EXACTLY ONE PIXEL CHANGE
        # ----------------------------------------------------

        modified_image = image.copy()

        row = H // 2
        col = W // 2

        old_value = int(
            image[row, col]
        )

        new_value = (
            old_value + 1
        ) % 256

        modified_image[row, col] = (
            np.uint8(new_value)
        )

        plaintext_changed_pixels = int(
            np.count_nonzero(
                image
                !=
                modified_image
            )
        )

        if plaintext_changed_pixels != 1:

            raise RuntimeError(
                f"{image_name}: expected exactly "
                f"one changed plaintext pixel, "
                f"got {plaintext_changed_pixels}."
            )

        # ----------------------------------------------------
        # CONTROLLED SALT + NONCE
        # ----------------------------------------------------
        #
        # The same salt and nonce MUST be used for both
        # plaintext versions so that this is a controlled
        # differential experiment.
        # ----------------------------------------------------

        salt = secrets.token_bytes(
            salt_length
        )

        nonce = secrets.token_bytes(
            nonce_length
        )

        # ----------------------------------------------------
        # ORIGINAL IMAGE
        # ----------------------------------------------------

        (
            cipher_original,
            state_original,
            _,
            _,
            _,
            _
        ) = encrypt_image(
            image=image,
            secret_key=secret_key,
            save_internal=False,
            image_name=str(image_name),
            salt=salt,
            nonce=nonce
        )

        # ----------------------------------------------------
        # MODIFIED IMAGE
        # ----------------------------------------------------

        (
            cipher_modified,
            state_modified,
            _,
            _,
            _,
            _
        ) = encrypt_image(
            image=modified_image,
            secret_key=secret_key,
            save_internal=False,
            image_name=str(image_name),
            salt=salt,
            nonce=nonce
        )

        # ----------------------------------------------------
        # FORCE NUMPY ARRAYS
        # ----------------------------------------------------

        cipher_original = np.asarray(
            cipher_original,
            dtype=np.uint8
        )

        cipher_modified = np.asarray(
            cipher_modified,
            dtype=np.uint8
        )

        # ----------------------------------------------------
        # VERIFY CIPHERTEXT SHAPES
        # ----------------------------------------------------

        if (
            cipher_original.shape
            !=
            cipher_modified.shape
        ):

            raise RuntimeError(
                f"{image_name}: ciphertext shape "
                f"mismatch: "
                f"{cipher_original.shape} vs "
                f"{cipher_modified.shape}."
            )

        if cipher_original.ndim != 2:

            raise RuntimeError(
                f"{image_name}: ciphertext is not "
                "a 2-D grayscale array."
            )

        # ----------------------------------------------------
        # VERIFY SAME SALT
        # ----------------------------------------------------

        metadata_original = (
            state_original["metadata"]
        )

        metadata_modified = (
            state_modified["metadata"]
        )

        if (
            metadata_original["salt"]
            !=
            metadata_modified["salt"]
        ):

            raise RuntimeError(
                f"{image_name}: salt mismatch. "
                "Controlled NPCR/UACI test is invalid."
            )

        # ----------------------------------------------------
        # VERIFY SAME NONCE
        # ----------------------------------------------------

        if (
            metadata_original["nonce"]
            !=
            metadata_modified["nonce"]
        ):

            raise RuntimeError(
                f"{image_name}: nonce mismatch. "
                "Controlled NPCR/UACI test is invalid."
            )

        # ----------------------------------------------------
        # VERIFY SAME SECRET-KEY CONTEXT
        # ----------------------------------------------------
        #
        # The secret key is supplied identically to both
        # encryptions by this experiment.
        #
        # We do not store the secret key in the results.
        # ----------------------------------------------------

        # ----------------------------------------------------
        # DESCRIPTOR DIAGNOSTIC
        # ----------------------------------------------------

        descriptor_changed = (
            metadata_original[
                "content_digest"
            ]
            !=
            metadata_modified[
                "content_digest"
            ]
        )

        # ----------------------------------------------------
        # NPCR
        # ----------------------------------------------------
        #
        # Use the CELL-42-local implementation rather than
        # the notebook-wide calculate_npcr().
        # ----------------------------------------------------

        (
            npcr,
            changed_pixels,
            total_pixels
        ) = calculate_npcr_cell42(
            cipher_original,
            cipher_modified
        )

        # ----------------------------------------------------
        # UACI
        # ----------------------------------------------------

        uaci = calculate_uaci_cell42(
            cipher_original,
            cipher_modified
        )

        # ----------------------------------------------------
        # CHANGED CIPHER PERCENT
        # ----------------------------------------------------

        changed_cipher_percent = float(
            100.0
            *
            changed_pixels
            /
            total_pixels
        )

        # ----------------------------------------------------
        # SANITY CHECKS
        # ----------------------------------------------------

        if not (
            0 <= npcr <= 100
        ):

            raise RuntimeError(
                f"{image_name}: invalid NPCR value "
                f"{npcr}."
            )

        if not (
            0 <= uaci <= 100
        ):

            raise RuntimeError(
                f"{image_name}: invalid UACI value "
                f"{uaci}."
            )

        if changed_pixels < 0:

            raise RuntimeError(
                f"{image_name}: negative changed "
                "pixel count."
            )

        if changed_pixels > total_pixels:

            raise RuntimeError(
                f"{image_name}: changed pixel count "
                f"{changed_pixels} exceeds total "
                f"pixel count {total_pixels}."
            )

        # ----------------------------------------------------
        # STORE
        # ----------------------------------------------------

        image_records.append({

            "Image":
                image_name,

            "Image_Index":
                image_index,

            "Changed_Row":
                row,

            "Changed_Column":
                col,

            "Original_Pixel":
                old_value,

            "Modified_Pixel":
                new_value,

            "Plaintext_Changed_Pixels":
                plaintext_changed_pixels,

            "Descriptor_Changed":
                bool(
                    descriptor_changed
                ),

            "NPCR_Percent":
                float(npcr),

            "UACI_Percent":
                float(uaci),

            "Changed_Cipher_Pixels":
                int(changed_pixels),

            "Total_Pixels":
                int(total_pixels),

            "Cipher_Changed_Percent":
                float(
                    changed_cipher_percent
                )
        })

        print(
            f"[CELL 42] "
            f"{image_index}/"
            f"{total_dataset_images} | "
            f"{image_name} | "
            f"NPCR={npcr:.4f}% | "
            f"UACI={uaci:.4f}% | "
            f"Changed="
            f"{changed_pixels}/"
            f"{total_pixels}"
        )

    # ========================================================
    # DATAFRAME
    # ========================================================

    image_results = pd.DataFrame(
        image_records
    )

    # ========================================================
    # SUMMARY
    # ========================================================

    npcr_summary = summarize_metric(
        image_results[
            "NPCR_Percent"
        ].values
    )

    uaci_summary = summarize_metric(
        image_results[
            "UACI_Percent"
        ].values
    )

    summary = pd.DataFrame([

        {
            "Metric":
                "NPCR_Percent",

            **npcr_summary
        },

        {
            "Metric":
                "UACI_Percent",

            **uaci_summary
        }

    ])

    return (
        image_results,
        summary
    )


# ============================================================
# LOAD COMPLETE DATASET
# ============================================================

dataset_images_cell42 = []

for image_path in valid_dataset_files:

    image = cv2.imread(
        str(image_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:

        print(
            f"Skipping unreadable image: "
            f"{image_path}"
        )

        continue

    image = np.asarray(
        image,
        dtype=np.uint8
    )

    dataset_images_cell42.append(
        (
            image_path.name,
            image
        )
    )


if len(dataset_images_cell42) == 0:

    raise RuntimeError(
        "No valid images are available "
        "for CELL 42."
    )


# ============================================================
# RUN COMPLETE DATASET
# ============================================================

(
    differential_results,
    differential_summary
) = run_differential_summary(
    dataset_images=dataset_images_cell42,
    secret_key=_TEST_SECRET_KEY_CELL42
)


# ============================================================
# DISPLAY
# ============================================================

print()
print("=" * 80)
print("CELL 42 — DIFFERENTIAL ATTACK SUMMARY")
print("=" * 80)

print()

print(
    f"Images processed: "
    f"{len(differential_results)}"
)

print()

print("Per-image results:")

print(
    differential_results.to_string(
        index=False
    )
)

print()

print("Dataset summary:")

print(
    differential_summary.to_string(
        index=False
    )
)


# ============================================================
# SAVE RESULTS
# ============================================================

cell42_output_path = (
    RESULT_DIR
    /
    "09_SECURITY"
    /
    "cell42_differential_all_images.csv"
)

cell42_output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

differential_results.to_csv(
    cell42_output_path,
    index=False
)


cell42_summary_path = (
    RESULT_DIR
    /
    "09_SECURITY"
    /
    "cell42_differential_summary.csv"
)

differential_summary.to_csv(
    cell42_summary_path,
    index=False
)


print()

print(
    "Per-image results saved to:"
)

print(
    cell42_output_path
)

print()

print(
    "Summary saved to:"
)

print(
    cell42_summary_path
)

print()

print(
    "Controlled differential test:"
)

print(
    "Exactly ONE plaintext pixel is modified."
)

print(
    "The SAME secret key is used."
)

print(
    "The SAME salt is used for each pair."
)

print(
    "The SAME nonce is used for each pair."
)

print(
    "NPCR/UACI are calculated directly on the "
    "final authenticated ciphertext."
)

print(
    "Notebook-wide calculate_npcr() is intentionally "
    "not used by CELL 42."
)

print(
    "Bidirectional ciphertext diffusion is enabled."
)

print()
print("CELL 42 COMPLETE")

Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
[CELL 42] 1/49 | Copy of Copy of image_0050.png | NPCR=99.5941% | UACI=33.6190% | Changed=65270/65536
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
[CELL 42] 2/49 | Copy of Copy of image_0051.png | NPCR=99.6353% | UACI=33.4517% | Changed=65297/65536
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
[CELL 42] 3/49 | Copy of Copy of image_0052.png | NPCR=99.6185% | UACI=33.4420% | Changed=65286/65536
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
[CELL 42] 4/49 | Copy of Copy of image_0053.png | NPCR=99.6185% | UACI=33.6211% | Changed=65286/65536
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
[CELL 42] 5/49 | Copy of Copy of image_0054.png | NPCR=99.6277% | UACI=33.4868% | Changed=65292/65536
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
[CELL 42] 6/49 | Copy 

In [ ]:
# ============================================================
# CELL 43 — THEORETICAL COMPLEXITY ANALYSIS
# ============================================================

import math
import pandas as pd
import numpy as np

def theoretical_complexity_analysis(
    image_shape,
    block_size=16
):
    """
    Estimate operation counts and report theoretical
    asymptotic complexity.

    N = H * W

    The implementation consists of:
        - one plaintext SHA-256
        - block SHA-256 operations
        - state updates
        - DNA encoding/decoding
        - block permutation
        - diffusion
    """

    H, W = image_shape
    N = H * W

    if H % block_size != 0 or W % block_size != 0:
        raise ValueError(
            "Image dimensions must be divisible by block_size."
        )

    number_of_blocks = (
        (H // block_size) *
        (W // block_size)
    )

    pixels_per_block = (
        block_size * block_size
    )

    estimated_operations = {
        "Image_Pixels_N": N,
        "Block_Size": block_size,
        "Number_of_Blocks": number_of_blocks,
        "Pixels_Per_Block": pixels_per_block,

        # Linear operations
        "Plaintext_SHA256": 1,
        "Block_SHA256": number_of_blocks,
        "State_SHA256": number_of_blocks,
        "DNA_Encoding_Pixel_Units": N,
        "DNA_Decoding_Pixel_Units": N,
        "Feedback_Diffusion_Pixel_Units": N,

        # Each block permutation acts on B^2 positions.
        "Permutation_Elements": (
            number_of_blocks *
            pixels_per_block
        )
    }

    result = pd.DataFrame(
        [
            {
                "Operation": key,
                "Estimated_Count": value
            }
            for key, value
            in estimated_operations.items()
        ]
    )

    result.loc[
        result["Operation"] == "Overall_Asymptotic_Complexity",
        "Estimated_Count"
    ] = "O(N)"

    overall = pd.DataFrame([
        {
            "Image_Size": f"{H}x{W}",
            "N_HxW": N,
            "Block_Size": block_size,
            "Number_of_Blocks": number_of_blocks,
            "Theoretical_Complexity": "O(N)",
            "Complexity_Condition": (
                "Fixed block size; bounded per-block permutation"
            )
        }
    ])

    return result, overall


complexity_operations, complexity_summary = (
    theoretical_complexity_analysis(
        image_shape=test_image.shape,
        block_size=CONFIG["BLOCK_SIZE"]
    )
)

print("=" * 80)
print("THEORETICAL COMPLEXITY ANALYSIS")
print("=" * 80)

print()
print(complexity_operations.to_string(index=False))

print()
print("Summary:")
print(
    complexity_summary.to_string(
        index=False
    )
)

print()
print(
    "Theoretical complexity is reported as O(N), "
    "where N = H × W, for fixed block size."
)

THEORETICAL COMPLEXITY ANALYSIS

                     Operation Estimated_Count
                Image_Pixels_N           65536
                    Block_Size              16
              Number_of_Blocks             256
              Pixels_Per_Block             256
              Plaintext_SHA256               1
                  Block_SHA256             256
                  State_SHA256             256
      DNA_Encoding_Pixel_Units           65536
      DNA_Decoding_Pixel_Units           65536
Feedback_Diffusion_Pixel_Units           65536
          Permutation_Elements           65536

Summary:
Image_Size  N_HxW  Block_Size  Number_of_Blocks Theoretical_Complexity                            Complexity_Condition
   256x256  65536          16               256                   O(N) Fixed block size; bounded per-block permutation

Theoretical complexity is reported as O(N), where N = H × W, for fixed block size.


/tmp/ipykernel_1335/413317335.py:77: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'O(N)' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result.loc[


In [ ]:
# ============================================================
# CELL 44 — MEMORY USAGE
# ============================================================

import os
import time
import gc
import psutil

def get_process_memory_mb():
    """
    Return current Python process resident memory in MB.
    """

    process = psutil.Process(
        os.getpid()
    )

    return (
        process.memory_info().rss /
        (1024 ** 2)
    )


def measure_encryption_memory(
    image,
    secret_key
):
    """
    Measure process memory immediately before and after
    the actual encryption operation.

    This is an implementation-level memory measurement,
    not the theoretical minimum memory requirement.
    """

    gc.collect()

    memory_before = (
        get_process_memory_mb()
    )

    start_time = time.perf_counter()

    cipher, encryption_state, block_states, (
        block_rules
    ), permutations, encryption_time = encrypt_image(
        image=image,
        secret_key=secret_key,
        save_internal=False
    )

    elapsed = (
        time.perf_counter() -
        start_time
    )

    gc.collect()

    memory_after = (
        get_process_memory_mb()
    )

    delta_memory = (
        memory_after -
        memory_before
    )

    result = {
        "Memory_Before_MB": memory_before,
        "Memory_After_MB": memory_after,
        "Delta_Memory_MB": delta_memory,
        "Encryption_Time_Seconds": elapsed
    }

    return (
        result,
        cipher
    )


# ------------------------------------------------------------
# TEST KEY
# Keep this consistent with CELL 29 and CELL 42
# ------------------------------------------------------------

_TEST_SECRET_KEY_CELL44 = (
    "DNA-Adaptive-V2-Test-Key-2026"
)


memory_result, memory_test_cipher = (
    measure_encryption_memory(
        image=test_image,
        secret_key=_TEST_SECRET_KEY_CELL44
    )
)


print("=" * 80)
print("MEMORY USAGE")
print("=" * 80)

for key, value in memory_result.items():

    if "Memory" in key:
        print(
            f"{key}: {value:.6f} MB"
        )
    else:
        print(
            f"{key}: {value:.6f} s"
        )

Backward ciphertext diffusion applied.
MEMORY USAGE
Memory_Before_MB: 270.355469 MB
Memory_After_MB: 270.359375 MB
Delta_Memory_MB: 0.003906 MB
Encryption_Time_Seconds: 0.748034 s


In [ ]:
# ============================================================
# CELL 45 — V2 ABLATION STUDY
# ============================================================

import os
import hashlib
import hmac
import time
import numpy as np
import pandas as pd


# ============================================================
# V2 ABLATION VARIANT ENCRYPTION
# ============================================================

def encrypt_ablation_variant(
    image,
    secret_key,
    variant="D",
    salt=None,
    nonce=None
):
    """
    Controlled ablation implementation for the current V2
    architecture.

    Variants
    --------
    A:
        V2 key schedule + Fixed DNA + DNA-XOR

    B:
        V2 key schedule + Adaptive DNA + DNA-XOR

    C:
        V2 key schedule + Adaptive DNA +
        Adaptive Permutation + DNA-XOR

    D:
        Full current V2 implementation from encrypt_image().

    The original and modified plaintexts can be encrypted
    with identical salt and nonce for controlled NPCR/UACI
    experiments.
    """

    # ========================================================
    # VALIDATE INPUT
    # ========================================================

    image = np.asarray(
        image,
        dtype=np.uint8
    )

    if image.ndim != 2:
        raise ValueError(
            "image must be a grayscale 2-D array."
        )

    if variant not in (
        "A",
        "B",
        "C",
        "D"
    ):
        raise ValueError(
            "variant must be A, B, C, or D."
        )

    H, W = image.shape

    B = int(
        CONFIG["BLOCK_SIZE"]
    )

    if H % B != 0 or W % B != 0:
        raise ValueError(
            f"Image dimensions {H}x{W} must be divisible "
            f"by block size {B}."
        )

        # ========================================================
    # VARIANT D
    #
    # IMPORTANT:
    # Use the actual current V2 implementation.
    #
    # The current encrypt_image() return value in CELL 26
    # contains diagnostic/timing information in the final
    # position, which is not guaranteed to be a scalar.
    #
    # Therefore measure runtime externally.
    # ========================================================

    if variant == "D":

        start_time = time.perf_counter()

        encryption_result = encrypt_image(
            image=image,
            secret_key=secret_key,
            salt=salt,
            nonce=nonce,
            save_internal=False
        )

        elapsed = (
            time.perf_counter()
            -
            start_time
        )

        # ----------------------------------------------------
        # Validate encrypt_image() output
        # ----------------------------------------------------

        if not isinstance(
            encryption_result,
            tuple
        ):

            raise RuntimeError(
                "encrypt_image() did not return a tuple."
            )

        if len(
            encryption_result
        ) == 0:

            raise RuntimeError(
                "encrypt_image() returned no ciphertext."
            )

        # ----------------------------------------------------
        # Current CELL 26 convention:
        #
        # encryption_result[0] = ciphertext
        #
        # Do NOT use encryption_result[5] as a numeric
        # encryption time.
        # ----------------------------------------------------

        cipher_image = np.asarray(
            encryption_result[0],
            dtype=np.uint8
        )

        if cipher_image.shape != image.shape:

            raise RuntimeError(
                "Variant D ciphertext shape mismatch."
            )

        return (
            cipher_image,
            float(elapsed)
        )

    # ========================================================
    # SECRET KEY
    # ========================================================

    if isinstance(
        secret_key,
        str
    ):

        secret_bytes = (
            secret_key.encode("utf-8")
        )

    elif isinstance(
        secret_key,
        bytes
    ):

        secret_bytes = secret_key

    else:

        raise TypeError(
            "secret_key must be str or bytes."
        )

    # ========================================================
    # CONTENT DESCRIPTOR
    #
    # Current notebook interface:
    #
    #     descriptor,
    #     descriptor_digest,
    #     feature_values
    # ========================================================

    (
        descriptor,
        descriptor_digest,
        feature_values
    ) = extract_content_descriptor(
        image
    )

    if descriptor_digest is None:
        raise RuntimeError(
            "extract_content_descriptor() returned "
            "no descriptor digest."
        )

    if isinstance(
        descriptor_digest,
        str
    ):

        descriptor_digest = bytes.fromhex(
            descriptor_digest
        )

    descriptor_digest = bytes(
        descriptor_digest
    )

    # ========================================================
    # SALT / NONCE
    # ========================================================

    if salt is None:
        salt = os.urandom(16)

    if nonce is None:
        nonce = os.urandom(16)

    salt = bytes(salt)
    nonce = bytes(nonce)

    # ========================================================
    # V2 KEY SCHEDULE
    # ========================================================

    key_schedule = derive_key_schedule(
        secret_key=secret_key,
        salt=salt,
        nonce=nonce,
        descriptor_digest=descriptor_digest
    )

    K_DNA = key_schedule["K_DNA"]
    K_PERM = key_schedule["K_PERM"]
    K_CHAOS = key_schedule["K_CHAOS"]
    K_DIFF = key_schedule["K_DIFF"]

    # ========================================================
    # INITIAL STATE
    # ========================================================

    state = hmac.new(
        K_DIFF,
        nonce +
        descriptor_digest +
        b"INITIAL_STATE",
        hashlib.sha256
    ).digest()

    # ========================================================
    # OUTPUT
    # ========================================================

    cipher_image = np.zeros_like(
        image,
        dtype=np.uint8
    )

    previous_cipher_block = np.zeros(
        B * B,
        dtype=np.uint8
    )

    # ========================================================
    # BLOCK INFORMATION
    # ========================================================

    blocks_per_row = W // B
    blocks_per_col = H // B

    chaos_transient = int(
        CONFIG.get(
            "CHAOS_TRANSIENT",
            1000
        )
    )

    # ========================================================
    # TIMING
    # ========================================================

    start_time = time.perf_counter()

    # ========================================================
    # PROCESS BLOCKS
    # ========================================================

    for block_row in range(
        blocks_per_col
    ):

        for block_col in range(
            blocks_per_row
        ):

            block_index = (
                block_row *
                blocks_per_row +
                block_col
            )

            index_bytes = (
                int(block_index).to_bytes(
                    8,
                    byteorder="big",
                    signed=False
                )
            )

            # ------------------------------------------------
            # BLOCK COORDINATES
            # ------------------------------------------------

            y0 = block_row * B
            y1 = y0 + B

            x0_block = block_col * B
            x1_block = x0_block + B

            block = np.asarray(
                image[
                    y0:y1,
                    x0_block:x1_block
                ],
                dtype=np.uint8
            ).copy()

            # ------------------------------------------------
            # ADAPTIVE DNA CONTROL
            # ------------------------------------------------

            Z_i = hmac.new(
                K_DNA,
                state +
                previous_cipher_block.tobytes() +
                descriptor_digest +
                index_bytes +
                b"ADAPTIVE_CONTROL",
                hashlib.sha256
            ).digest()

            # ------------------------------------------------
            # DNA RULE
            # ------------------------------------------------

            if variant == "A":

                rule_index = 1

                rule = DNA_RULES[
                    rule_index
                ]

            else:

                adaptive_result = (
                    select_adaptive_dna_rule(
                        Z_i
                    )
                )

                if (
                    isinstance(
                        adaptive_result,
                        tuple
                    )
                    and len(
                        adaptive_result
                    ) == 3
                ):

                    (
                        q_i,
                        r_i,
                        selected_rule
                    ) = adaptive_result

                    rule_index = int(
                        r_i
                    )

                    rule = selected_rule

                else:

                    rule = adaptive_result

            # ------------------------------------------------
            # ADAPTIVE PERMUTATION
            #
            # Only Variant C contains permutation.
            # ------------------------------------------------

            if variant == "C":

                permutation_state = hmac.new(
                    K_PERM,
                    state +
                    descriptor_digest +
                    index_bytes +
                    b"PERMUTATION_STATE",
                    hashlib.sha256
                ).digest()

                (
                    permutation,
                    inverse_permutation,
                    permutation_seed
                ) = generate_block_permutation(
                    S_i=permutation_state,
                    block_index=block_index,
                    block_size=B
                )

                working_block = (
                    block.reshape(-1)[
                        permutation
                    ].reshape(
                        B,
                        B
                    )
                )

            else:

                working_block = (
                    block.copy()
                )

            # ------------------------------------------------
            # DNA ENCODING
            # ------------------------------------------------

            plaintext_dna = dna_encode_array(
                working_block,
                rule
            )

            plaintext_dna = (
                plaintext_dna.reshape(
                    B,
                    B,
                    4
                )
            )

            # ------------------------------------------------
            # BLOCK CHAOS
            # ------------------------------------------------

            chaos_state = hmac.new(
                K_CHAOS,
                state +
                descriptor_digest +
                index_bytes +
                b"CHAOS",
                hashlib.sha256
            ).digest()

            (
                x0_chaos,
                r_chaos,
                _,
                _
            ) = state_to_chaotic_parameters(
                chaos_state
            )

            block_pixels = B * B

            chaotic_sequence = (
                generate_logistic_sequence(
                    x0=x0_chaos,
                    r=r_chaos,
                    sequence_length=block_pixels,
                    transient=chaos_transient
                )
            )

            block_chaos = np.asarray(
                chaotic_to_bytes(
                    chaotic_sequence
                ),
                dtype=np.uint8
            ).reshape(-1)

            if len(
                block_chaos
            ) != block_pixels:

                raise RuntimeError(
                    "Incorrect chaotic block length."
                )

            chaos_dna = dna_encode_array(
                block_chaos,
                rule
            )

            chaos_dna = (
                chaos_dna.reshape(
                    B,
                    B,
                    4
                )
            )

            # ------------------------------------------------
            # DNA-XOR
            # ------------------------------------------------

            X_i = dna_xor(
                plaintext_dna,
                chaos_dna
            )

            # ------------------------------------------------
            # NO FEEDBACK DIFFUSION
            #
            # A/B/C are reduced variants.
            # ------------------------------------------------

            cipher_block = dna_decode_array(
                X_i,
                rule
            )

            cipher_block = np.asarray(
                cipher_block,
                dtype=np.uint8
            ).reshape(
                B,
                B
            )

            # ------------------------------------------------
            # STORE BLOCK
            # ------------------------------------------------

            cipher_image[
                y0:y1,
                x0_block:x1_block
            ] = cipher_block

            # ------------------------------------------------
            # CIPHERTEXT FEEDBACK
            # ------------------------------------------------

            previous_cipher_block = (
                cipher_block.reshape(-1).copy()
            )

            # ------------------------------------------------
            # CIPHERTEXT-ONLY STATE EVOLUTION
            # ------------------------------------------------

            state = evolve_state_v2(
                current_state=state,
                cipher_block=cipher_block,
                block_index=block_index,
                k_diff=K_DIFF
            )

    elapsed = (
        time.perf_counter() -
        start_time
    )

    # ========================================================
    # FINAL VALIDATION
    # ========================================================

    if cipher_image.shape != image.shape:
        raise RuntimeError(
            "Ablation cipher shape mismatch."
        )

    if cipher_image.dtype != np.uint8:
        raise RuntimeError(
            "Ablation cipher dtype mismatch."
        )

    return (
        cipher_image,
        float(elapsed)
    )


# ============================================================
# RUN ABLATION STUDY
# ============================================================

def run_ablation_study(
    image,
    secret_key
):
    """
    Run controlled V2 ablation study.

    A = Fixed DNA
    B = Adaptive DNA
    C = Adaptive DNA + Adaptive Permutation
    D = Full current V2 method

    Metrics:
        Entropy
        Horizontal correlation
        Vertical correlation
        Diagonal correlation
        NPCR
        UACI
        Changed pixels
        Encryption runtime

    Exactly one plaintext pixel is modified by +1 modulo 256.

    Original and modified plaintexts use identical salt and
    nonce for each variant.
    """

    variants = {

        "A":
            "V2 key schedule + Fixed DNA + DNA-XOR",

        "B":
            "V2 key schedule + Adaptive DNA + DNA-XOR",

        "C":
            "V2 key schedule + Adaptive DNA + "
            "Adaptive Permutation + DNA-XOR",

        "D":
            "Full Proposed V2 Method"
    }

    records = []

    image = np.asarray(
        image,
        dtype=np.uint8
    )

    # ========================================================
    # MODIFY EXACTLY ONE PLAINTEXT PIXEL
    # ========================================================

    modified = image.copy()

    row = image.shape[0] // 2
    col = image.shape[1] // 2

    original_pixel = int(
        modified[row, col]
    )

    modified[row, col] = np.uint8(
        (
            original_pixel + 1
        ) % 256
    )

    # Verify exactly one plaintext pixel changed.
    plaintext_changed = np.count_nonzero(
        image != modified
    )

    if plaintext_changed != 1:
        raise RuntimeError(
            "Controlled plaintext modification failed: "
            f"{plaintext_changed} pixels changed."
        )

    # ========================================================
    # RUN VARIANTS
    # ========================================================

    for variant_id, variant_name in variants.items():

        print()
        print(
            f"Running ablation variant {variant_id}: "
            f"{variant_name}"
        )

        # ----------------------------------------------------
        # SAME SALT / NONCE FOR ORIGINAL + MODIFIED
        # ----------------------------------------------------

        salt = os.urandom(16)
        nonce = os.urandom(16)

        # ----------------------------------------------------
        # ENCRYPT ORIGINAL
        # ----------------------------------------------------

        cipher, encryption_time = (
            encrypt_ablation_variant(
                image=image,
                secret_key=secret_key,
                variant=variant_id,
                salt=salt,
                nonce=nonce
            )
        )

        # ----------------------------------------------------
        # ENTROPY
        # ----------------------------------------------------

        entropy = calculate_entropy(
            cipher
        )

        # ----------------------------------------------------
        # CORRELATION
        # ----------------------------------------------------

        correlations = (
            calculate_directional_correlations(
                cipher
            )
        )

        # ----------------------------------------------------
        # ENCRYPT MODIFIED
        # ----------------------------------------------------

        modified_cipher, _ = (
            encrypt_ablation_variant(
                image=modified,
                secret_key=secret_key,
                variant=variant_id,
                salt=salt,
                nonce=nonce
            )
        )

        # ----------------------------------------------------
        # NPCR
        #
        # Use notebook helper for NPCR value and changed
        # pixel count, but do NOT assume its third return
        # value is a scalar.
        # ----------------------------------------------------

        npcr_result = calculate_npcr(
            cipher,
            modified_cipher
        )

        npcr = float(
            np.asarray(
                npcr_result[0]
            ).squeeze()
        )

        changed_pixels = int(
            np.asarray(
                npcr_result[1]
            ).squeeze()
        )

        # ----------------------------------------------------
        # DIRECT CHANGED-PIXEL CALCULATION
        #
        # This is deliberately independent of the helper's
        # third return value.
        # ----------------------------------------------------

        changed_mask = (
            cipher != modified_cipher
        )

        direct_changed_pixels = int(
            np.count_nonzero(
                changed_mask
            )
        )

        total_pixels = int(
            cipher.size
        )

        changed_percent = (
            100.0 *
            direct_changed_pixels /
            total_pixels
        )

        # ----------------------------------------------------
        # CONSISTENCY CHECK
        # ----------------------------------------------------

        if changed_pixels != direct_changed_pixels:

            changed_pixels = (
                direct_changed_pixels
            )

        # ----------------------------------------------------
        # UACI
        # ----------------------------------------------------

        uaci = float(
            np.asarray(
                calculate_uaci(
                    cipher,
                    modified_cipher
                )
            ).squeeze()
        )

        # ----------------------------------------------------
        # STORE RESULTS
        # ----------------------------------------------------

        records.append({

            "Variant":
                variant_id,

            "Description":
                variant_name,

            "Entropy":
                float(entropy),

            "Correlation_H":
                float(
                    correlations["Horizontal"]
                ),

            "Correlation_V":
                float(
                    correlations["Vertical"]
                ),

            "Correlation_D":
                float(
                    correlations["Diagonal"]
                ),

            "NPCR_Percent":
                float(npcr),

            "UACI_Percent":
                float(uaci),

            "Changed_Pixels":
                int(direct_changed_pixels),

            "Changed_Pixels_Percent":
                float(changed_percent),

            "Encryption_Time_Seconds":
                float(encryption_time)
        })

        # ----------------------------------------------------
        # DISPLAY
        # ----------------------------------------------------

        print(
            f"  Entropy : {entropy:.6f}"
        )

        print(
            f"  Corr-H  : "
            f"{correlations['Horizontal']:.6f}"
        )

        print(
            f"  Corr-V  : "
            f"{correlations['Vertical']:.6f}"
        )

        print(
            f"  Corr-D  : "
            f"{correlations['Diagonal']:.6f}"
        )

        print(
            f"  NPCR    : {npcr:.6f}%"
        )

        print(
            f"  UACI    : {uaci:.6f}%"
        )

        print(
            f"  Changed : "
            f"{direct_changed_pixels} / "
            f"{total_pixels} "
            f"({changed_percent:.6f}%)"
        )

        print(
            f"  Time    : "
            f"{encryption_time:.6f} s"
        )

    return pd.DataFrame(
        records
    )


# ============================================================
# RUN ABLATION STUDY
# ============================================================

_TEST_SECRET_KEY_CELL45 = (
    "DNA-Adaptive-V2-Test-Key-2026"
)

ablation_results = run_ablation_study(
    image=test_image,
    secret_key=_TEST_SECRET_KEY_CELL45
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print()
print("=" * 80)
print("V2 ABLATION STUDY")
print("=" * 80)

print(
    ablation_results.to_string(
        index=False
    )
)

print()
print(
    "Ablation study: COMPLETED"
)


Running ablation variant A: V2 key schedule + Fixed DNA + DNA-XOR
  Entropy : 7.997491
  Corr-H  : -0.007005
  Corr-V  : 0.016180
  Corr-D  : -0.006209
  NPCR    : 46.305847%
  UACI    : 15.529109%
  Changed : 30347 / 65536 (46.305847%)
  Time    : 0.482326 s

Running ablation variant B: V2 key schedule + Adaptive DNA + DNA-XOR
  Entropy : 7.997095
  Corr-H  : 0.000168
  Corr-V  : 0.008724
  Corr-D  : -0.004854
  NPCR    : 46.299744%
  UACI    : 15.683642%
  Changed : 30343 / 65536 (46.299744%)
  Time    : 0.479824 s

Running ablation variant C: V2 key schedule + Adaptive DNA + Adaptive Permutation + DNA-XOR
  Entropy : 7.996783
  Corr-H  : -0.000114
  Corr-V  : 0.001650
  Corr-D  : 0.002290
  NPCR    : 46.318054%
  UACI    : 15.592388%
  Changed : 30355 / 65536 (46.318054%)
  Time    : 0.500987 s

Running ablation variant D: Full Proposed V2 Method
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
  Entropy : 7.997437
  Corr-H  : -0.001879
  Corr-V  : -0.0

In [ ]:

# ============================================================
# CELL 45B — METRIC HELPERS
# ============================================================

"""
PURPOSE:
    Define correctness metrics required by:
        CELL 46
        CELL 48
        CELL 49
        CELL 53

METRICS:
    MSE
    PSNR

FORMULAS:

    MSE =
        (1 / (H*W))
        * Σ [I(i,j) - R(i,j)]²

    PSNR =
        10 * log10(MAX² / MSE)

    For exact reconstruction:
        MSE = 0
        PSNR = infinity
"""


import numpy as np


# ============================================================
# MSE
# ============================================================

def calculate_mse(
    original,
    reconstructed
):

    original = np.asarray(
        original,
        dtype=np.float64
    )

    reconstructed = np.asarray(
        reconstructed,
        dtype=np.float64
    )

    if original.shape != reconstructed.shape:

        raise ValueError(
            "MSE requires identical image shapes. "
            f"Original={original.shape}, "
            f"Reconstructed={reconstructed.shape}"
        )

    difference = (
        original -
        reconstructed
    )

    mse = np.mean(
        difference ** 2
    )

    return float(mse)


# ============================================================
# PSNR
# ============================================================

def calculate_psnr(
    original,
    reconstructed,
    max_pixel_value=255.0
):

    mse = calculate_mse(
        original,
        reconstructed
    )

    if mse == 0:

        return float("inf")

    psnr = (
        10.0 *
        np.log10(
            (
                max_pixel_value ** 2
            ) /
            mse
        )
    )

    return float(psnr)


# ============================================================
# VALIDATION
# ============================================================

print("=" * 80)
print("CELL 45B — METRIC HELPERS")
print("=" * 80)

print()
print("calculate_mse()  : AVAILABLE")
print("calculate_psnr() : AVAILABLE")
print()
print("CELL 45B COMPLETE")

CELL 45B — METRIC HELPERS

calculate_mse()  : AVAILABLE
calculate_psnr() : AVAILABLE

CELL 45B COMPLETE


In [ ]:
# ============================================================
# CELL 46 — BLOCK-SIZE ABLATION
# ============================================================

import time
import numpy as np
import pandas as pd


def run_block_size_ablation(
    image,
    secret_key,
    block_sizes=(8, 16, 32)
):

    image = np.asarray(
        image,
        dtype=np.uint8
    )

    if image.ndim != 2:
        raise ValueError(
            "image must be a grayscale 2-D array."
        )

    H, W = image.shape

    original_block_size = int(
        CONFIG["BLOCK_SIZE"]
    )

    records = []

    try:

        for B in block_sizes:

            B = int(B)

            if B <= 0:
                raise ValueError(
                    "Block size must be positive."
                )

            if H % B != 0 or W % B != 0:

                print(
                    f"Skipping block size {B}x{B}: "
                    f"{H}x{W} is not divisible by {B}."
                )

                continue

            print()
            print(
                f"Running block size ablation: "
                f"{B}x{B}"
            )

            # ------------------------------------------------
            # SET BLOCK SIZE
            # ------------------------------------------------

            CONFIG["BLOCK_SIZE"] = B

            # ------------------------------------------------
            # GENERATE CONTROLLED SALT / NONCE
            #
            # The same salt and nonce are used for the
            # original and one-pixel-modified plaintext.
            # ------------------------------------------------

            salt = os.urandom(16)
            nonce = os.urandom(16)

            # ------------------------------------------------
            # ENCRYPT ORIGINAL
            # ------------------------------------------------

            start = time.perf_counter()

            encryption_result = encrypt_image(
                image=image,
                secret_key=secret_key,
                salt=salt,
                nonce=nonce,
                save_internal=False
            )

            enc_time = (
                time.perf_counter()
                -
                start
            )

            if not isinstance(
                encryption_result,
                tuple
            ):

                raise RuntimeError(
                    "encrypt_image() did not return a tuple."
                )

            if len(
                encryption_result
            ) == 0:

                raise RuntimeError(
                    "encrypt_image() returned no ciphertext."
                )

            cipher = np.asarray(
                encryption_result[0],
                dtype=np.uint8
            )

            encryption_state = (
                encryption_result[1]
                if len(encryption_result) > 1
                else None
            )

            # ------------------------------------------------
            # VERIFY CIPHERTEXT
            # ------------------------------------------------

            if cipher.shape != image.shape:

                raise RuntimeError(
                    "Ciphertext shape mismatch."
                )

            # ------------------------------------------------
            # DIFFERENTIAL TEST
            # ------------------------------------------------

            modified = image.copy()

            row = H // 2
            col = W // 2

            original_pixel = int(
                modified[row, col]
            )

            modified[row, col] = np.uint8(
                (
                    original_pixel + 1
                ) % 256
            )

            plaintext_changed = int(
                np.count_nonzero(
                    image != modified
                )
            )

            if plaintext_changed != 1:

                raise RuntimeError(
                    "Controlled plaintext modification failed: "
                    f"{plaintext_changed} pixels changed."
                )

            # ------------------------------------------------
            # ENCRYPT MODIFIED IMAGE
            # ------------------------------------------------

            modified_result = encrypt_image(
                image=modified,
                secret_key=secret_key,
                salt=salt,
                nonce=nonce,
                save_internal=False
            )

            if not isinstance(
                modified_result,
                tuple
            ):

                raise RuntimeError(
                    "Modified encrypt_image() "
                    "did not return a tuple."
                )

            modified_cipher = np.asarray(
                modified_result[0],
                dtype=np.uint8
            )

            if modified_cipher.shape != image.shape:

                raise RuntimeError(
                    "Modified ciphertext shape mismatch."
                )

            # ------------------------------------------------
            # DIRECT NPCR
            #
            # Avoid assumptions about the return format of
            # calculate_npcr().
            # ------------------------------------------------

            difference_mask = (
                cipher != modified_cipher
            )

            changed_pixels = int(
                np.count_nonzero(
                    difference_mask
                )
            )

            total_pixels = int(
                cipher.size
            )

            npcr = (
                100.0 *
                changed_pixels /
                total_pixels
            )

            # ------------------------------------------------
            # DIRECT UACI
            # ------------------------------------------------

            cipher_int = cipher.astype(
                np.int16
            )

            modified_cipher_int = (
                modified_cipher.astype(
                    np.int16
                )
            )

            uaci = float(
                np.mean(
                    np.abs(
                        cipher_int -
                        modified_cipher_int
                    )
                )
                / 255.0
                * 100.0
            )

            # ------------------------------------------------
            # ENTROPY
            # ------------------------------------------------

            entropy = calculate_entropy(
                cipher
            )

            # ------------------------------------------------
            # CORRELATION
            # ------------------------------------------------

            correlations = (
                calculate_directional_correlations(
                    cipher
                )
            )

            # ------------------------------------------------
            # DECRYPT ORIGINAL
            # ------------------------------------------------

            if encryption_state is None:

                raise RuntimeError(
                    "encrypt_image() did not return "
                    "the encryption state required "
                    "for metadata."
                )

            if (
                not isinstance(
                    encryption_state,
                    dict
                )
            ):

                raise RuntimeError(
                    "Unexpected encryption_state format."
                )

            metadata = (
                encryption_state.get(
                    "metadata"
                )
            )

            if metadata is None:

                raise RuntimeError(
                    "Encryption metadata not found."
                )

            start = time.perf_counter()

            decryption_result = decrypt_image(
                cipher_image=cipher,
                metadata=metadata,
                secret_key=secret_key,
                save_internal=False
            )

            dec_time = (
                time.perf_counter()
                -
                start
            )

            if not isinstance(
                decryption_result,
                tuple
            ):

                raise RuntimeError(
                    "decrypt_image() did not return a tuple."
                )

            if len(
                decryption_result
            ) == 0:

                raise RuntimeError(
                    "decrypt_image() returned no image."
                )

            decrypted = np.asarray(
                decryption_result[0],
                dtype=np.uint8
            )

            # ------------------------------------------------
            # CORRECTNESS
            # ------------------------------------------------

            mse = calculate_mse(
                image,
                decrypted
            )

            exact = bool(
                np.array_equal(
                    image,
                    decrypted
                )
            )

            # ------------------------------------------------
            # RECORD
            # ------------------------------------------------

            records.append({

                "Block_Size":
                    f"{B}x{B}",

                "Block_Size_Pixels":
                    B * B,

                "Number_of_Blocks":
                    (H // B) *
                    (W // B),

                "Entropy":
                    float(entropy),

                "Correlation_H":
                    float(
                        correlations["Horizontal"]
                    ),

                "Correlation_V":
                    float(
                        correlations["Vertical"]
                    ),

                "Correlation_D":
                    float(
                        correlations["Diagonal"]
                    ),

                "NPCR_Percent":
                    float(npcr),

                "UACI_Percent":
                    float(uaci),

                "Changed_Pixels":
                    int(changed_pixels),

                "Changed_Pixels_Percent":
                    float(
                        100.0 *
                        changed_pixels /
                        total_pixels
                    ),

                "Encryption_Time_Seconds":
                    float(enc_time),

                "Decryption_Time_Seconds":
                    float(dec_time),

                "MSE":
                    float(mse),

                "Exact_Reconstruction":
                    exact
            })

            # ------------------------------------------------
            # DISPLAY
            # ------------------------------------------------

            print(
                f"  Entropy : "
                f"{entropy:.6f}"
            )

            print(
                f"  Corr-H  : "
                f"{correlations['Horizontal']:.6f}"
            )

            print(
                f"  Corr-V  : "
                f"{correlations['Vertical']:.6f}"
            )

            print(
                f"  Corr-D  : "
                f"{correlations['Diagonal']:.6f}"
            )

            print(
                f"  NPCR    : "
                f"{npcr:.6f}%"
            )

            print(
                f"  UACI    : "
                f"{uaci:.6f}%"
            )

            print(
                f"  Changed : "
                f"{changed_pixels} / "
                f"{total_pixels} "
                f"({100.0 * changed_pixels / total_pixels:.6f}%)"
            )

            print(
                f"  Encrypt : "
                f"{enc_time:.6f} s"
            )

            print(
                f"  Decrypt : "
                f"{dec_time:.6f} s"
            )

            print(
                f"  MSE     : "
                f"{mse:.12f}"
            )

            print(
                f"  Exact   : "
                f"{exact}"
            )

    finally:

        # ----------------------------------------------------
        # ALWAYS RESTORE ORIGINAL BLOCK SIZE
        # ----------------------------------------------------

        CONFIG["BLOCK_SIZE"] = (
            original_block_size
        )

    return pd.DataFrame(
        records
    )


# ============================================================
# RUN BLOCK-SIZE ABLATION
# ============================================================

_TEST_SECRET_KEY_CELL46 = (
    "DNA-Adaptive-V2-Test-Key-2026"
)

block_size_ablation_results = (
    run_block_size_ablation(
        image=test_image,
        secret_key=_TEST_SECRET_KEY_CELL46,
        block_sizes=(8, 16, 32)
    )
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print()
print("=" * 80)
print("V2 BLOCK-SIZE ABLATION")
print("=" * 80)

print(
    block_size_ablation_results.to_string(
        index=False
    )
)


Running block size ablation: 8x8
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
  Entropy : 7.997089
  Corr-H  : 0.001645
  Corr-V  : 0.001915
  Corr-D  : -0.003092
  NPCR    : 99.595642%
  UACI    : 33.354971%
  Changed : 65271 / 65536 (99.595642%)
  Encrypt : 2.450491 s
  Decrypt : 1.391143 s
  MSE     : 0.000000000000
  Exact   : True

Running block size ablation: 16x16
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
  Entropy : 7.997066
  Corr-H  : -0.002348
  Corr-V  : 0.003615
  Corr-D  : -0.000375
  NPCR    : 99.563599%
  UACI    : 33.369745%
  Changed : 65250 / 65536 (99.563599%)
  Encrypt : 0.724428 s
  Decrypt : 0.734343 s
  MSE     : 0.000000000000
  Exact   : True

Running block size ablation: 32x32
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
  Entropy : 7.997083
  Corr-H  : 0.003312
  Corr-V  : 0.002411
  Corr-D  : -0.005211
  NPCR    : 99.586487%
  UACI    : 33.245221%
  Change

In [ ]:
# ============================================================
# CELL 47 — CHAOTIC-SEQUENCE ANALYSIS — V2
# ============================================================

"""
PURPOSE:
    Analyze the finite-precision Logistic chaotic sequence
    used by the UPDATED DNA-ADAPTIVE-V2 encryption architecture.

The analysis uses:
    - local V2 test secret
    - content descriptor
    - descriptor digest
    - PBKDF2/HKDF-derived V2 key schedule
    - K_DIFF
    - fixed reproducible analysis nonce
    - state_to_chaotic_parameters()
    - generate_logistic_sequence()

This cell does NOT use the old plaintext SHA-256 bootstrap.
"""


# ============================================================
# IMPORTS
# ============================================================

import hashlib
import hmac
import numpy as np
import pandas as pd

from pathlib import Path


# ============================================================
# LOCAL TEST SECRET
# ============================================================

_TEST_SECRET_KEY_CELL47 = (
    "DNA-Adaptive-V2-Test-Key-2026"
)


# ============================================================
# REQUIRED VARIABLE CHECK
# ============================================================

required_cell47_variables = [

    "test_image",
    "CONFIG",
    "state_to_chaotic_parameters",
    "generate_logistic_sequence",
    "extract_content_descriptor",
    "derive_key_schedule"

]


missing_cell47_variables = [

    name
    for name in required_cell47_variables
    if name not in globals()

]


if missing_cell47_variables:

    raise NameError(
        "CELL 47 is missing required V2 variables/functions:\n"
        +
        "\n".join(
            f"    {name}"
            for name in missing_cell47_variables
        )
        +
        "\n\n"
        "Run the updated V2 architecture cells before "
        "running CELL 47."
    )


# ============================================================
# SECRET → BYTES
# ============================================================

secret_bytes = (
    _TEST_SECRET_KEY_CELL47.encode(
        "utf-8"
    )
)


# ============================================================
# CREATE CONTENT DESCRIPTOR
# ============================================================

test_image_array = np.asarray(
    test_image,
    dtype=np.uint8
)


(
    descriptor,
    descriptor_digest,
    feature_values
) = extract_content_descriptor(
    test_image_array
)


if descriptor_digest is None:

    raise RuntimeError(
        "extract_content_descriptor() returned "
        "no descriptor digest."
    )


if isinstance(
    descriptor_digest,
    str
):

    descriptor_digest = bytes.fromhex(
        descriptor_digest
    )


descriptor_digest = bytes(
    descriptor_digest
)


# ============================================================
# FIXED EXPERIMENTAL SALT / NONCE
# ============================================================
#
# These values are ONLY for reproducible analysis.
#
# Production encryption must continue to use fresh random
# salt and nonce values.
# ============================================================

analysis_salt = hashlib.sha256(
    b"CELL47-ANALYSIS-SALT"
    +
    secret_bytes
).digest()[
    :int(
        CONFIG["SALT_BYTES"]
    )
]


analysis_nonce = hashlib.sha256(
    b"CELL47-ANALYSIS-NONCE"
    +
    secret_bytes
).digest()[
    :int(
        CONFIG["NONCE_BYTES"]
    )
]


# ============================================================
# DERIVE V2 KEY SCHEDULE
# ============================================================

analysis_keys = derive_key_schedule(

    secret_key=
        _TEST_SECRET_KEY_CELL47,

    salt=
        analysis_salt,

    nonce=
        analysis_nonce,

    descriptor_digest=
        descriptor_digest

)


required_keys = [
    "K_DNA",
    "K_PERM",
    "K_DIFF",
    "K_CHAOS",
    "K_AUTH"
]


missing_keys = [

    key
    for key in required_keys
    if key not in analysis_keys

]


if missing_keys:

    raise KeyError(
        "derive_key_schedule() did not return "
        "required V2 keys: "
        +
        ", ".join(
            missing_keys
        )
    )


# ============================================================
# BUILD V2 INITIAL STATE
# ============================================================
#
# Match the current CELL 26 bootstrap:
#
#     HMAC(
#         K_DIFF,
#         nonce +
#         descriptor_digest +
#         b"INITIAL_STATE"
#     )
#
# This is the current V2 convention.
# ============================================================

test_state = hmac.new(

    analysis_keys["K_DIFF"],

    (
        analysis_nonce
        +
        descriptor_digest
        +
        b"INITIAL_STATE"
    ),

    hashlib.sha256

).digest()


# ============================================================
# RESOLVE CHAOTIC PARAMETERS
# ============================================================

def resolve_chaotic_parameters(
    state
):

    parameters = (
        state_to_chaotic_parameters(
            state
        )
    )


    if not isinstance(
        parameters,
        (tuple, list, np.ndarray)
    ):

        raise TypeError(
            "state_to_chaotic_parameters() must "
            "return (x0, r, u, v)."
        )


    if len(parameters) < 4:

        raise ValueError(
            "Expected four values from "
            "state_to_chaotic_parameters(): "
            "x0, r, u, v."
        )


    x0 = float(
        parameters[0]
    )

    r = float(
        parameters[1]
    )

    u = int(
        parameters[2]
    )

    v = int(
        parameters[3]
    )


    return (
        x0,
        r,
        u,
        v
    )


# ============================================================
# GENERATE TEST CHAOTIC SEQUENCE
# ============================================================

def generate_test_chaotic_sequence(
    state,
    N=100000
):

    (
        x0,
        r,
        u,
        v
    ) = resolve_chaotic_parameters(
        state
    )


    chaos_transient = int(
        CONFIG.get(
            "CHAOS_TRANSIENT",
            1000
        )
    )


    sequence = generate_logistic_sequence(

        x0=
            x0,

        r=
            r,

        sequence_length=
            N,

        transient=
            chaos_transient

    )


    sequence = np.asarray(
        sequence,
        dtype=np.float64
    )


    return (

        x0,
        r,
        u,
        v,
        sequence

    )


# ============================================================
# AUTOCORRELATION
# ============================================================

def calculate_autocorrelation(
    sequence,
    lag=1
):

    sequence = np.asarray(
        sequence,
        dtype=np.float64
    )


    if len(sequence) <= lag:

        return np.nan


    x = sequence[:-lag]

    y = sequence[lag:]


    std_x = np.std(x)

    std_y = np.std(y)


    if (
        std_x == 0
        or
        std_y == 0
    ):

        return np.nan


    return float(
        np.corrcoef(
            x,
            y
        )[0, 1]
    )


# ============================================================
# MAIN CHAOTIC ANALYSIS
# ============================================================

def chaotic_sequence_analysis(
    state,
    N=100000
):

    (
        x0,
        r,
        u,
        v,
        sequence
    ) = generate_test_chaotic_sequence(

        state=
            state,

        N=
            N

    )


    finite_mask = np.isfinite(
        sequence
    )

    sequence = sequence[
        finite_mask
    ]


    if len(sequence) == 0:

        raise ValueError(
            "Chaotic sequence contains "
            "no finite values."
        )


    unique_values = np.unique(
        sequence
    )


    duplicate_count = (
        len(sequence)
        -
        len(unique_values)
    )


    result = {

        "Initial_x0":
            float(x0),

        "Control_r":
            float(r),

        "u":
            int(u),

        "v":
            int(v),

        "Samples":
            int(len(sequence)),

        "Mean":
            float(
                np.mean(sequence)
            ),

        "Variance":
            float(
                np.var(sequence)
            ),

        "Std":
            float(
                np.std(sequence)
            ),

        "Minimum":
            float(
                np.min(sequence)
            ),

        "Maximum":
            float(
                np.max(sequence)
            ),

        "Unique_Values":
            int(
                len(unique_values)
            ),

        "Duplicate_Count":
            int(
                duplicate_count
            ),

        "Lag_1":
            calculate_autocorrelation(
                sequence,
                lag=1
            ),

        "Lag_2":
            calculate_autocorrelation(
                sequence,
                lag=2
            ),

        "Lag_5":
            calculate_autocorrelation(
                sequence,
                lag=5
            ),

        "Lag_10":
            calculate_autocorrelation(
                sequence,
                lag=10
            )

    }


    return (
        result,
        sequence
    )


# ============================================================
# RUN MAIN ANALYSIS
# ============================================================

(
    chaos_result,
    chaos_sequence
) = chaotic_sequence_analysis(

    state=
        test_state,

    N=
        100000

)


# ============================================================
# DISPLAY MAIN RESULTS
# ============================================================

print("=" * 80)
print("CELL 47 — V2 CHAOTIC-SEQUENCE ANALYSIS")
print("=" * 80)

print()

print(
    "Architecture: "
    "PBKDF2 + HKDF + Content Descriptor + "
    "Ciphertext-Compatible State"
)

print()

print(
    "Test secret:"
    " local CELL 47 test key"
)

print()

print(
    "Content descriptor:"
)

for key, value in feature_values.items():

    print(
        f"    {key}: {value}"
    )

print()

print(
    "Chaotic sequence statistics:"
)

for key, value in chaos_result.items():

    print(
        f"    {key}: {value}"
    )


# ============================================================
# x0 SENSITIVITY
# ============================================================

def chaotic_parameter_sensitivity_x0(

    x0,
    r,

    N=10000,

    delta=1e-14

):

    x0_modified = min(

        max(

            x0 + delta,

            np.finfo(
                float
            ).eps

        ),

        1.0
        -
        np.finfo(
            float
        ).eps

    )


    chaos_transient = int(
        CONFIG.get(
            "CHAOS_TRANSIENT",
            1000
        )
    )


    seq1 = generate_logistic_sequence(

        x0=
            x0,

        r=
            r,

        sequence_length=
            N,

        transient=
            chaos_transient

    )


    seq2 = generate_logistic_sequence(

        x0=
            x0_modified,

        r=
            r,

        sequence_length=
            N,

        transient=
            chaos_transient

    )


    seq1 = np.asarray(
        seq1,
        dtype=np.float64
    )

    seq2 = np.asarray(
        seq2,
        dtype=np.float64
    )


    min_length = min(
        len(seq1),
        len(seq2)
    )


    seq1 = seq1[
        :min_length
    ]

    seq2 = seq2[
        :min_length
    ]


    different_values = int(
        np.count_nonzero(
            seq1 != seq2
        )
    )


    return {

        "Original_x0":
            float(x0),

        "Modified_x0":
            float(x0_modified),

        "Delta":
            float(
                x0_modified - x0
            ),

        "Mean_Absolute_Difference":
            float(
                np.mean(
                    np.abs(
                        seq1 - seq2
                    )
                )
            ),

        "Different_Values":
            different_values,

        "Different_Value_Percent":
            float(
                100.0
                *
                different_values
                /
                max(
                    len(seq1),
                    1
                )
            ),

        "Total_Values":
            int(
                len(seq1)
            )

    }


x0_sensitivity = (
    chaotic_parameter_sensitivity_x0(

        x0=
            chaos_result[
                "Initial_x0"
            ],

        r=
            chaos_result[
                "Control_r"
            ]

    )
)


print()

print(
    "x0 sensitivity:"
)

for key, value in x0_sensitivity.items():

    print(
        f"    {key}: {value}"
    )


# ============================================================
# r SENSITIVITY
# ============================================================

def chaotic_parameter_sensitivity_r(

    x0,
    r,

    N=10000,

    delta=1e-12

):

    r_modified = min(

        r + delta,

        4.0

    )


    chaos_transient = int(
        CONFIG.get(
            "CHAOS_TRANSIENT",
            1000
        )
    )


    seq1 = generate_logistic_sequence(

        x0=
            x0,

        r=
            r,

        sequence_length=
            N,

        transient=
            chaos_transient

    )


    seq2 = generate_logistic_sequence(

        x0=
            x0,

        r=
            r_modified,

        sequence_length=
            N,

        transient=
            chaos_transient

    )


    seq1 = np.asarray(
        seq1,
        dtype=np.float64
    )

    seq2 = np.asarray(
        seq2,
        dtype=np.float64
    )


    min_length = min(
        len(seq1),
        len(seq2)
    )


    seq1 = seq1[
        :min_length
    ]

    seq2 = seq2[
        :min_length
    ]


    different_values = int(
        np.count_nonzero(
            seq1 != seq2
        )
    )


    return {

        "Original_r":
            float(r),

        "Modified_r":
            float(r_modified),

        "Delta":
            float(
                r_modified - r
            ),

        "Mean_Absolute_Difference":
            float(
                np.mean(
                    np.abs(
                        seq1 - seq2
                    )
                )
            ),

        "Different_Values":
            different_values,

        "Different_Value_Percent":
            float(
                100.0
                *
                different_values
                /
                max(
                    len(seq1),
                    1
                )
            ),

        "Total_Values":
            int(
                len(seq1)
            )

    }


r_sensitivity = (
    chaotic_parameter_sensitivity_r(

        x0=
            chaos_result[
                "Initial_x0"
            ],

        r=
            chaos_result[
                "Control_r"
            ]

    )
)


print()

print(
    "r sensitivity:"
)

for key, value in r_sensitivity.items():

    print(
        f"    {key}: {value}"
    )


# ============================================================
# SAVE RESULT
# ============================================================

if (

    "RESULT_ROOT" in globals()

    and

    RESULT_ROOT is not None

):

    chaos_output_root = Path(
        RESULT_ROOT
    )

else:

    chaos_output_root = Path(

        "/content/drive/MyDrive/"
        "DNA Adaptive RESULT"

    )


chaos_output_dir = (
    chaos_output_root
    /
    "03_CHAOTIC"
)


chaos_output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# COMBINE RESULTS
# ============================================================

chaos_analysis_record = {

    **chaos_result,

    "Content_Mean":
        float(
            feature_values["Mean"]
        ),

    "Content_Std":
        float(
            feature_values["Std"]
        ),

    "Content_Entropy":
        float(
            feature_values["Entropy"]
        ),

    "Content_Edge_Density":
        float(
            feature_values["Edge_Density"]
        ),

    "Feature_Descriptor_Bytes":
        int(
            len(
                descriptor
            )
        ),

    "Feature_Digest":
        descriptor_digest.hex(),

    "x0_Sensitivity_Mean_Absolute_Difference":
        x0_sensitivity[
            "Mean_Absolute_Difference"
        ],

    "x0_Sensitivity_Different_Values":
        x0_sensitivity[
            "Different_Values"
        ],

    "x0_Sensitivity_Different_Value_Percent":
        x0_sensitivity[
            "Different_Value_Percent"
        ],

    "r_Sensitivity_Mean_Absolute_Difference":
        r_sensitivity[
            "Mean_Absolute_Difference"
        ],

    "r_Sensitivity_Different_Values":
        r_sensitivity[
            "Different_Values"
        ],

    "r_Sensitivity_Different_Value_Percent":
        r_sensitivity[
            "Different_Value_Percent"
        ]

}


chaos_analysis_df = pd.DataFrame(
    [chaos_analysis_record]
)


chaos_analysis_path = (
    chaos_output_dir
    /
    "chaotic_sequence_analysis.csv"
)


chaos_analysis_df.to_csv(
    chaos_analysis_path,
    index=False
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print()

print(
    "Saved:"
)

print(
    chaos_analysis_path
)

print()

print(
    "CELL 47 COMPLETE"
)

CELL 47 — V2 CHAOTIC-SEQUENCE ANALYSIS

Architecture: PBKDF2 + HKDF + Content Descriptor + Ciphertext-Compatible State

Test secret: local CELL 47 test key

Content descriptor:
    Mean: 89.38372802734375
    Std: 41.43975870091069
    Entropy: 6.983830134512038
    Edge_Density: 0.100799560546875

Chaotic sequence statistics:
    Initial_x0: 0.005112554876806085
    Control_r: 3.915954947546347
    u: 21958255
    v: 692181595
    Samples: 100000
    Mean: 0.5922039199300969
    Variance: 0.09027198697836124
    Std: 0.3004529696614118
    Minimum: 0.08055037195996205
    Maximum: 0.9789887365424729
    Unique_Values: 100000
    Duplicate_Count: 0
    Lag_1: -0.43450105530709043
    Lag_2: -0.06045809940732782
    Lag_5: 0.25384510833495444
    Lag_10: 0.05543433155728202

x0 sensitivity:
    Original_x0: 0.005112554876806085
    Modified_x0: 0.005112554876816085
    Delta: 9.999813477268304e-15
    Mean_Absolute_Difference: 0.3436719731988304
    Different_Values: 10000
    Different

In [ ]:
# ============================================================
# CELL 48 — COMPLETE PER-IMAGE METRICS — DNA-ADAPTIVE-V2
# ============================================================

"""
PURPOSE:
    Process ALL VALIDATED dataset images using the current
    DNA-ADAPTIVE-V2 architecture.

CURRENT V2 ARCHITECTURE:

    Plain Image
        |
        v
    Content Descriptor
        |
        v
    PBKDF2-HMAC-SHA256
        |
        v
    HKDF / Domain-Separated Key Schedule
        |
        +---- K_DNA
        +---- K_PERM
        +---- K_CHAOS
        +---- K_DIFF
        +---- K_AUTH
        |
        v
    Adaptive DNA Encryption
        |
        v
    Forward Feedback Diffusion
        |
        v
    Backward Ciphertext Diffusion
        |
        v
    Authenticated Cipher Image


    Cipher Image + Authenticated Metadata + Secret Key
        |
        v
    HMAC Authentication
        |
        v
    Reverse Backward Diffusion
        |
        v
    Ciphertext-Only Decryption
        |
        v
    Original Image


IMPORTANT:
    This cell DOES NOT use:

        USER_SECRET
        decrypt_image_with_known_plaintext()

    The current V2 API receives the secret key explicitly.

    This cell therefore uses a local experiment key:

        _TEST_SECRET_KEY_CELL48

    This avoids dependence on the legacy USER_SECRET
    global variable.


NPCR/UACI EXPERIMENT:

    Original plaintext
            |
            | same secret
            | same salt
            | same nonce
            v
        Cipher 1

    One-pixel modified plaintext
            |
            | same secret
            | same salt
            | same nonce
            v
        Cipher 2

    NPCR/UACI are calculated directly from the two
    final authenticated ciphertext images.

    In normal production encryption:
        salt and nonce must be freshly generated.


RECONSTRUCTION TEST:

    Ciphertext
        +
    Authenticated metadata
        +
    Secret key
        |
        v
    decrypt_image()
        |
        v
    Reconstructed plaintext


OUTPUT:

    /08_METRICS/proposed_method_metrics.csv
"""


# ============================================================
# IMPORTS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import time
import secrets


# ============================================================
# LOCAL V2 EXPERIMENT SECRET
# ============================================================
#
# IMPORTANT:
# Do NOT depend on the old USER_SECRET global.
#
# The current V2 encrypt_image() / decrypt_image()
# functions receive secret_key explicitly.
#
# This is the same controlled test key used in the
# other corrected V2 experimental cells.
# ============================================================

_TEST_SECRET_KEY_CELL48 = (
    "DNA-Adaptive-V2-Test-Key-2026"
)


# ============================================================
# RESULT ROOT
# ============================================================

if (
    "RESULT_ROOT" in globals()
    and
    RESULT_ROOT is not None
):

    RESULT_ROOT_LOCAL = Path(
        RESULT_ROOT
    )

else:

    RESULT_ROOT_LOCAL = Path(
        "/content/drive/MyDrive/"
        "DNA Adaptive RESULT"
    )


RESULT_ROOT_LOCAL.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# METRICS DIRECTORY
# ============================================================

METRICS_ROOT = (
    RESULT_ROOT_LOCAL
    /
    "08_METRICS"
)


METRICS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


METRICS_CSV_PATH = (
    METRICS_ROOT
    /
    "proposed_method_metrics.csv"
)


# ============================================================
# REQUIRED DATASET CHECK
# ============================================================

if (
    "valid_dataset_files" not in globals()
    or
    valid_dataset_files is None
):

    raise NameError(
        "\nCELL 48 requires valid_dataset_files.\n\n"
        "valid_dataset_files is created by CELL 7.\n"
        "Run CELL 7 — DATASET VALIDATION first."
    )


valid_dataset_files = list(
    valid_dataset_files
)


if len(
    valid_dataset_files
) == 0:

    raise RuntimeError(
        "\nCELL 48 cannot run.\n\n"
        "CELL 7 found zero valid dataset images."
    )


# ============================================================
# REQUIRED V2 FUNCTIONS
# ============================================================

required_functions = [

    "encrypt_image",

    "decrypt_image",

    "calculate_entropy",

    "calculate_directional_correlations",

    "calculate_mse",

    "calculate_psnr"

]


missing_functions = [

    name

    for name in required_functions

    if name not in globals()

]


if missing_functions:

    raise NameError(

        "CELL 48 is missing required V2 functions:\n"

        +

        "\n".join(

            f"    {name}"

            for name in missing_functions

        )

        +

        "\n\n"

        "Required execution order:\n"

        "    Updated V2 architecture cells\n"

        "    CELL 45B — METRIC HELPERS\n"

        "    CELL 48 — COMPLETE PER-IMAGE METRICS"

    )


# ============================================================
# REQUIRED CONFIGURATION CHECK
# ============================================================

if "CONFIG" not in globals():

    raise NameError(
        "CELL 48 requires the global CONFIG dictionary."
    )


required_config_keys = [

    "BLOCK_SIZE",

    "SALT_BYTES",

    "NONCE_BYTES",

    "KDF_ITERATIONS"

]


missing_config_keys = [

    key

    for key in required_config_keys

    if key not in CONFIG

]


if missing_config_keys:

    raise KeyError(

        "CELL 48 is missing CONFIG entries:\n"

        +

        "\n".join(

            f"    CONFIG['{key}']"

            for key in missing_config_keys

        )

    )


# ============================================================
# IMAGE PATH RESOLVER
# ============================================================

def resolve_dataset_image_path(
    item
):

    if isinstance(
        item,
        dict
    ):

        image_path = (

            item.get("path")

            or

            item.get("image_path")

        )

    else:

        image_path = item


    if image_path is None:

        return None


    return Path(
        image_path
    )


# ============================================================
# EXTRACT METADATA FROM CURRENT V2 ENCRYPTION RESULT
# ============================================================

def extract_encryption_metadata(
    encryption_result
):

    if not isinstance(
        encryption_result,
        tuple
    ):

        raise RuntimeError(
            "encrypt_image() must return a tuple."
        )


    if len(
        encryption_result
    ) < 2:

        raise RuntimeError(
            "encrypt_image() must return ciphertext "
            "and encryption state."
        )


    # --------------------------------------------------------
    # CURRENT V2:
    #
    # encryption_result[0] = ciphertext
    # encryption_result[1] = encryption_state
    # --------------------------------------------------------

    cipher = np.asarray(
        encryption_result[0],
        dtype=np.uint8
    )


    encryption_state = (
        encryption_result[1]
    )


    if not isinstance(
        encryption_state,
        dict
    ):

        raise RuntimeError(
            "encrypt_image()[1] must be a dictionary "
            "containing V2 encryption metadata/state."
        )


    # --------------------------------------------------------
    # Current V2 normally stores:
    #
    # encryption_state["metadata"]
    # --------------------------------------------------------

    if "metadata" in encryption_state:

        metadata = dict(
            encryption_state["metadata"]
        )

    else:

        metadata = dict(
            encryption_state
        )


    # --------------------------------------------------------
    # CURRENT V2 METADATA NAMES
    # --------------------------------------------------------

    required_metadata = [

        "version",

        "height",

        "width",

        "block_size",

        "salt",

        "nonce",

        "content_descriptor",

        "content_digest",

        "kdf_iterations",

        "authentication_tag"

    ]


    missing_metadata = [

        key

        for key in required_metadata

        if key not in metadata

    ]


    if missing_metadata:

        raise RuntimeError(

            "Current V2 encryption metadata is missing:\n"

            +

            "\n".join(

                f"    {key}"

                for key in missing_metadata

            )

            +

            "\n\n"

            "Available metadata keys:\n"

            +

            "\n".join(

                f"    {key}"

                for key in sorted(metadata.keys())

            )

        )


    return (
        cipher,
        metadata
    )


# ============================================================
# DIRECT NPCR / UACI CALCULATION
# ============================================================
#
# IMPORTANT:
# The current notebook has had different helper return
# structures for calculate_npcr().
#
# CELL 48 therefore calculates NPCR and UACI directly.
#
# This prevents CELL 48 from depending on the legacy helper
# return signature.
# ============================================================

def calculate_direct_npcr_uaci(
    cipher_a,
    cipher_b
):

    cipher_a = np.asarray(
        cipher_a,
        dtype=np.uint8
    )

    cipher_b = np.asarray(
        cipher_b,
        dtype=np.uint8
    )


    if cipher_a.shape != cipher_b.shape:

        raise ValueError(

            "NPCR/UACI images must have identical shapes: "

            f"{cipher_a.shape} != {cipher_b.shape}"

        )


    # --------------------------------------------------------
    # Pixel-change mask
    # --------------------------------------------------------

    diff_mask = (
        cipher_a != cipher_b
    )


    changed_pixels = int(
        np.count_nonzero(
            diff_mask
        )
    )


    total_pixels = int(
        diff_mask.size
    )


    if total_pixels == 0:

        raise ValueError(
            "Cannot calculate NPCR/UACI for an empty image."
        )


    # --------------------------------------------------------
    # NPCR
    # --------------------------------------------------------

    npcr_percent = (

        100.0

        *

        changed_pixels

        /

        total_pixels

    )


    # --------------------------------------------------------
    # UACI
    # --------------------------------------------------------

    absolute_difference = np.abs(

        cipher_a.astype(
            np.int16
        )

        -

        cipher_b.astype(
            np.int16
        )

    )


    uaci_percent = float(

        np.mean(
            absolute_difference
        )

        /

        255.0

        *

        100.0

    )


    return (

        float(npcr_percent),

        float(uaci_percent),

        changed_pixels,

        total_pixels

    )


# ============================================================
# CALCULATE IMAGE METRICS
# ============================================================

def calculate_image_metrics(

    plaintext,

    cipher,

    decrypted,

    modified_cipher,

    encryption_time,

    decryption_time

):

    plaintext = np.asarray(
        plaintext,
        dtype=np.uint8
    )


    cipher = np.asarray(
        cipher,
        dtype=np.uint8
    )


    decrypted = np.asarray(
        decrypted,
        dtype=np.uint8
    )


    modified_cipher = np.asarray(
        modified_cipher,
        dtype=np.uint8
    )


    # ========================================================
    # SHAPE VALIDATION
    # ========================================================

    if plaintext.shape != cipher.shape:

        raise ValueError(

            "Plaintext/cipher shape mismatch: "

            f"{plaintext.shape} != {cipher.shape}"

        )


    if plaintext.shape != decrypted.shape:

        raise ValueError(

            "Plaintext/decrypted shape mismatch: "

            f"{plaintext.shape} != {decrypted.shape}"

        )


    if cipher.shape != modified_cipher.shape:

        raise ValueError(

            "Cipher/modified-cipher shape mismatch: "

            f"{cipher.shape} != {modified_cipher.shape}"

        )


    # ========================================================
    # ENTROPY
    # ========================================================

    entropy_original = calculate_entropy(
        plaintext
    )


    entropy_cipher = calculate_entropy(
        cipher
    )


    entropy_decrypted = calculate_entropy(
        decrypted
    )


    # ========================================================
    # CORRELATION
    # ========================================================

    correlations = (
        calculate_directional_correlations(
            cipher
        )
    )


    # ========================================================
    # NPCR + UACI
    # ========================================================

    (
        npcr,
        uaci,
        changed_cipher_pixels,
        total_cipher_pixels

    ) = calculate_direct_npcr_uaci(

        cipher_a=
            cipher,

        cipher_b=
            modified_cipher

    )


    changed_cipher_percent = (

        100.0

        *

        changed_cipher_pixels

        /

        total_cipher_pixels

    )


    # ========================================================
    # RECONSTRUCTION
    # ========================================================

    mse = calculate_mse(

        plaintext,

        decrypted

    )


    psnr = calculate_psnr(

        plaintext,

        decrypted

    )


    different_pixels = int(

        np.count_nonzero(

            plaintext != decrypted

        )

    )


    exact_reconstruction = bool(

        np.array_equal(

            plaintext,

            decrypted

        )

    )


    # ========================================================
    # RETURN
    # ========================================================

    return {

        "Entropy_Original":
            float(
                entropy_original
            ),

        "Entropy_Cipher":
            float(
                entropy_cipher
            ),

        "Entropy_Decrypted":
            float(
                entropy_decrypted
            ),

        "Correlation_H":
            float(
                correlations["Horizontal"]
            ),

        "Correlation_V":
            float(
                correlations["Vertical"]
            ),

        "Correlation_D":
            float(
                correlations["Diagonal"]
            ),

        "NPCR_Percent":
            float(
                npcr
            ),

        "UACI_Percent":
            float(
                uaci
            ),

        "Changed_Cipher_Pixels":
            int(
                changed_cipher_pixels
            ),

        "Changed_Cipher_Percent":
            float(
                changed_cipher_percent
            ),

        "Total_Cipher_Pixels":
            int(
                total_cipher_pixels
            ),

        "MSE":
            float(
                mse
            ),

        "PSNR_dB":
            float(
                psnr
            ),

        "Encryption_Time_Seconds":
            float(
                encryption_time
            ),

        "Decryption_Time_Seconds":
            float(
                decryption_time
            ),

        "Different_Pixels":
            int(
                different_pixels
            ),

        "Exact_Reconstruction":
            exact_reconstruction

    }


# ============================================================
# PROCESS ALL DATASET IMAGES
# ============================================================

def process_all_images_metrics(

    image_files,

    secret_key,

    output_csv_path

):

    image_files = list(
        image_files
    )


    if len(
        image_files
    ) == 0:

        raise ValueError(
            "No validated images were supplied."
        )


    records = []


    print("=" * 80)

    print(
        "CELL 48 — PROCESSING ALL VALIDATED DATASET IMAGES"
    )

    print("=" * 80)


    print(
        f"Total valid images: "
        f"{len(image_files)}"
    )

    print()


    # ========================================================
    # IMAGE LOOP
    # ========================================================

    for image_number, item in enumerate(

        image_files,

        start=1

    ):

        image_path = (
            resolve_dataset_image_path(
                item
            )
        )


        if image_path is None:

            print(

                f"WARNING [{image_number}]: "
                "No image path."

            )

            continue


        # ====================================================
        # LOAD GRAYSCALE IMAGE
        # ====================================================

        image = cv2.imread(

            str(image_path),

            cv2.IMREAD_GRAYSCALE

        )


        if image is None:

            print(

                f"WARNING [{image_number}]: "
                f"Could not read {image_path}"

            )

            continue


        image = np.asarray(

            image,

            dtype=np.uint8

        )


        # ====================================================
        # VALIDATE IMAGE
        # ====================================================

        if image.ndim != 2:

            print(

                f"WARNING [{image_number}]: "
                f"{image_path.name} is not 2-D grayscale."

            )

            continue


        H, W = image.shape


        B = int(
            CONFIG["BLOCK_SIZE"]
        )


        if (

            H % B != 0

            or

            W % B != 0

        ):

            print(

                f"WARNING [{image_number}]: "

                f"{image_path.name} "

                f"{H}x{W} incompatible with "

                f"block size {B}."

            )

            continue


        # ====================================================
        # GENERATE CONTROLLED SALT / NONCE
        # ====================================================
        #
        # IMPORTANT:
        #
        # Same salt + same nonce are intentionally used
        # ONLY for the controlled differential experiment.
        #
        # This isolates the plaintext modification.
        #
        # Production encryption must generate fresh values.
        # ====================================================

        differential_salt = secrets.token_bytes(

            int(
                CONFIG["SALT_BYTES"]
            )

        )


        differential_nonce = secrets.token_bytes(

            int(
                CONFIG["NONCE_BYTES"]
            )

        )


        # ====================================================
        # ENCRYPT ORIGINAL IMAGE
        # ====================================================

        start_enc = time.perf_counter()


        encryption_result = encrypt_image(

            image=
                image,

            secret_key=
                secret_key,

            save_internal=
                False,

            image_name=
                image_path.stem,

            salt=
                differential_salt,

            nonce=
                differential_nonce

        )


        encryption_time = (

            time.perf_counter()

            -

            start_enc

        )


        (
            cipher,
            metadata
        ) = extract_encryption_metadata(

            encryption_result

        )


        # ====================================================
        # VERIFY CIPHER SHAPE
        # ====================================================

        if cipher.shape != image.shape:

            raise RuntimeError(

                f"Cipher image shape mismatch for "
                f"{image_path.name}: "

                f"plaintext={image.shape}, "

                f"cipher={cipher.shape}"

            )


        # ====================================================
        # ONE-PIXEL PLAINTEXT MODIFICATION
        # ====================================================

        modified = image.copy()


        row = H // 2

        col = W // 2


        original_value = int(

            modified[
                row,
                col
            ]

        )


        modified[
            row,
            col
        ] = np.uint8(

            (
                original_value + 1
            )
            %
            256

        )


        # ====================================================
        # VERIFY EXACTLY ONE PLAINTEXT PIXEL CHANGED
        # ====================================================

        plaintext_changed_pixels = int(

            np.count_nonzero(

                image != modified

            )

        )


        if plaintext_changed_pixels != 1:

            raise RuntimeError(

                f"Controlled differential experiment failed "
                f"for {image_path.name}: "

                f"expected 1 changed plaintext pixel, "

                f"found {plaintext_changed_pixels}."

            )


        # ====================================================
        # ENCRYPT MODIFIED IMAGE
        # ====================================================
        #
        # SAME secret
        # SAME salt
        # SAME nonce
        #
        # Only plaintext differs.
        # ====================================================

        modified_result = encrypt_image(

            image=
                modified,

            secret_key=
                secret_key,

            save_internal=
                False,

            image_name=
                image_path.stem
                +
                "_NPCR",

            salt=
                differential_salt,

            nonce=
                differential_nonce

        )


        (
            modified_cipher,
            modified_metadata
        ) = extract_encryption_metadata(

            modified_result

        )


        # ====================================================
        # VERIFY CONTROLLED EXPERIMENT METADATA
        # ====================================================

        if (

            metadata["salt"]

            !=

            modified_metadata["salt"]

        ):

            raise RuntimeError(

                "NPCR experiment failed: "
                "salt differs between original "
                "and modified image."

            )


        if (

            metadata["nonce"]

            !=

            modified_metadata["nonce"]

        ):

            raise RuntimeError(

                "NPCR experiment failed: "
                "nonce differs between original "
                "and modified image."

            )


        # ====================================================
        # VERIFY DIMENSIONS / BLOCK SIZE
        # ====================================================

        if (

            int(metadata["height"])

            !=

            int(modified_metadata["height"])

        ):

            raise RuntimeError(
                "NPCR experiment failed: "
                "metadata height differs."
            )


        if (

            int(metadata["width"])

            !=

            int(modified_metadata["width"])

        ):

            raise RuntimeError(
                "NPCR experiment failed: "
                "metadata width differs."
            )


        if (

            int(metadata["block_size"])

            !=

            int(modified_metadata["block_size"])

        ):

            raise RuntimeError(
                "NPCR experiment failed: "
                "metadata block size differs."
            )


        # ====================================================
        # DECRYPT ORIGINAL IMAGE
        # ====================================================

        start_dec = time.perf_counter()


        decryption_result = decrypt_image(

            cipher_image=
                cipher,

            metadata=
                metadata,

            secret_key=
                secret_key,

            save_internal=
                False,

            image_name=
                image_path.stem

        )


        decryption_time = (

            time.perf_counter()

            -

            start_dec

        )


        # ====================================================
        # VALIDATE DECRYPTION RETURN
        # ====================================================

        if not isinstance(

            decryption_result,

            tuple

        ):

            raise RuntimeError(

                "decrypt_image() did not return a tuple."

            )


        if len(
            decryption_result
        ) == 0:

            raise RuntimeError(

                "decrypt_image() returned no image."

            )


        decrypted = np.asarray(

            decryption_result[0],

            dtype=np.uint8

        )


        # ====================================================
        # CHECK DECRYPTED SHAPE
        # ====================================================

        if decrypted.shape != image.shape:

            raise RuntimeError(

                f"Decrypted image shape mismatch: "

                f"original={image.shape}, "

                f"decrypted={decrypted.shape}"

            )


        # ====================================================
        # CALCULATE METRICS
        # ====================================================

        metrics = calculate_image_metrics(

            plaintext=
                image,

            cipher=
                cipher,

            decrypted=
                decrypted,

            modified_cipher=
                modified_cipher,

            encryption_time=
                encryption_time,

            decryption_time=
                decryption_time

        )


        # ====================================================
        # METADATA BYTE COUNTS
        # ====================================================

        salt_bytes = len(

            bytes.fromhex(
                metadata["salt"]
            )

        )


        nonce_bytes = len(

            bytes.fromhex(
                metadata["nonce"]
            )

        )


        descriptor_bytes = len(

            bytes.fromhex(
                metadata["content_descriptor"]
            )

        )


        # ====================================================
        # CREATE RECORD
        # ====================================================

        record = {

            "Image":
                image_path.name,

            "Height":
                int(H),

            "Width":
                int(W),

            "Block_Size":
                int(B),

            "Method_Version":
                metadata.get(
                    "version",
                    "DNA-ADAPTIVE-V2"
                ),

            "KDF":
                "PBKDF2-HMAC-SHA256 + HKDF-SHA256",

            "KDF_Iterations":
                int(
                    metadata[
                        "kdf_iterations"
                    ]
                ),

            "Salt_Bytes":
                int(
                    salt_bytes
                ),

            "Nonce_Bytes":
                int(
                    nonce_bytes
                ),

            "Feature_Descriptor_Bytes":
                int(
                    descriptor_bytes
                ),

            "Authentication":
                bool(
                    "authentication_tag"
                    in metadata
                ),

            "Controlled_NPCR_Nonce":
                True,

            "Plaintext_Changed_Pixels":
                int(
                    plaintext_changed_pixels
                ),

            "Differential_Salt_Shared":
                bool(
                    metadata["salt"]
                    ==
                    modified_metadata["salt"]
                ),

            "Differential_Nonce_Shared":
                bool(
                    metadata["nonce"]
                    ==
                    modified_metadata["nonce"]
                ),

            "Content_Digest_Changed":
                bool(
                    metadata["content_digest"]
                    !=
                    modified_metadata["content_digest"]
                ),

            **metrics

        }


        records.append(
            record
        )


        # ====================================================
        # PROGRESS
        # ====================================================

        print(

            f"[{image_number:03d}/"
            f"{len(image_files):03d}] "

            f"{image_path.name} | "

            f"Entropy="
            f"{metrics['Entropy_Cipher']:.6f} | "

            f"NPCR="
            f"{metrics['NPCR_Percent']:.6f}% | "

            f"UACI="
            f"{metrics['UACI_Percent']:.6f}% | "

            f"Changed="
            f"{metrics['Changed_Cipher_Pixels']}/"
            f"{metrics['Total_Cipher_Pixels']} | "

            f"MSE="
            f"{metrics['MSE']:.12f} | "

            f"Exact="
            f"{metrics['Exact_Reconstruction']}"

        )


    # ========================================================
    # CREATE DATAFRAME
    # ========================================================

    results = pd.DataFrame(
        records
    )


    # ========================================================
    # FIXED COLUMN ORDER
    # ========================================================

    metric_columns = [

        "Image",

        "Height",

        "Width",

        "Block_Size",

        "Method_Version",

        "KDF",

        "KDF_Iterations",

        "Salt_Bytes",

        "Nonce_Bytes",

        "Feature_Descriptor_Bytes",

        "Authentication",

        "Controlled_NPCR_Nonce",

        "Plaintext_Changed_Pixels",

        "Differential_Salt_Shared",

        "Differential_Nonce_Shared",

        "Content_Digest_Changed",

        "Entropy_Original",

        "Entropy_Cipher",

        "Entropy_Decrypted",

        "Correlation_H",

        "Correlation_V",

        "Correlation_D",

        "NPCR_Percent",

        "UACI_Percent",

        "Changed_Cipher_Pixels",

        "Changed_Cipher_Percent",

        "Total_Cipher_Pixels",

        "MSE",

        "PSNR_dB",

        "Encryption_Time_Seconds",

        "Decryption_Time_Seconds",

        "Different_Pixels",

        "Exact_Reconstruction"

    ]


    if not results.empty:

        for column in metric_columns:

            if column not in results.columns:

                results[column] = np.nan


        results = results[
            metric_columns
        ]


    else:

        results = pd.DataFrame(

            columns=
                metric_columns

        )


    # ========================================================
    # SAVE CSV
    # ========================================================

    output_csv_path = Path(
        output_csv_path
    )


    output_csv_path.parent.mkdir(

        parents=True,

        exist_ok=True

    )


    results.to_csv(

        output_csv_path,

        index=False

    )


    # ========================================================
    # SUMMARY
    # ========================================================

    print()

    print("=" * 80)

    print(
        "V2 METRICS SUMMARY"
    )

    print("=" * 80)


    if not results.empty:

        print()

        print(

            f"Images processed: "
            f"{len(results)}"

        )


        print(

            f"Exact reconstruction count: "
            f"{int(results['Exact_Reconstruction'].sum())}"
            f"/{len(results)}"

        )


        print(

            f"Mean cipher entropy: "
            f"{results['Entropy_Cipher'].mean():.6f}"

        )


        print(

            f"Mean NPCR: "
            f"{results['NPCR_Percent'].mean():.6f}%"

        )


        print(

            f"Mean UACI: "
            f"{results['UACI_Percent'].mean():.6f}%"

        )


        print(

            f"Mean changed ciphertext pixels: "
            f"{results['Changed_Cipher_Pixels'].mean():.2f}"

        )


        print(

            f"Mean changed ciphertext percentage: "
            f"{results['Changed_Cipher_Percent'].mean():.6f}%"

        )


        print(

            f"Mean MSE: "
            f"{results['MSE'].mean():.12f}"

        )


        print(

            f"Mean PSNR: "
            f"{results['PSNR_dB'].mean():.6f} dB"

        )


        print(

            f"Mean encryption time: "
            f"{results['Encryption_Time_Seconds'].mean():.6f} s"

        )


        print(

            f"Mean decryption time: "
            f"{results['Decryption_Time_Seconds'].mean():.6f} s"

        )


        print()

        print(
            "Per-image results:"
        )

        print()


        print(
            results.to_string(
                index=False
            )
        )


    else:

        print(
            "WARNING: No image metrics generated."
        )


    # ========================================================
    # SAVE MESSAGE
    # ========================================================

    print()

    print(
        "Metrics CSV saved:"
    )

    print(
        output_csv_path
    )


    return results


# ============================================================
# RUN CELL 48
# ============================================================

all_image_metrics = (

    process_all_images_metrics(

        image_files=
            valid_dataset_files,

        secret_key=
            _TEST_SECRET_KEY_CELL48,

        output_csv_path=
            METRICS_CSV_PATH

    )

)


# ============================================================
# FINAL DISPLAY
# ============================================================

print()

print("=" * 80)

print(
    "CELL 48 — COMPLETE"
)

print("=" * 80)


print(

    f"Validated images available : "
    f"{len(valid_dataset_files)}"

)


print(

    f"Images successfully processed: "
    f"{len(all_image_metrics)}"

)


print(

    f"Columns: "
    f"{len(all_image_metrics.columns)}"

)


print()

print(
    "Decryption mode:"
)

print(
    "STANDARD CIPHERTEXT + METADATA + SECRET-KEY DECRYPTION"
)


print()

print(
    "NPCR/UACI mode:"
)

print(
    "CONTROLLED ONE-PIXEL PLAINTEXT DIFFERENTIAL TEST"
)

print(
    "SAME SECRET + SAME SALT + SAME NONCE"
)


print()

print(
    "Ciphertext diffusion:"
)

print(
    "FORWARD FEEDBACK + BACKWARD CIPHERTEXT DIFFUSION"
)


print()

print(
    "Authentication:"
)

print(
    "HMAC-SHA256 AUTHENTICATED CIPHERTEXT + METADATA"
)


print()

print(
    "Saved to:"
)

print(
    METRICS_CSV_PATH
)


print()

print(
    "CELL 48 COMPLETE"
)

CELL 48 — PROCESSING ALL VALIDATED DATASET IMAGES
Total valid images: 49

Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
[001/049] Copy of Copy of image_0050.png | Entropy=7.997166 | NPCR=99.629211% | UACI=33.340257% | Changed=65293/65536 | MSE=0.000000000000 | Exact=True
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
[002/049] Copy of Copy of image_0051.png | Entropy=7.997641 | NPCR=99.597168% | UACI=33.558194% | Changed=65272/65536 | MSE=0.000000000000 | Exact=True
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
[003/049] Copy of Copy of image_0052.png | Entropy=7.996941 | NPCR=99.630737% | UACI=33.491462% | Changed=65294/65536 | MSE=0.000000000000 | Exact=True
Backward ciphertext diffusion applied.
Backward ciphertext diffusion applied.
[004/049] Copy of Copy of image_0053.png | Entropy=7.997140 | NPCR=99.658203% | UACI=33.579347% | Changed=65312/65536 | MSE=0.000000000000 | Exact=True
Backwa

In [ ]:
# ============================================================
# CELL 49 — SUMMARY STATISTICS
# ============================================================

SUMMARY_METRICS = [
    "Entropy_Original",
    "Entropy_Cipher",
    "Entropy_Decrypted",
    "Correlation_H",
    "Correlation_V",
    "Correlation_D",
    "MSE",
    "PSNR_dB",
    "Encryption_Time_Seconds",
    "Decryption_Time_Seconds"
]


def generate_summary_statistics(
    metrics_dataframe
):
    """
    Calculate:

        Mean
        Median
        Std
        Min
        Max

    for the requested metrics.
    """

    records = []

    for metric in SUMMARY_METRICS:

        if metric not in metrics_dataframe.columns:
            continue

        values = pd.to_numeric(
            metrics_dataframe[metric],
            errors="coerce"
        )

        finite_values = values[
            np.isfinite(values)
        ]

        if len(finite_values) == 0:

            records.append({
                "Metric": metric,
                "Mean": np.nan,
                "Median": np.nan,
                "Std": np.nan,
                "Min": np.nan,
                "Max": np.nan
            })

            continue

        records.append({
            "Metric": metric,

            "Mean": float(
                finite_values.mean()
            ),

            "Median": float(
                finite_values.median()
            ),

            "Std": float(
                finite_values.std(
                    ddof=1
                )
                if len(finite_values) > 1
                else 0.0
            ),

            "Min": float(
                finite_values.min()
            ),

            "Max": float(
                finite_values.max()
            )
        })

    return pd.DataFrame(
        records
    )


summary_statistics = (
    generate_summary_statistics(
        all_image_metrics
    )
)

SUMMARY_CSV_PATH = (
    RESULT_ROOT /
    "11_SUMMARY" /
    "summary_statistics.csv"
)

SUMMARY_CSV_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

summary_statistics.to_csv(
    SUMMARY_CSV_PATH,
    index=False
)

print("=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

print(
    summary_statistics.to_string(
        index=False
    )
)

print()
print(
    f"Saved summary to: "
    f"{SUMMARY_CSV_PATH}"
)

SUMMARY STATISTICS
                 Metric      Mean    Median      Std       Min      Max
       Entropy_Original  7.066817  7.150229 0.565440  5.875633 7.699482
         Entropy_Cipher  7.997175  7.997166 0.000213  7.996655 7.997641
      Entropy_Decrypted  7.066817  7.150229 0.565440  5.875633 7.699482
          Correlation_H -0.000704 -0.001081 0.004270 -0.010097 0.010060
          Correlation_V -0.000786 -0.001256 0.003792 -0.009299 0.009817
          Correlation_D -0.001014 -0.000726 0.003383 -0.009119 0.006324
                    MSE  0.000000  0.000000 0.000000  0.000000 0.000000
                PSNR_dB       NaN       NaN      NaN       NaN      NaN
Encryption_Time_Seconds  0.866953  0.741298 0.232520  0.712966 1.347313
Decryption_Time_Seconds  0.866106  0.746236 0.233630  0.714590 1.379228

Saved summary to: /content/drive/MyDrive/DNA Adaptive RESULT/11_SUMMARY/summary_statistics.csv


In [ ]:
# ============================================================
# CELL 50 — COMPARE WITH FOUR EXISTING PAPERS
# ============================================================

import pandas as pd
from pathlib import Path
import numpy as np

# ------------------------------------------------------------
# Common comparison framework
# ------------------------------------------------------------

COMPARISON_METHODS = [
    "P1 DNA Tree/CSMM",
    "P2 SHA-256/DNA",
    "P3 Hyperchaotic DNA-DCT",
    "P4 4D Chaos/DNA",
    "Proposed Method"
]

# ------------------------------------------------------------
# Paths to existing paper result CSV files
#
# Update these paths only if your existing result folders
# use different filenames.
# ------------------------------------------------------------

COMPARISON_ROOT = (
    RESULT_ROOT.parent /
    "COMPARISON"
)

COMPARISON_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

EXISTING_RESULTS = {
    "P1 DNA Tree/CSMM": (
        COMPARISON_ROOT /
        "P1_metrics.csv"
    ),

    "P2 SHA-256/DNA": (
        COMPARISON_ROOT /
        "P2_metrics.csv"
    ),

    "P3 Hyperchaotic DNA-DCT": (
        COMPARISON_ROOT /
        "P3_metrics.csv"
    ),

    "P4 4D Chaos/DNA": (
        COMPARISON_ROOT /
        "P4_metrics.csv"
    )
}

PROPOSED_RESULTS_PATH = (
    RESULT_ROOT /
    "08_METRICS" /
    "proposed_method_metrics.csv"
)


# ------------------------------------------------------------
# Standardize metric column names
# ------------------------------------------------------------

COLUMN_ALIASES = {
    "Entropy": "Entropy_Cipher",
    "Cipher Entropy": "Entropy_Cipher",
    "Cipher_Entropy": "Entropy_Cipher",

    "NPCR": "NPCR_Percent",
    "NPCR (%)": "NPCR_Percent",
    "NPCR %": "NPCR_Percent",

    "UACI": "UACI_Percent",
    "UACI (%)": "UACI_Percent",
    "UACI %": "UACI_Percent",

    "Corr-H": "Correlation_H",
    "Corr-H (%)": "Correlation_H",
    "Correlation-H": "Correlation_H",

    "Corr-V": "Correlation_V",
    "Corr-V (%)": "Correlation_V",
    "Correlation-V": "Correlation_V",

    "Corr-D": "Correlation_D",
    "Corr-D (%)": "Correlation_D",
    "Correlation-D": "Correlation_D",

    "Enc Time": "Encryption_Time_Seconds",
    "Encryption Time": "Encryption_Time_Seconds",

    "Dec Time": "Decryption_Time_Seconds",
    "Decryption Time": "Decryption_Time_Seconds",

    "Exact": "Exact_Reconstruction",
    "Exact Recovery": "Exact_Reconstruction"
}


def standardize_columns(df):
    df = df.copy()

    rename_map = {}

    for column in df.columns:

        stripped = str(column).strip()

        if stripped in COLUMN_ALIASES:
            rename_map[column] = (
                COLUMN_ALIASES[stripped]
            )

    df = df.rename(
        columns=rename_map
    )

    return df


# ------------------------------------------------------------
# Load proposed results
# ------------------------------------------------------------

comparison_records = []

if PROPOSED_RESULTS_PATH.exists():

    proposed_df = pd.read_csv(
        PROPOSED_RESULTS_PATH
    )

    proposed_df = standardize_columns(
        proposed_df
    )

    proposed_df["Method"] = (
        "Proposed Method"
    )

    comparison_records.append(
        proposed_df
    )

else:

    print(
        "WARNING: Proposed metrics file not found:"
    )
    print(
        PROPOSED_RESULTS_PATH
    )


# ------------------------------------------------------------
# Load existing paper results
# ------------------------------------------------------------

for method_name, csv_path in EXISTING_RESULTS.items():

    if csv_path.exists():

        df = pd.read_csv(
            csv_path
        )

        df = standardize_columns(
            df
        )

        df["Method"] = method_name

        comparison_records.append(
            df
        )

    else:

        print(
            f"INFO: Result file not found for {method_name}: "
            f"{csv_path}"
        )


# ------------------------------------------------------------
# Combine all available results
# ------------------------------------------------------------

if comparison_records:

    comparison_all_images = pd.concat(
        comparison_records,
        ignore_index=True,
        sort=False
    )

else:

    comparison_all_images = pd.DataFrame()


COMPARISON_ALL_PATH = (
    COMPARISON_ROOT /
    "all_methods_image_level_results.csv"
)

comparison_all_images.to_csv(
    COMPARISON_ALL_PATH,
    index=False
)

print("=" * 80)
print("COMMON EXPERIMENTAL COMPARISON")
print("=" * 80)

print(
    f"Methods with available results: "
    f"{comparison_all_images['Method'].nunique()}"
    if not comparison_all_images.empty
    else "No comparison result files available."
)

print(
    f"Saved: {COMPARISON_ALL_PATH}"
)

INFO: Result file not found for P1 DNA Tree/CSMM: /content/drive/MyDrive/COMPARISON/P1_metrics.csv
INFO: Result file not found for P2 SHA-256/DNA: /content/drive/MyDrive/COMPARISON/P2_metrics.csv
INFO: Result file not found for P3 Hyperchaotic DNA-DCT: /content/drive/MyDrive/COMPARISON/P3_metrics.csv
INFO: Result file not found for P4 4D Chaos/DNA: /content/drive/MyDrive/COMPARISON/P4_metrics.csv
COMMON EXPERIMENTAL COMPARISON
Methods with available results: 1
Saved: /content/drive/MyDrive/COMPARISON/all_methods_image_level_results.csv


In [ ]:
# ============================================================
# CELL 51 — FINAL COMPARISON TABLE
# ============================================================

"""
Purpose:
    Create the final descriptive comparison table for all
    implemented encryption methods.

This cell:
    1. Safely defines COMPARISON_ROOT.
    2. Detects available comparison DataFrames.
    3. Combines per-image results.
    4. Computes descriptive mean values.
    5. Reports exact reconstruction as passed/total.
    6. Saves the final CSV.

No ranking or winner selection is performed.
"""

# ============================================================
# IMPORTS / BASIC REQUIREMENTS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# DEFINE COMPARISON ROOT SAFELY
# ============================================================

# If COMPARISON_ROOT was already created by an earlier cell,
# preserve it.

if (
    "COMPARISON_ROOT" in globals()
    and COMPARISON_ROOT is not None
):

    COMPARISON_ROOT = Path(
        COMPARISON_ROOT
    )

else:

    # Try to use an existing project/result root first.

    if (
        "RESULT_ROOT" in globals()
        and RESULT_ROOT is not None
    ):

        RESULT_ROOT_LOCAL = Path(
            RESULT_ROOT
        )

    elif (
        "RESULTS_ROOT" in globals()
        and RESULTS_ROOT is not None
    ):

        RESULT_ROOT_LOCAL = Path(
            RESULTS_ROOT
        )

    elif (
        "PROJECT_RESULT_ROOT" in globals()
        and PROJECT_RESULT_ROOT is not None
    ):

        RESULT_ROOT_LOCAL = Path(
            PROJECT_RESULT_ROOT
        )

    else:

        # Known result folder for the proposed method.
        RESULT_ROOT_LOCAL = Path(
            "/content/drive/MyDrive/"
            "DNA Adaptive RESULT"
        )

    COMPARISON_ROOT = (
        RESULT_ROOT_LOCAL /
        "10_COMPARISON"
    )


# Make sure the folder exists.

COMPARISON_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# REQUIRED COMPARISON METRICS
# ============================================================

COMPARISON_METRIC_COLUMNS = [

    "Entropy_Cipher",

    "NPCR_Percent",

    "UACI_Percent",

    "Correlation_H",

    "Correlation_V",

    "Correlation_D",

    "Encryption_Time_Seconds",

    "Decryption_Time_Seconds",

    "Exact_Reconstruction"
]


# ============================================================
# HELPER — NORMALIZE EXACT RECONSTRUCTION
# ============================================================

def normalize_exact_reconstruction(value):

    if pd.isna(value):
        return np.nan

    # Boolean
    if isinstance(
        value,
        (bool, np.bool_)
    ):

        return bool(value)

    # Numeric
    if isinstance(
        value,
        (int, float, np.integer, np.floating)
    ):

        if np.isnan(value):
            return np.nan

        return bool(value)

    # String
    text = str(
        value
    ).strip().lower()

    if text in (
        "true",
        "1",
        "yes",
        "pass",
        "passed",
        "exact",
        "success"
    ):

        return True

    if text in (
        "false",
        "0",
        "no",
        "fail",
        "failed",
        "not exact"
    ):

        return False

    return np.nan


# ============================================================
# HELPER — SUMMARIZE ONE METHOD
# ============================================================

def summarize_method_results(
    df,
    method_name
):

    row = {
        "Method": method_name
    }

    # --------------------------------------------------------
    # Numeric metrics
    # --------------------------------------------------------

    numeric_metrics = [

        "Entropy_Cipher",

        "NPCR_Percent",

        "UACI_Percent",

        "Correlation_H",

        "Correlation_V",

        "Correlation_D",

        "Encryption_Time_Seconds",

        "Decryption_Time_Seconds"
    ]

    for metric in numeric_metrics:

        if metric not in df.columns:

            row[metric] = np.nan
            continue

        values = pd.to_numeric(
            df[metric],
            errors="coerce"
        )

        values = values[
            np.isfinite(values)
        ]

        if len(values) > 0:

            row[metric] = float(
                values.mean()
            )

        else:

            row[metric] = np.nan

    # --------------------------------------------------------
    # Exact reconstruction
    # --------------------------------------------------------

    if (
        "Exact_Reconstruction"
        not in df.columns
    ):

        row[
            "Exact_Reconstruction"
        ] = np.nan

    else:

        reconstruction_values = (
            df[
                "Exact_Reconstruction"
            ]
            .apply(
                normalize_exact_reconstruction
            )
        )

        valid = (
            reconstruction_values
            .dropna()
        )

        if len(valid) > 0:

            passed = int(
                valid.sum()
            )

            total = int(
                len(valid)
            )

            row[
                "Exact_Reconstruction"
            ] = (
                f"{passed}/{total}"
            )

        else:

            row[
                "Exact_Reconstruction"
            ] = np.nan

    return row


# ============================================================
# FIND EXISTING COMPARISON DATA
# ============================================================

comparison_source = pd.DataFrame()


# ------------------------------------------------------------
# FIRST PRIORITY:
# Existing comparison_all_images
# ------------------------------------------------------------

if (
    "comparison_all_images" in globals()
    and isinstance(
        comparison_all_images,
        pd.DataFrame
    )
    and not comparison_all_images.empty
):

    comparison_source = (
        comparison_all_images.copy()
    )


# ------------------------------------------------------------
# SECOND PRIORITY:
# Existing candidate result tables
# ------------------------------------------------------------

if comparison_source.empty:

    candidate_names = [

        "proposed_all_results",

        "proposed_results",

        "comparison_results",

        "all_method_results",

        "method_results",

        "results_all_images",

        "all_results",

        "final_results",

        "all_comparison_results"
    ]

    candidate_tables = []

    for variable_name in candidate_names:

        if variable_name not in globals():
            continue

        candidate = globals()[
            variable_name
        ]

        if not isinstance(
            candidate,
            pd.DataFrame
        ):

            continue

        if candidate.empty:
            continue

        # Must contain at least one comparison metric.

        has_metric = any(
            metric in candidate.columns
            for metric in
            COMPARISON_METRIC_COLUMNS
        )

        if not has_metric:
            continue

        candidate_tables.append(
            candidate.copy()
        )

    if len(candidate_tables) > 0:

        comparison_source = pd.concat(
            candidate_tables,
            ignore_index=True
        )


# ============================================================
# TRY TO ADD PROPOSED METHOD DATA
# ============================================================

# If a proposed-method DataFrame exists but has no Method
# column, assign the proposed method name here.

if (
    not comparison_source.empty
    and "Method" not in comparison_source.columns
):

    # If there is only one table, it can reasonably be treated
    # as the proposed method result table.

    if len(
        comparison_source
    ) > 0:

        comparison_source[
            "Method"
        ] = "Proposed Adaptive DNA"


# ============================================================
# VALIDATE SOURCE
# ============================================================

if comparison_source.empty:

    print("=" * 80)
    print("FINAL METHOD COMPARISON")
    print("=" * 80)
    print()
    print(
        "No comparison data was found."
    )
    print()
    print(
        "Expected a DataFrame containing:"
    )
    print(
        "    Method"
    )

    for metric in COMPARISON_METRIC_COLUMNS:

        print(
            f"    {metric}"
        )

    print()
    print(
        "Run the previous result-generation/"
        "comparison cells first."
    )

else:

    if "Method" not in comparison_source.columns:

        raise ValueError(
            "Comparison data exists, but the "
            "'Method' column is missing."
        )


# ============================================================
# METHOD LIST
# ============================================================

if (
    "COMPARISON_METHODS" in globals()
    and COMPARISON_METHODS is not None
):

    comparison_methods = list(
        COMPARISON_METHODS
    )

else:

    if (
        not comparison_source.empty
        and "Method"
        in comparison_source.columns
    ):

        comparison_methods = list(
            comparison_source[
                "Method"
            ]
            .dropna()
            .unique()
        )

    else:

        comparison_methods = []


# ============================================================
# BUILD FINAL SUMMARY
# ============================================================

final_comparison_records = []


if not comparison_source.empty:

    for method_name in comparison_methods:

        method_df = (
            comparison_source[
                comparison_source[
                    "Method"
                ] == method_name
            ]
            .copy()
        )

        if method_df.empty:
            continue

        final_comparison_records.append(

            summarize_method_results(
                method_df,
                method_name
            )
        )


# ============================================================
# IF METHOD LIST WAS EMPTY, USE ACTUAL METHODS
# ============================================================

if (
    len(final_comparison_records) == 0
    and not comparison_source.empty
):

    for method_name in (
        comparison_source[
            "Method"
        ]
        .dropna()
        .unique()
    ):

        method_df = (
            comparison_source[
                comparison_source[
                    "Method"
                ] == method_name
            ]
            .copy()
        )

        if method_df.empty:
            continue

        final_comparison_records.append(

            summarize_method_results(
                method_df,
                method_name
            )
        )


# ============================================================
# FINAL DATAFRAME
# ============================================================

final_comparison_table = pd.DataFrame(
    final_comparison_records
)


# ============================================================
# COLUMN ORDER
# ============================================================

final_columns = [

    "Method"

] + COMPARISON_METRIC_COLUMNS


if not final_comparison_table.empty:

    for column in final_columns:

        if column not in (
            final_comparison_table.columns
        ):

            final_comparison_table[
                column
            ] = np.nan

    final_comparison_table = (
        final_comparison_table[
            final_columns
        ]
    )


# ============================================================
# SAVE CSV
# ============================================================

FINAL_COMPARISON_PATH = (
    COMPARISON_ROOT /
    "final_comparison_table.csv"
)

final_comparison_table.to_csv(
    FINAL_COMPARISON_PATH,
    index=False
)


# ============================================================
# DISPLAY
# ============================================================

print("=" * 80)
print("FINAL METHOD COMPARISON")
print("=" * 80)

print()

if not final_comparison_table.empty:

    print(
        final_comparison_table.to_string(
            index=False
        )
    )

    print()

    print(
        f"Methods included: "
        f"{len(final_comparison_table)}"
    )

    print(
        f"Rows in source data: "
        f"{len(comparison_source)}"
    )

else:

    print(
        "No method-level comparison data available."
    )


# ============================================================
# OUTPUT LOCATION
# ============================================================

print()
print(
    "Comparison is descriptive only; "
    "no ranking or winner is assigned."
)

print()
print(
    f"Comparison directory:"
)

print(
    f"    {COMPARISON_ROOT}"
)

print()
print(
    f"Saved:"
)

print(
    f"    {FINAL_COMPARISON_PATH}"
)



FINAL METHOD COMPARISON

         Method  Entropy_Cipher  NPCR_Percent  UACI_Percent  Correlation_H  Correlation_V  Correlation_D  Encryption_Time_Seconds  Decryption_Time_Seconds Exact_Reconstruction
Proposed Method        7.997175     99.613392     33.479082      -0.000704      -0.000786      -0.001014                 0.866953                 0.866106                49/49

Methods included: 1
Rows in source data: 49

Comparison is descriptive only; no ranking or winner is assigned.

Comparison directory:
    /content/drive/MyDrive/COMPARISON

Saved:
    /content/drive/MyDrive/COMPARISON/final_comparison_table.csv


In [ ]:
# ============================================================
# CELL 52 — V2 VISUAL OUTPUTS + ENCRYPTED/DECRYPTED FILES
# ============================================================
#
# This cell:
#
#   1. Loads the selected test images
#   2. Encrypts using the CURRENT V2 encrypt_image()
#   3. Saves the actual ciphertext image
#   4. Saves the ciphertext NumPy array
#   5. Saves the V2 encryption metadata
#   6. Decrypts using the CURRENT ciphertext-only decrypt_image()
#   7. Saves the actual decrypted image
#   8. Saves the decrypted NumPy array
#   9. Creates original/cipher/decrypted comparison figures
#  10. Creates histogram figures
#  11. Creates H/V/D correlation scatter plots
#  12. Saves a complete CSV index
#
# IMPORTANT:
#
#   The current V2 encrypt_image() and decrypt_image()
#   functions DO NOT themselves save PNG files.
#
#   Therefore this cell explicitly writes the images using
#   cv2.imwrite().
#
# ============================================================


from pathlib import Path
import json
import time

import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd


# ============================================================
# 1. RESOLVE THE PROJECT RESULT ROOT
# ============================================================
#
# Use the RESULT_ROOT variable already defined by the project.
# If it does not exist, use the same Drive location used by
# the notebook.
#
# ============================================================

if (
    "RESULT_ROOT" in globals()
    and RESULT_ROOT is not None
):

    RESULT_ROOT_LOCAL = Path(
        RESULT_ROOT
    )

else:

    RESULT_ROOT_LOCAL = Path(
        "/content/drive/MyDrive/"
        "DNA Adaptive RESULT"
    )


RESULT_ROOT_LOCAL.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. PROJECT OUTPUT DIRECTORIES
# ============================================================
#
# These are the directories already used by the project.
#
# ============================================================

ENCRYPTION_DIR = (
    RESULT_ROOT_LOCAL /
    "06_ENCRYPTION"
)

CIPHER_IMAGE_DIR = (
    ENCRYPTION_DIR /
    "cipher_images"
)

ENCRYPTION_ARRAY_DIR = (
    ENCRYPTION_DIR /
    "encryption_arrays"
)

DECRYPTION_DIR = (
    RESULT_ROOT_LOCAL /
    "07_DECRYPTION"
)

DECRYPTED_IMAGE_DIR = (
    DECRYPTION_DIR /
    "decrypted_images"
)

DECRYPTION_ARRAY_DIR = (
    DECRYPTION_DIR /
    "decryption_arrays"
)

SECURITY_DIR = (
    RESULT_ROOT_LOCAL /
    "09_SECURITY"
)

HISTOGRAM_DIR = (
    SECURITY_DIR /
    "histogram"
)

CORRELATION_DIR = (
    SECURITY_DIR /
    "correlation"
)

SUMMARY_DIR = (
    RESULT_ROOT_LOCAL /
    "11_SUMMARY"
)

VISUAL_OUTPUT_DIR = (
    SUMMARY_DIR /
    "visual_outputs"
)


# ============================================================
# 3. CREATE ALL REQUIRED DIRECTORIES
# ============================================================

for directory in [

    CIPHER_IMAGE_DIR,
    ENCRYPTION_ARRAY_DIR,

    DECRYPTED_IMAGE_DIR,
    DECRYPTION_ARRAY_DIR,

    HISTOGRAM_DIR,
    CORRELATION_DIR,

    VISUAL_OUTPUT_DIR,
    SUMMARY_DIR

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 4. DISPLAY EXACT PROJECT DIRECTORIES
# ============================================================

print()
print("=" * 80)
print("CELL 52 — OUTPUT DIRECTORIES")
print("=" * 80)

print(
    f"RESULT ROOT          : {RESULT_ROOT_LOCAL}"
)

print(
    f"CIPHER IMAGES        : {CIPHER_IMAGE_DIR}"
)

print(
    f"ENCRYPTION ARRAYS    : {ENCRYPTION_ARRAY_DIR}"
)

print(
    f"DECRYPTED IMAGES     : {DECRYPTED_IMAGE_DIR}"
)

print(
    f"DECRYPTION ARRAYS    : {DECRYPTION_ARRAY_DIR}"
)

print(
    f"HISTOGRAMS           : {HISTOGRAM_DIR}"
)

print(
    f"CORRELATION PLOTS    : {CORRELATION_DIR}"
)

print(
    f"VISUAL COMPARISONS   : {VISUAL_OUTPUT_DIR}"
)

print()


# ============================================================
# 5. RESOLVE TEST SECRET
# ============================================================
#
# Current V2 cells do not require USER_SECRET to exist.
#
# If USER_SECRET already exists, use it.
#
# Otherwise use the same reproducible local test key used
# in the repaired V2 experiments.
#
# ============================================================

if (
    "USER_SECRET" in globals()
    and USER_SECRET is not None
):

    CELL52_SECRET_KEY = USER_SECRET

    CELL52_SECRET_SOURCE = (
        "USER_SECRET"
    )

else:

    CELL52_SECRET_KEY = (
        "DNA-Adaptive-V2-Test-Key-2026"
    )

    CELL52_SECRET_SOURCE = (
        "CELL52_LOCAL_TEST_KEY"
    )


# ============================================================
# 6. HELPER — SAFE FILE NAME
# ============================================================

def cell52_safe_stem(
    image_name
):

    stem = Path(
        str(image_name)
    ).stem

    # Remove characters that can create problematic
    # filenames in Drive/Colab.
    unsafe = [
        "/",
        "\\",
        ":",
        "*",
        "?",
        "\"",
        "<",
        ">",
        "|"
    ]

    for char in unsafe:

        stem = stem.replace(
            char,
            "_"
        )

    if stem == "":

        stem = "test_image"

    return stem


# ============================================================
# 7. HELPER — CONVERT BYTE-LIKE METADATA TO LENGTH
# ============================================================

def cell52_byte_length(
    value
):

    if value is None:

        return 0

    if isinstance(
        value,
        (
            bytes,
            bytearray,
            memoryview
        )
    ):

        return len(
            value
        )

    if isinstance(
        value,
        str
    ):

        try:

            return len(
                bytes.fromhex(
                    value
                )
            )

        except Exception:

            return len(
                value.encode(
                    "utf-8"
                )
            )

    try:

        return len(
            value
        )

    except Exception:

        return 0


# ============================================================
# 8. HELPER — JSON-SAFE METADATA
# ============================================================
#
# Current V2 metadata is expected to be JSON-compatible, but
# this helper also handles bytes / NumPy values safely.
#
# ============================================================

def cell52_json_safe(
    value
):

    if isinstance(
        value,
        dict
    ):

        return {
            str(key):
                cell52_json_safe(
                    val
                )
            for key, val in value.items()
        }

    if isinstance(
        value,
        (list, tuple)
    ):

        return [
            cell52_json_safe(
                item
            )
            for item in value
        ]

    if isinstance(
        value,
        np.ndarray
    ):

        return [
            cell52_json_safe(
                item
            )
            for item in value.tolist()
        ]

    if isinstance(
        value,
        (
            np.integer,
            np.floating
        )
    ):

        return value.item()

    if isinstance(
        value,
        np.bool_
    ):

        return bool(
            value
        )

    if isinstance(
        value,
        (
            bytes,
            bytearray
        )
    ):

        return bytes(
            value
        ).hex()

    return value


# ============================================================
# 9. SAVE ORIGINAL / CIPHER / DECRYPTED COMPARISON
# ============================================================

def save_visual_comparison(
    original,
    cipher,
    decrypted,
    image_name,
    output_root
):

    output_root = Path(
        output_root
    )

    image_stem = cell52_safe_stem(
        image_name
    )

    visual_dir = (
        output_root /
        "11_SUMMARY" /
        "visual_outputs"
    )

    histogram_dir = (
        output_root /
        "09_SECURITY" /
        "histogram"
    )

    visual_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    histogram_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    original = np.asarray(
        original,
        dtype=np.uint8
    )

    cipher = np.asarray(
        cipher,
        dtype=np.uint8
    )

    decrypted = np.asarray(
        decrypted,
        dtype=np.uint8
    )


    # ========================================================
    # ORIGINAL / CIPHER / DECRYPTED
    # ========================================================

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(15, 5)
    )


    axes[0].imshow(
        original,
        cmap="gray",
        vmin=0,
        vmax=255
    )

    axes[0].set_title(
        "Original"
    )


    axes[1].imshow(
        cipher,
        cmap="gray",
        vmin=0,
        vmax=255
    )

    axes[1].set_title(
        "Cipher"
    )


    axes[2].imshow(
        decrypted,
        cmap="gray",
        vmin=0,
        vmax=255
    )

    axes[2].set_title(
        "Decrypted"
    )


    for ax in axes:

        ax.axis(
            "off"
        )


    fig.tight_layout()


    visual_path = (
        visual_dir /
        f"{image_stem}_comparison.png"
    )


    fig.savefig(
        visual_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    # ========================================================
    # HISTOGRAM
    # ========================================================

    fig = plt.figure(
        figsize=(10, 6)
    )


    plt.hist(
        original.ravel(),
        bins=256,
        range=(0, 256),
        alpha=0.5,
        label="Original"
    )


    plt.hist(
        cipher.ravel(),
        bins=256,
        range=(0, 256),
        alpha=0.5,
        label="Cipher"
    )


    plt.xlabel(
        "Pixel Intensity"
    )

    plt.ylabel(
        "Frequency"
    )

    plt.title(
        f"Original vs Cipher Histogram — "
        f"{image_stem}"
    )


    plt.legend()

    plt.tight_layout()


    histogram_path = (
        histogram_dir /
        f"{image_stem}_histogram.png"
    )


    fig.savefig(
        histogram_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    return {

        "visual":
            visual_path,

        "histogram":
            histogram_path

    }


# ============================================================
# 10. CORRELATION SCATTER PLOTS
# ============================================================

def save_correlation_scatter_plots(
    original,
    cipher,
    image_name,
    output_root
):

    output_root = Path(
        output_root
    )

    correlation_dir = (
        output_root /
        "09_SECURITY" /
        "correlation"
    )

    correlation_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    image_stem = cell52_safe_stem(
        image_name
    )


    original = np.asarray(
        original,
        dtype=np.uint8
    )

    cipher = np.asarray(
        cipher,
        dtype=np.uint8
    )


    # ========================================================
    # HORIZONTAL
    # ========================================================

    fig = plt.figure(
        figsize=(8, 6)
    )


    plt.scatter(
        original[:, :-1].ravel(),
        original[:, 1:].ravel(),
        s=1,
        alpha=0.25,
        label="Original"
    )


    plt.scatter(
        cipher[:, :-1].ravel(),
        cipher[:, 1:].ravel(),
        s=1,
        alpha=0.25,
        label="Cipher"
    )


    plt.xlabel(
        "Pixel Value"
    )

    plt.ylabel(
        "Adjacent Pixel Value"
    )

    plt.title(
        f"Horizontal Correlation — "
        f"{image_stem}"
    )


    plt.legend()

    plt.tight_layout()


    horizontal_path = (
        correlation_dir /
        f"{image_stem}_horizontal.png"
    )


    fig.savefig(
        horizontal_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    # ========================================================
    # VERTICAL
    # ========================================================

    fig = plt.figure(
        figsize=(8, 6)
    )


    plt.scatter(
        original[:-1, :].ravel(),
        original[1:, :].ravel(),
        s=1,
        alpha=0.25,
        label="Original"
    )


    plt.scatter(
        cipher[:-1, :].ravel(),
        cipher[1:, :].ravel(),
        s=1,
        alpha=0.25,
        label="Cipher"
    )


    plt.xlabel(
        "Pixel Value"
    )

    plt.ylabel(
        "Adjacent Pixel Value"
    )

    plt.title(
        f"Vertical Correlation — "
        f"{image_stem}"
    )


    plt.legend()

    plt.tight_layout()


    vertical_path = (
        correlation_dir /
        f"{image_stem}_vertical.png"
    )


    fig.savefig(
        vertical_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    # ========================================================
    # DIAGONAL
    # ========================================================

    fig = plt.figure(
        figsize=(8, 6)
    )


    plt.scatter(
        original[:-1, :-1].ravel(),
        original[1:, 1:].ravel(),
        s=1,
        alpha=0.25,
        label="Original"
    )


    plt.scatter(
        cipher[:-1, :-1].ravel(),
        cipher[1:, 1:].ravel(),
        s=1,
        alpha=0.25,
        label="Cipher"
    )


    plt.xlabel(
        "Pixel Value"
    )

    plt.ylabel(
        "Diagonal Pixel Value"
    )

    plt.title(
        f"Diagonal Correlation — "
        f"{image_stem}"
    )


    plt.legend()

    plt.tight_layout()


    diagonal_path = (
        correlation_dir /
        f"{image_stem}_diagonal.png"
    )


    fig.savefig(
        diagonal_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    return {

        "horizontal":
            horizontal_path,

        "vertical":
            vertical_path,

        "diagonal":
            diagonal_path

    }


# ============================================================
# 11. BUILD IMAGE LIST
# ============================================================

image_items = []


if (
    "dataset_files" in globals()
    and dataset_files is not None
):

    try:

        image_items = list(
            dataset_files
        )

    except Exception:

        image_items = []


if len(
    image_items
) == 0:

    if (
        "valid_dataset_files" in globals()
        and valid_dataset_files is not None
    ):

        try:

            image_items = list(
                valid_dataset_files
            )

        except Exception:

            image_items = []


if len(
    image_items
) == 0:

    if (
        "test_image_path" in globals()
        and test_image_path is not None
    ):

        image_items = [
            test_image_path
        ]

    elif (
        "test_image" in globals()
        and test_image is not None
    ):

        image_items = [
            {
                "image":
                    test_image,

                "name":
                    "test_image"
            }
        ]

    else:

        raise NameError(
            "Neither dataset_files, "
            "valid_dataset_files, nor "
            "test_image/test_image_path "
            "is available."
        )


# ============================================================
# 12. NUMBER OF VISUAL TEST IMAGES
# ============================================================

# ============================================================
# PROCESS ALL DATASET IMAGES
# ============================================================

VISUAL_TEST_COUNT = len(
    image_items
)

print(
    f"Total images available for CELL 52: "
    f"{VISUAL_TEST_COUNT}"
)

# ============================================================
# 13. OUTPUT RECORDS
# ============================================================

visual_output_records = []


# ============================================================
# 14. PROCESS IMAGES
# ============================================================

for item in image_items[
    :VISUAL_TEST_COUNT
]:

    original = None

    image_name = "unknown_image"

    try:

        # ====================================================
        # RESOLVE IMAGE
        # ====================================================

        if isinstance(
            item,
            dict
        ):

            image_path = (
                item.get(
                    "path"
                )
                or
                item.get(
                    "image_path"
                )
            )


            if (
                image_path is None
                and
                item.get(
                    "image"
                ) is not None
            ):

                original = np.asarray(
                    item[
                        "image"
                    ],
                    dtype=np.uint8
                )


                image_name = str(
                    item.get(
                        "name",
                        "test_image"
                    )
                )


            else:

                if image_path is None:

                    print(
                        "[WARNING] "
                        "Image path missing; "
                        "skipping item."
                    )

                    continue


                image_path = Path(
                    image_path
                )


                original = cv2.imread(
                    str(
                        image_path
                    ),
                    cv2.IMREAD_GRAYSCALE
                )


                if original is None:

                    print(
                        f"[WARNING] "
                        f"Could not read "
                        f"{image_path}"
                    )

                    continue


                original = np.asarray(
                    original,
                    dtype=np.uint8
                )


                image_name = (
                    image_path.name
                )


        else:

            image_path = Path(
                item
            )


            original = cv2.imread(
                str(
                    image_path
                ),
                cv2.IMREAD_GRAYSCALE
            )


            if original is None:

                print(
                    f"[WARNING] "
                    f"Could not read "
                    f"{image_path}"
                )

                continue


            original = np.asarray(
                original,
                dtype=np.uint8
            )


            image_name = (
                image_path.name
            )


        # ====================================================
        # SAFE IMAGE STEM
        # ====================================================

        image_stem = cell52_safe_stem(
            image_name
        )


        # ====================================================
        # VALIDATE GRAYSCALE IMAGE
        # ====================================================

        if original.ndim != 2:

            raise ValueError(
                "Image is not a 2-D grayscale image."
            )


        # ====================================================
        # CURRENT BLOCK SIZE
        # ====================================================

        if (
            "CONFIG" not in globals()
            or CONFIG is None
        ):

            raise RuntimeError(
                "CONFIG is not available. "
                "Run the V2 configuration cells first."
            )


        current_block_size = int(
            CONFIG.get(
                "BLOCK_SIZE",
                16
            )
        )


        # ====================================================
        # BLOCK COMPATIBILITY
        # ====================================================

        if (
            original.shape[0]
            %
            current_block_size
            != 0
            or
            original.shape[1]
            %
            current_block_size
            != 0
        ):

            raise ValueError(
                f"Image shape {original.shape} "
                f"is not divisible by "
                f"BLOCK_SIZE={current_block_size}."
            )


        # ====================================================
        # ENCRYPTION START
        # ====================================================

        encryption_start_time = (
            time.perf_counter()
        )


        # ====================================================
        # CURRENT V2 ENCRYPTION
        # ====================================================

        encryption_result = encrypt_image(
            image=original,
            secret_key=CELL52_SECRET_KEY,
            save_internal=False,
            image_name=image_stem
        )


        encryption_elapsed = (
            time.perf_counter()
            -
            encryption_start_time
        )


        # ====================================================
        # VALIDATE ENCRYPTION RESULT
        # ====================================================

        if not isinstance(
            encryption_result,
            tuple
        ):

            raise RuntimeError(
                "encrypt_image() did not return "
                "the expected tuple."
            )


        if len(
            encryption_result
        ) < 2:

            raise RuntimeError(
                "encrypt_image() did not return "
                "ciphertext and encryption state."
            )


        # ====================================================
        # CURRENT V2 CIPHERTEXT
        # ====================================================

        cipher = np.asarray(
            encryption_result[0],
            dtype=np.uint8
        )


        # ====================================================
        # CURRENT V2 ENCRYPTION STATE
        # ====================================================

        encryption_state = (
            encryption_result[1]
        )


        if not isinstance(
            encryption_state,
            dict
        ):

            raise RuntimeError(
                "encrypt_image()[1] is not "
                "the expected encryption-state dictionary."
            )


        # ====================================================
        # CURRENT V2 METADATA
        # ====================================================

        metadata = (
            encryption_state.get(
                "metadata"
            )
        )


        if not isinstance(
            metadata,
            dict
        ):

            raise RuntimeError(
                "Current V2 metadata was not found "
                "inside encryption_result[1]['metadata']."
            )


        # ====================================================
        # REQUIRED V2 METADATA
        # ====================================================

        required_metadata = [

            "version",

            "height",

            "width",

            "block_size",

            "salt",

            "nonce",

            "content_descriptor",

            "content_digest",

            "kdf",

            "kdf_iterations",

            "key_schedule",

            "authentication_tag",

            "reverse_diffusion"

        ]


        missing_metadata = [

            key

            for key in required_metadata

            if key not in metadata

        ]


        if len(
            missing_metadata
        ) > 0:

            raise RuntimeError(
                "Current V2 metadata is missing: "
                +
                ", ".join(
                    missing_metadata
                )
            )


        # ====================================================
        # VALIDATE CIPHERTEXT SHAPE
        # ====================================================

        if cipher.shape != original.shape:

            raise RuntimeError(
                f"Cipher shape {cipher.shape} "
                f"does not match original shape "
                f"{original.shape}."
            )


        # ====================================================
        # SAVE ACTUAL ENCRYPTED PNG
        # ====================================================
        #
        # THIS is the part that was missing from the previous
        # CELL 52.
        #
        # ====================================================

        cipher_png_path = (
            CIPHER_IMAGE_DIR /
            f"{image_stem}_cipher.png"
        )


        cipher_write_ok = cv2.imwrite(
            str(
                cipher_png_path
            ),
            cipher
        )


        if not cipher_write_ok:

            raise IOError(
                "cv2.imwrite() failed for encrypted image: "
                f"{cipher_png_path}"
            )


        # ====================================================
        # SAVE ENCRYPTED NUMPY ARRAY
        # ====================================================

        cipher_npy_path = (
            ENCRYPTION_ARRAY_DIR /
            f"{image_stem}_cipher.npy"
        )


        np.save(
            str(
                cipher_npy_path
            ),
            cipher
        )


        # ====================================================
        # SAVE V2 ENCRYPTION METADATA
        # ====================================================

        metadata_json_path = (
            ENCRYPTION_ARRAY_DIR /
            f"{image_stem}_encryption_metadata.json"
        )


        metadata_json_safe = (
            cell52_json_safe(
                metadata
            )
        )


        with open(
            metadata_json_path,
            "w",
            encoding="utf-8"
        ) as metadata_file:

            json.dump(
                metadata_json_safe,
                metadata_file,
                indent=4
            )


        # ====================================================
        # DECRYPTION START
        # ====================================================

        decryption_start_time = (
            time.perf_counter()
        )


        # ====================================================
        # CURRENT V2 CIPHERTEXT-ONLY DECRYPTION
        # ====================================================
        #
        # IMPORTANT:
        #
        # original plaintext is NOT passed to decrypt_image().
        #
        # ====================================================

        decryption_result = decrypt_image(
            cipher_image=cipher,
            metadata=metadata,
            secret_key=CELL52_SECRET_KEY,
            save_internal=False,
            image_name=image_stem
        )


        decryption_elapsed = (
            time.perf_counter()
            -
            decryption_start_time
        )


        # ====================================================
        # VALIDATE DECRYPTION RESULT
        # ====================================================

        if not isinstance(
            decryption_result,
            tuple
        ):

            raise RuntimeError(
                "decrypt_image() did not return "
                "the expected tuple."
            )


        if len(
            decryption_result
        ) < 1:

            raise RuntimeError(
                "decrypt_image() returned no decrypted image."
            )


        # ====================================================
        # CURRENT V2 DECRYPTED IMAGE
        # ====================================================

        decrypted = np.asarray(
            decryption_result[0],
            dtype=np.uint8
        )


        # ====================================================
        # VALIDATE DECRYPTED SHAPE
        # ====================================================

        if decrypted.shape != original.shape:

            raise RuntimeError(
                f"Decrypted shape {decrypted.shape} "
                f"does not match original shape "
                f"{original.shape}."
            )


        # ====================================================
        # VERIFY EXACT RECONSTRUCTION
        # ====================================================

        exact_recovery = bool(
            np.array_equal(
                original,
                decrypted
            )
        )


        if not exact_recovery:

            raise RuntimeError(
                "Decryption completed, but the decrypted "
                "image is NOT exactly equal to the original."
            )


        # ====================================================
        # AUTHENTICATION STATUS
        # ====================================================
        #
        # Current decrypt_image() verifies the HMAC before
        # performing decryption. Successful return therefore
        # means the authenticated ciphertext passed the
        # current verification path.
        #
        # ====================================================

        authentication_verified = True


        # ====================================================
        # SAVE ACTUAL DECRYPTED PNG
        # ====================================================

        decrypted_png_path = (
            DECRYPTED_IMAGE_DIR /
            f"{image_stem}_decrypted.png"
        )


        decrypted_write_ok = cv2.imwrite(
            str(
                decrypted_png_path
            ),
            decrypted
        )


        if not decrypted_write_ok:

            raise IOError(
                "cv2.imwrite() failed for decrypted image: "
                f"{decrypted_png_path}"
            )


        # ====================================================
        # SAVE DECRYPTED NUMPY ARRAY
        # ====================================================

        decrypted_npy_path = (
            DECRYPTION_ARRAY_DIR /
            f"{image_stem}_decrypted.npy"
        )


        np.save(
            str(
                decrypted_npy_path
            ),
            decrypted
        )


        # ====================================================
        # SAVE DECRYPTION METADATA / LINK TO ENCRYPTION
        # ====================================================

        decryption_metadata = {

            "version":
                metadata.get(
                    "version"
                ),

            "image":
                image_stem,

            "height":
                int(
                    decrypted.shape[0]
                ),

            "width":
                int(
                    decrypted.shape[1]
                ),

            "block_size":
                int(
                    metadata.get(
                        "block_size"
                    )
                ),

            "authentication_verified":
                bool(
                    authentication_verified
                ),

            "exact_reconstruction":
                bool(
                    exact_recovery
                ),

            "cipher_image":
                str(
                    cipher_png_path
                ),

            "decrypted_image":
                str(
                    decrypted_png_path
                ),

            "cipher_array":
                str(
                    cipher_npy_path
                ),

            "decrypted_array":
                str(
                    decrypted_npy_path
                ),

            "encryption_metadata":
                str(
                    metadata_json_path
                )

        }


        decryption_metadata_json_path = (
            DECRYPTION_ARRAY_DIR /
            f"{image_stem}_decryption_metadata.json"
        )


        with open(
            decryption_metadata_json_path,
            "w",
            encoding="utf-8"
        ) as metadata_file:

            json.dump(
                decryption_metadata,
                metadata_file,
                indent=4
            )


        # ====================================================
        # SAVE VISUAL COMPARISON + HISTOGRAM
        # ====================================================

        visual_paths = (
            save_visual_comparison(
                original=original,
                cipher=cipher,
                decrypted=decrypted,
                image_name=image_stem,
                output_root=RESULT_ROOT_LOCAL
            )
        )


        # ====================================================
        # SAVE CORRELATION PLOTS
        # ====================================================

        correlation_paths = (
            save_correlation_scatter_plots(
                original=original,
                cipher=cipher,
                image_name=image_stem,
                output_root=RESULT_ROOT_LOCAL
            )
        )


        # ====================================================
        # V2 METADATA INFORMATION
        # ====================================================

        salt_value = metadata.get(
            "salt"
        )

        nonce_value = metadata.get(
            "nonce"
        )

        content_descriptor = metadata.get(
            "content_descriptor"
        )

        content_digest = metadata.get(
            "content_digest"
        )

        reverse_diffusion_mode = metadata.get(
            "reverse_diffusion"
        )


        salt_bytes = (
            cell52_byte_length(
                salt_value
            )
        )

        nonce_bytes = (
            cell52_byte_length(
                nonce_value
            )
        )

        descriptor_bytes = (
            cell52_byte_length(
                content_descriptor
            )
        )


        # ====================================================
        # RECORD ALL OUTPUTS
        # ====================================================

        visual_output_records.append({

            "Image":
                image_stem,

            "Height":
                int(
                    original.shape[0]
                ),

            "Width":
                int(
                    original.shape[1]
                ),

            "Block_Size":
                int(
                    metadata.get(
                        "block_size"
                    )
                ),

            "Version":
                metadata.get(
                    "version"
                ),

            "KDF":
                metadata.get(
                    "kdf"
                ),

            "KDF_Iterations":
                int(
                    metadata.get(
                        "kdf_iterations"
                    )
                ),

            "Salt_Bytes":
                int(
                    salt_bytes
                ),

            "Nonce_Bytes":
                int(
                    nonce_bytes
                ),

            "Content_Descriptor_Bytes":
                int(
                    descriptor_bytes
                ),

            "Reverse_Diffusion":
                reverse_diffusion_mode,

            "Authentication_Verified":
                authentication_verified,

            "Exact_Reconstruction":
                exact_recovery,

            "Encryption_Time_Seconds":
                float(
                    encryption_elapsed
                ),

            "Decryption_Time_Seconds":
                float(
                    decryption_elapsed
                ),

            "Cipher_Image":
                str(
                    cipher_png_path
                ),

            "Cipher_Array":
                str(
                    cipher_npy_path
                ),

            "Encryption_Metadata":
                str(
                    metadata_json_path
                ),

            "Decrypted_Image":
                str(
                    decrypted_png_path
                ),

            "Decrypted_Array":
                str(
                    decrypted_npy_path
                ),

            "Decryption_Metadata":
                str(
                    decryption_metadata_json_path
                ),

            "Visual_Comparison":
                str(
                    visual_paths[
                        "visual"
                    ]
                ),

            "Histogram":
                str(
                    visual_paths[
                        "histogram"
                    ]
                ),

            "Horizontal_Correlation":
                str(
                    correlation_paths[
                        "horizontal"
                    ]
                ),

            "Vertical_Correlation":
                str(
                    correlation_paths[
                        "vertical"
                    ]
                ),

            "Diagonal_Correlation":
                str(
                    correlation_paths[
                        "diagonal"
                    ]
                )

        })


        # ====================================================
        # PRINT EXACT FILE LOCATIONS
        # ====================================================

        print()
        print("-" * 80)

        print(
            f"[OK] {image_stem}"
        )

        print(
            f"     Authentication       : "
            f"{authentication_verified}"
        )

        print(
            f"     Exact reconstruction : "
            f"{exact_recovery}"
        )

        print(
            f"     Encryption time      : "
            f"{encryption_elapsed:.6f} s"
        )

        print(
            f"     Decryption time      : "
            f"{decryption_elapsed:.6f} s"
        )

        print()

        print(
            f"     ENCRYPTED PNG       : "
            f"{cipher_png_path}"
        )

        print(
            f"     ENCRYPTED ARRAY     : "
            f"{cipher_npy_path}"
        )

        print(
            f"     ENCRYPTION METADATA : "
            f"{metadata_json_path}"
        )

        print()

        print(
            f"     DECRYPTED PNG      : "
            f"{decrypted_png_path}"
        )

        print(
            f"     DECRYPTED ARRAY    : "
            f"{decrypted_npy_path}"
        )

        print(
            f"     DECRYPTION METADATA: "
            f"{decryption_metadata_json_path}"
        )

        print()

        print(
            f"     COMPARISON          : "
            f"{visual_paths['visual']}"
        )

        print(
            f"     HISTOGRAM           : "
            f"{visual_paths['histogram']}"
        )

        print(
            f"     HORIZONTAL         : "
            f"{correlation_paths['horizontal']}"
        )

        print(
            f"     VERTICAL           : "
            f"{correlation_paths['vertical']}"
        )

        print(
            f"     DIAGONAL           : "
            f"{correlation_paths['diagonal']}"
        )


    # ========================================================
    # ERROR HANDLING
    # ========================================================

    except Exception as exc:

        print()
        print(
            f"[ERROR] {image_name}: "
            f"{type(exc).__name__}: {exc}"
        )


        error_height = None

        error_width = None


        if (
            isinstance(
                original,
                np.ndarray
            )
            and
            original.ndim >= 2
        ):

            error_height = int(
                original.shape[0]
            )

            error_width = int(
                original.shape[1]
            )


        visual_output_records.append({

            "Image":
                cell52_safe_stem(
                    image_name
                ),

            "Height":
                error_height,

            "Width":
                error_width,

            "Block_Size":
                None,

            "Version":
                None,

            "KDF":
                None,

            "KDF_Iterations":
                None,

            "Salt_Bytes":
                None,

            "Nonce_Bytes":
                None,

            "Content_Descriptor_Bytes":
                None,

            "Reverse_Diffusion":
                None,

            "Authentication_Verified":
                False,

            "Exact_Reconstruction":
                False,

            "Encryption_Time_Seconds":
                None,

            "Decryption_Time_Seconds":
                None,

            "Cipher_Image":
                None,

            "Cipher_Array":
                None,

            "Encryption_Metadata":
                None,

            "Decrypted_Image":
                None,

            "Decrypted_Array":
                None,

            "Decryption_Metadata":
                None,

            "Visual_Comparison":
                None,

            "Histogram":
                None,

            "Horizontal_Correlation":
                None,

            "Vertical_Correlation":
                None,

            "Diagonal_Correlation":
                None,

            "Error":
                f"{type(exc).__name__}: {exc}"

        })


# ============================================================
# 15. SAVE COMPLETE OUTPUT INDEX
# ============================================================

visual_output_index = pd.DataFrame(
    visual_output_records
)


visual_index_path = (
    SUMMARY_DIR /
    "visual_output_index.csv"
)


visual_output_index.to_csv(
    visual_index_path,
    index=False
)


# ============================================================
# 16. SAVE A SIMPLE CELL 52 SUMMARY
# ============================================================

successful_count = 0

exact_count = 0


if not visual_output_index.empty:

    successful_count = int(
        visual_output_index[
            "Authentication_Verified"
        ]
        .fillna(False)
        .astype(bool)
        .sum()
    )


    exact_count = int(
        visual_output_index[
            "Exact_Reconstruction"
        ]
        .fillna(False)
        .astype(bool)
        .sum()
    )


processed_count = (
    len(
        visual_output_records
    )
)


error_count = (
    processed_count
    -
    successful_count
)


summary_path = (
    SUMMARY_DIR /
    "cell52_output_summary.txt"
)


with open(
    summary_path,
    "w",
    encoding="utf-8"
) as summary_file:

    summary_file.write(
        "CELL 52 — V2 OUTPUT SUMMARY\n"
    )

    summary_file.write(
        "=" * 70
        + "\n"
    )

    summary_file.write(
        f"Secret source: "
        f"{CELL52_SECRET_SOURCE}\n"
    )

    summary_file.write(
        f"Images requested: "
        f"{VISUAL_TEST_COUNT}\n"
    )

    summary_file.write(
        f"Images processed: "
        f"{processed_count}\n"
    )

    summary_file.write(
        f"Authentication verified: "
        f"{successful_count}\n"
    )

    summary_file.write(
        f"Exact reconstruction: "
        f"{exact_count}\n"
    )

    summary_file.write(
        f"Errors: "
        f"{error_count}\n"
    )

    summary_file.write(
        f"Result root: "
        f"{RESULT_ROOT_LOCAL}\n"
    )

    summary_file.write(
        f"Cipher images: "
        f"{CIPHER_IMAGE_DIR}\n"
    )

    summary_file.write(
        f"Decrypted images: "
        f"{DECRYPTED_IMAGE_DIR}\n"
    )

    summary_file.write(
        f"Visual comparisons: "
        f"{VISUAL_OUTPUT_DIR}\n"
    )

    summary_file.write(
        f"Histograms: "
        f"{HISTOGRAM_DIR}\n"
    )

    summary_file.write(
        f"Correlation plots: "
        f"{CORRELATION_DIR}\n"
    )

    summary_file.write(
        f"Output index: "
        f"{visual_index_path}\n"
    )


# ============================================================
# 17. FINAL DISPLAY
# ============================================================

print()
print("=" * 80)
print("CELL 52 — V2 ALL OUTPUTS GENERATED")
print("=" * 80)

print()

print(
    f"Secret source       : "
    f"{CELL52_SECRET_SOURCE}"
)

print(
    f"Images requested    : "
    f"{VISUAL_TEST_COUNT}"
)

print(
    f"Images processed    : "
    f"{processed_count}"
)

print(
    f"Authentication      : "
    f"{successful_count}/{processed_count}"
)

print(
    f"Exact reconstruction: "
    f"{exact_count}/{processed_count}"
)

print(
    f"Errors              : "
    f"{error_count}"
)

print()

print(
    "============================================================"
)

print(
    "FINAL PROJECT OUTPUT LOCATIONS"
)

print(
    "============================================================"
)

print()

print(
    f"RESULT ROOT:"
)

print(
    f"  {RESULT_ROOT_LOCAL}"
)

print()

print(
    "ENCRYPTED IMAGES:"
)

print(
    f"  {CIPHER_IMAGE_DIR}"
)

print()

print(
    "ENCRYPTED ARRAYS + METADATA:"
)

print(
    f"  {ENCRYPTION_ARRAY_DIR}"
)

print()

print(
    "DECRYPTED IMAGES:"
)

print(
    f"  {DECRYPTED_IMAGE_DIR}"
)

print()

print(
    "DECRYPTED ARRAYS + METADATA:"
)

print(
    f"  {DECRYPTION_ARRAY_DIR}"
)

print()

print(
    "HISTOGRAMS:"
)

print(
    f"  {HISTOGRAM_DIR}"
)

print()

print(
    "CORRELATION PLOTS:"
)

print(
    f"  {CORRELATION_DIR}"
)

print()

print(
    "VISUAL COMPARISONS:"
)

print(
    f"  {VISUAL_OUTPUT_DIR}"
)

print()

print(
    "COMPLETE OUTPUT INDEX:"
)

print(
    f"  {visual_index_path}"
)

print()

print(
    "CELL 52 SUMMARY:"
)

print(
    f"  {summary_path}"
)

print()

# ============================================================
# 18. SHOW THE COMPLETE OUTPUT TABLE
# ============================================================

if not visual_output_index.empty:

    print(
        visual_output_index.to_string(
            index=False
        )
    )

print()

print("=" * 80)
print("CELL 52 COMPLETE")
print("=" * 80)


CELL 52 — OUTPUT DIRECTORIES
RESULT ROOT          : /content/drive/MyDrive/DNA Adaptive RESULT
CIPHER IMAGES        : /content/drive/MyDrive/DNA Adaptive RESULT/06_ENCRYPTION/cipher_images
ENCRYPTION ARRAYS    : /content/drive/MyDrive/DNA Adaptive RESULT/06_ENCRYPTION/encryption_arrays
DECRYPTED IMAGES     : /content/drive/MyDrive/DNA Adaptive RESULT/07_DECRYPTION/decrypted_images
DECRYPTION ARRAYS    : /content/drive/MyDrive/DNA Adaptive RESULT/07_DECRYPTION/decryption_arrays
HISTOGRAMS           : /content/drive/MyDrive/DNA Adaptive RESULT/09_SECURITY/histogram
CORRELATION PLOTS    : /content/drive/MyDrive/DNA Adaptive RESULT/09_SECURITY/correlation
VISUAL COMPARISONS   : /content/drive/MyDrive/DNA Adaptive RESULT/11_SUMMARY/visual_outputs

Total images available for CELL 52: 49
Backward ciphertext diffusion applied.

--------------------------------------------------------------------------------
[OK] Copy of Copy of image_0050
     Authentication       : True
     Exact reconstruc

In [ ]:
# ============================================================
# NEW CELL — CIPHER MSE, PSNR AND CHI-SQUARE ANALYSIS
# ============================================================
#
# PURPOSE:
#   Calculate MSE, PSNR and Chi-Square between:
#
#       Original Plaintext Image
#                    vs
#       Final Cipher Image
#
# IMPORTANT:
#   The existing CELL 48 MSE/PSNR values measure:
#
#       Original Image vs Decrypted Image
#
#   Those metrics are reconstruction metrics.
#
#   This cell calculates:
#
#       Original Image vs Cipher Image
#
#   which measures how different the ciphertext is from the
#   original image.
#
# OUTPUT:
#
#   /08_METRICS/cipher_quality_metrics.csv
#
#   and an updated:
#
#   /08_METRICS/proposed_method_metrics_with_cipher_quality.csv
#
# ============================================================


from pathlib import Path

import numpy as np
import pandas as pd
import cv2


# ============================================================
# 1. RESULT ROOT
# ============================================================

if (
    "RESULT_ROOT" in globals()
    and RESULT_ROOT is not None
):

    RESULT_ROOT_LOCAL = Path(
        RESULT_ROOT
    )

else:

    RESULT_ROOT_LOCAL = Path(
        "/content/drive/MyDrive/"
        "DNA Adaptive RESULT"
    )


# ============================================================
# 2. PROJECT DIRECTORIES
# ============================================================

METRICS_ROOT = (
    RESULT_ROOT_LOCAL /
    "08_METRICS"
)

METRICS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Cipher arrays generated by CELL 52
# ------------------------------------------------------------

if "ENCRYPTION_ARRAY_DIR" in globals():

    CIPHER_ARRAY_DIR = Path(
        ENCRYPTION_ARRAY_DIR
    )

else:

    CIPHER_ARRAY_DIR = (
        RESULT_ROOT_LOCAL /
        "06_ENCRYPTION" /
        "encryption_arrays"
    )


# ------------------------------------------------------------
# Output files
# ------------------------------------------------------------

CIPHER_QUALITY_CSV = (
    METRICS_ROOT /
    "cipher_quality_metrics.csv"
)

COMBINED_METRICS_CSV = (
    METRICS_ROOT /
    "proposed_method_metrics_with_cipher_quality.csv"
)


# ============================================================
# 3. METRIC FUNCTIONS
# ============================================================

def calculate_cipher_mse(
    original,
    cipher
):
    """
    MSE between the original plaintext image and
    the final ciphertext image.
    """

    original = np.asarray(
        original,
        dtype=np.float64
    )

    cipher = np.asarray(
        cipher,
        dtype=np.float64
    )

    if original.shape != cipher.shape:

        raise ValueError(
            "Original and cipher images must have "
            "the same shape."
        )

    mse = np.mean(
        (original - cipher) ** 2
    )

    return float(mse)


def calculate_cipher_psnr(
    original,
    cipher,
    max_pixel_value=255.0
):
    """
    PSNR between the original plaintext image and
    the final ciphertext image.
    """

    mse = calculate_cipher_mse(
        original,
        cipher
    )

    if mse == 0:

        return float("inf")

    psnr = (
        10.0
        *
        np.log10(
            (max_pixel_value ** 2)
            /
            mse
        )
    )

    return float(psnr)


def calculate_cipher_chi_square(
    cipher,
    levels=256
):
    """
    Pearson Chi-Square statistic for the ciphertext
    histogram against an ideal uniform distribution.

    Expected frequency for every intensity level:

        E = N / 256

    where N is the total number of ciphertext pixels.

    Degrees of freedom:

        df = 255
    """

    cipher = np.asarray(
        cipher,
        dtype=np.uint8
    )

    values = cipher.ravel()

    total_pixels = len(values)

    if total_pixels == 0:

        raise ValueError(
            "Cannot calculate Chi-Square for an empty image."
        )

    # --------------------------------------------------------
    # Observed histogram
    # --------------------------------------------------------

    observed = np.bincount(
        values,
        minlength=levels
    ).astype(
        np.float64
    )

    # --------------------------------------------------------
    # Expected uniform histogram
    # --------------------------------------------------------

    expected_frequency = (
        total_pixels
        /
        float(levels)
    )

    expected = np.full(
        levels,
        expected_frequency,
        dtype=np.float64
    )

    # --------------------------------------------------------
    # Pearson Chi-Square
    # --------------------------------------------------------

    chi_square = np.sum(
        (
            (observed - expected) ** 2
        )
        /
        expected
    )

    degrees_of_freedom = (
        levels - 1
    )

    return (
        float(chi_square),
        int(degrees_of_freedom)
    )


# ============================================================
# 4. RESOLVE DATASET IMAGE PATH
# ============================================================

def resolve_cipher_quality_image_path(
    item
):

    if isinstance(
        item,
        dict
    ):

        image_path = (

            item.get("path")

            or

            item.get("image_path")

        )

    else:

        image_path = item


    if image_path is None:

        return None


    return Path(
        image_path
    )


# ============================================================
# 5. BUILD IMAGE LIST
# ============================================================

if (
    "valid_dataset_files" in globals()
    and valid_dataset_files is not None
):

    image_items = list(
        valid_dataset_files
    )

elif (
    "dataset_files" in globals()
    and dataset_files is not None
):

    image_items = list(
        dataset_files
    )

else:

    raise NameError(
        "Neither valid_dataset_files nor dataset_files "
        "is available. Run the dataset validation cell first."
    )


if len(image_items) == 0:

    raise RuntimeError(
        "No dataset images were found."
    )


# ============================================================
# 6. PROCESS ALL IMAGES
# ============================================================

records = []


print()
print("=" * 80)
print("CIPHER MSE / PSNR / CHI-SQUARE ANALYSIS")
print("=" * 80)

print(
    f"Dataset images : {len(image_items)}"
)

print(
    f"Cipher arrays  : {CIPHER_ARRAY_DIR}"
)

print()


for image_number, item in enumerate(
    image_items,
    start=1
):

    # --------------------------------------------------------
    # Resolve original image
    # --------------------------------------------------------

    image_path = (
        resolve_cipher_quality_image_path(
            item
        )
    )


    if image_path is None:

        print(
            f"WARNING [{image_number}]: "
            "Invalid image path."
        )

        continue


    # --------------------------------------------------------
    # Load original image
    # --------------------------------------------------------

    original = cv2.imread(
        str(image_path),
        cv2.IMREAD_GRAYSCALE
    )


    if original is None:

        print(
            f"WARNING [{image_number}]: "
            f"Could not read {image_path}"
        )

        continue


    original = np.asarray(
        original,
        dtype=np.uint8
    )


    # --------------------------------------------------------
    # Locate corresponding ciphertext array
    # --------------------------------------------------------
    #
    # CELL 52 saves:
    #
    #   image_stem_cipher.npy
    #
    # --------------------------------------------------------

    image_stem = image_path.stem

    cipher_npy_path = (
        CIPHER_ARRAY_DIR /
        f"{image_stem}_cipher.npy"
    )


    if not cipher_npy_path.exists():

        print(
            f"WARNING [{image_number}]: "
            f"Cipher array not found: "
            f"{cipher_npy_path.name}"
        )

        continue


    # --------------------------------------------------------
    # Load ciphertext
    # --------------------------------------------------------

    cipher = np.load(
        str(cipher_npy_path)
    )

    cipher = np.asarray(
        cipher,
        dtype=np.uint8
    )


    # --------------------------------------------------------
    # Shape validation
    # --------------------------------------------------------

    if original.shape != cipher.shape:

        raise RuntimeError(

            f"Shape mismatch for {image_path.name}: "

            f"original={original.shape}, "

            f"cipher={cipher.shape}"

        )


    # ========================================================
    # MSE
    # ========================================================

    mse_cipher = calculate_cipher_mse(
        original,
        cipher
    )


    # ========================================================
    # PSNR
    # ========================================================

    psnr_cipher = calculate_cipher_psnr(
        original,
        cipher
    )


    # ========================================================
    # CHI-SQUARE
    # ========================================================

    (
        chi_square_cipher,
        chi_square_df

    ) = calculate_cipher_chi_square(
        cipher
    )


    # ========================================================
    # BASIC CIPHER STATISTICS
    # ========================================================

    cipher_mean = float(
        np.mean(cipher)
    )

    cipher_std = float(
        np.std(cipher)
    )

    cipher_min = int(
        np.min(cipher)
    )

    cipher_max = int(
        np.max(cipher)
    )


    # ========================================================
    # STORE RESULT
    # ========================================================

    records.append({

        "Image":
            image_path.name,

        "Height":
            int(original.shape[0]),

        "Width":
            int(original.shape[1]),

        "Cipher_MSE":
            mse_cipher,

        "Cipher_PSNR_dB":
            psnr_cipher,

        "Cipher_Chi_Square":
            chi_square_cipher,

        "Chi_Square_Degrees_of_Freedom":
            chi_square_df,

        "Cipher_Mean":
            cipher_mean,

        "Cipher_Std":
            cipher_std,

        "Cipher_Min":
            cipher_min,

        "Cipher_Max":
            cipher_max

    })


    print(
        f"[{image_number:02d}/{len(image_items):02d}] "
        f"{image_path.name} | "
        f"MSE={mse_cipher:.4f} | "
        f"PSNR={psnr_cipher:.4f} dB | "
        f"Chi-Square={chi_square_cipher:.4f}"
    )


# ============================================================
# 7. CREATE DATAFRAME
# ============================================================

cipher_quality_df = pd.DataFrame(
    records
)


if cipher_quality_df.empty:

    raise RuntimeError(
        "No cipher-quality metrics were calculated."
    )


# ============================================================
# 8. SAVE CIPHER METRICS
# ============================================================

cipher_quality_df.to_csv(
    CIPHER_QUALITY_CSV,
    index=False
)


# ============================================================
# 9. MERGE WITH EXISTING CELL 48 RESULTS
# ============================================================

existing_metrics_df = None


existing_metrics_path = (
    METRICS_ROOT /
    "proposed_method_metrics.csv"
)


if existing_metrics_path.exists():

    existing_metrics_df = pd.read_csv(
        existing_metrics_path
    )


elif (
    "all_image_metrics" in globals()
    and isinstance(
        all_image_metrics,
        pd.DataFrame
    )
):

    existing_metrics_df = (
        all_image_metrics.copy()
    )


# ------------------------------------------------------------
# Merge if Cell 48 metrics are available
# ------------------------------------------------------------

if (
    existing_metrics_df is not None
    and
    not existing_metrics_df.empty
):

    combined_metrics_df = (
        existing_metrics_df
        .merge(
            cipher_quality_df[
                [
                    "Image",
                    "Cipher_MSE",
                    "Cipher_PSNR_dB",
                    "Cipher_Chi_Square",
                    "Chi_Square_Degrees_of_Freedom",
                    "Cipher_Mean",
                    "Cipher_Std"
                ]
            ],
            on="Image",
            how="left"
        )
    )


    combined_metrics_df.to_csv(
        COMBINED_METRICS_CSV,
        index=False
    )

else:

    combined_metrics_df = (
        cipher_quality_df.copy()
    )

    combined_metrics_df.to_csv(
        COMBINED_METRICS_CSV,
        index=False
    )


# ============================================================
# 10. SUMMARY STATISTICS
# ============================================================

mean_mse = float(
    cipher_quality_df[
        "Cipher_MSE"
    ].mean()
)

mean_psnr = float(
    cipher_quality_df[
        "Cipher_PSNR_dB"
    ].mean()
)

mean_chi_square = float(
    cipher_quality_df[
        "Cipher_Chi_Square"
    ].mean()
)


median_mse = float(
    cipher_quality_df[
        "Cipher_MSE"
    ].median()
)

median_psnr = float(
    cipher_quality_df[
        "Cipher_PSNR_dB"
    ].median()
)

median_chi_square = float(
    cipher_quality_df[
        "Cipher_Chi_Square"
    ].median()
)


# ============================================================
# 11. PRINT SUMMARY
# ============================================================

print()
print("=" * 80)
print("CIPHER QUALITY SUMMARY")
print("=" * 80)

print(
    f"Images processed              : "
    f"{len(cipher_quality_df)}"
)

print(
    f"Mean Cipher MSE               : "
    f"{mean_mse:.6f}"
)

print(
    f"Mean Cipher PSNR              : "
    f"{mean_psnr:.6f} dB"
)

print(
    f"Mean Cipher Chi-Square        : "
    f"{mean_chi_square:.6f}"
)

print(
    f"Median Cipher MSE             : "
    f"{median_mse:.6f}"
)

print(
    f"Median Cipher PSNR            : "
    f"{median_psnr:.6f} dB"
)

print(
    f"Median Cipher Chi-Square      : "
    f"{median_chi_square:.6f}"
)

print()
print(
    f"Cipher metrics saved to:"
)

print(
    f"    {CIPHER_QUALITY_CSV}"
)

print()

print(
    f"Combined metrics saved to:"
)

print(
    f"    {COMBINED_METRICS_CSV}"
)

print("=" * 80)


CIPHER MSE / PSNR / CHI-SQUARE ANALYSIS
Dataset images : 49
Cipher arrays  : /content/drive/MyDrive/DNA Adaptive RESULT/06_ENCRYPTION/encryption_arrays

[01/49] Copy of Copy of image_0050.png | MSE=8606.4010 | PSNR=8.7826 dB | Chi-Square=240.1719
[02/49] Copy of Copy of image_0051.png | MSE=14485.6245 | PSNR=6.5214 dB | Chi-Square=268.2031
[03/49] Copy of Copy of image_0052.png | MSE=14602.5554 | PSNR=6.4865 dB | Chi-Square=281.2578
[04/49] Copy of Copy of image_0053.png | MSE=14518.6692 | PSNR=6.5115 dB | Chi-Square=219.8984
[05/49] Copy of Copy of image_0054.png | MSE=14604.7454 | PSNR=6.4859 dB | Chi-Square=286.6797
[06/49] Copy of Copy of image_0055.png | MSE=8763.0715 | PSNR=8.7042 dB | Chi-Square=255.4141
[07/49] Copy of Copy of image_0056.png | MSE=8716.9186 | PSNR=8.7272 dB | Chi-Square=230.5234
[08/49] Copy of Copy of image_0057.png | MSE=8753.8100 | PSNR=8.7088 dB | Chi-Square=282.5469
[09/49] Copy of Copy of image_0058.png | MSE=8745.9763 | PSNR=8.7127 dB | Chi-Square=241.0

In [ ]:
# ============================================================
# CELL 53 — SECURITY DASHBOARD
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# RESULT ROOT
# ============================================================

if (
    "RESULT_ROOT" in globals()
    and RESULT_ROOT is not None
):

    RESULT_ROOT_LOCAL = Path(
        RESULT_ROOT
    )

else:

    RESULT_ROOT_LOCAL = Path(
        "/content/drive/MyDrive/"
        "DNA Adaptive RESULT"
    )


SUMMARY_ROOT = (
    RESULT_ROOT_LOCAL /
    "11_SUMMARY"
)

SUMMARY_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


METRICS_ROOT = (
    RESULT_ROOT_LOCAL /
    "08_METRICS"
)

METRICS_CSV_PATH = (
    METRICS_ROOT /
    "proposed_method_metrics.csv"
)

SECURITY_DASHBOARD_PATH = (
    SUMMARY_ROOT /
    "security_dashboard.xlsx"
)


# ============================================================
# LOAD ACTUAL CELL 48 RESULTS
# ============================================================

if (
    "all_image_metrics" in globals()
    and
    isinstance(
        all_image_metrics,
        pd.DataFrame
    )
    and
    not all_image_metrics.empty
):

    metrics_df = (
        all_image_metrics
        .copy()
        .reset_index(drop=True)
    )

    metrics_source = (
        "all_image_metrics"
    )

elif METRICS_CSV_PATH.exists():

    metrics_df = pd.read_csv(
        METRICS_CSV_PATH
    )

    metrics_source = str(
        METRICS_CSV_PATH
    )

else:

    raise RuntimeError(
        "\nCELL 53 cannot run yet.\n\n"
        "CELL 48 has not generated the required "
        "metrics table.\n\n"
        "Run CELL 48 successfully first."
    )


# ============================================================
# REQUIRED VALIDATION
# ============================================================

if metrics_df.empty:

    raise RuntimeError(
        "CELL 48 produced an empty metrics table."
    )


print("=" * 80)
print("CELL 53 — SECURITY DASHBOARD")
print("=" * 80)

print()

print(
    f"Metrics source : {metrics_source}"
)

print(
    f"Images / rows  : {len(metrics_df)}"
)

print(
    f"Columns        : {len(metrics_df.columns)}"
)

print()

print(
    "Available metrics:"
)

for column in metrics_df.columns:

    print(
        f"    {column}"
    )


# ============================================================
# CREATE DASHBOARD
# ============================================================

def create_security_dashboard(
    metrics_df,
    output_path
):

    metrics_df = (
        metrics_df
        .copy()
        .reset_index(drop=True)
    )


    with pd.ExcelWriter(
        output_path,
        engine="openpyxl"
    ) as writer:


        # ====================================================
        # 1 — COMPLETE
        # ====================================================

        metrics_df.to_excel(
            writer,
            sheet_name="Per_Image_Metrics",
            index=False
        )


        # ====================================================
        # 2 — STATISTICAL
        # ====================================================

        statistical_columns = [

            "Image",

            "Entropy_Original",
            "Entropy_Cipher",
            "Entropy_Decrypted",

            "Correlation_H",
            "Correlation_V",
            "Correlation_D",

            "Contrast",
            "LSE"
        ]

        available = [
            c
            for c in statistical_columns
            if c in metrics_df.columns
        ]

        if available:

            metrics_df[
                available
            ].to_excel(
                writer,
                sheet_name="Statistical",
                index=False
            )

        else:

            pd.DataFrame({
                "Information": [
                    "No statistical metrics available."
                ]
            }).to_excel(
                writer,
                sheet_name="Statistical",
                index=False
            )


        # ====================================================
        # 3 — DIFFERENTIAL
        # ====================================================

        differential_columns = [

            "Image",
            "NPCR_Percent",
            "UACI_Percent",
            "Changed_Cipher_Pixels"
        ]

        available = [
            c
            for c in differential_columns
            if c in metrics_df.columns
        ]

        if available:

            metrics_df[
                available
            ].to_excel(
                writer,
                sheet_name="Differential",
                index=False
            )

        else:

            pd.DataFrame({
                "Information": [
                    "NPCR/UACI metrics not available."
                ]
            }).to_excel(
                writer,
                sheet_name="Differential",
                index=False
            )


        # ====================================================
        # 4 — KEY SENSITIVITY
        # ====================================================

        key_columns = [

            "Image",
            "Key_Sensitivity_Percent",
            "Modified_Key_Sensitivity_Percent",
            "Key_Space",
            "Key_Space_Bits"
        ]

        available = [
            c
            for c in key_columns
            if c in metrics_df.columns
        ]

        if available:

            metrics_df[
                available
            ].to_excel(
                writer,
                sheet_name="Key_Sensitivity",
                index=False
            )

        else:

            pd.DataFrame({
                "Information": [
                    "Key-sensitivity results are generated "
                    "by separate experiment cells."
                ]
            }).to_excel(
                writer,
                sheet_name="Key_Sensitivity",
                index=False
            )


        # ====================================================
        # 5 — ATTACK
        # ====================================================

        attack_columns = [

            "Image",
            "Plaintext_Change_Block",
            "Propagation_Percent",
            "Noise_Level",
            "Noise_Reconstruction_Percent",
            "Data_Loss_Percent",
            "Data_Loss_Reconstruction_Percent"
        ]

        available = [
            c
            for c in attack_columns
            if c in metrics_df.columns
        ]

        if available:

            metrics_df[
                available
            ].to_excel(
                writer,
                sheet_name="Attack",
                index=False
            )

        else:

            pd.DataFrame({
                "Information": [
                    "Attack-analysis results are generated "
                    "by separate experiment cells."
                ]
            }).to_excel(
                writer,
                sheet_name="Attack",
                index=False
            )


        # ====================================================
        # 6 — CORRECTNESS
        # ====================================================

        correctness_columns = [

            "Image",
            "MSE",
            "PSNR_dB",
            "Different_Pixels",
            "Exact_Reconstruction"
        ]

        available = [
            c
            for c in correctness_columns
            if c in metrics_df.columns
        ]

        if available:

            metrics_df[
                available
            ].to_excel(
                writer,
                sheet_name="Correctness",
                index=False
            )


        # ====================================================
        # 7 — PERFORMANCE
        # ====================================================

        performance_columns = [

            "Image",

            "Encryption_Time_Seconds",
            "Decryption_Time_Seconds",

            "Throughput_Mbps",

            "Memory_Before_MB",
            "Memory_After_MB",
            "Memory_Delta_MB"
        ]

        available = [
            c
            for c in performance_columns
            if c in metrics_df.columns
        ]

        if available:

            metrics_df[
                available
            ].to_excel(
                writer,
                sheet_name="Performance",
                index=False
            )

        else:

            pd.DataFrame({
                "Information": [
                    "No performance metrics available."
                ]
            }).to_excel(
                writer,
                sheet_name="Performance",
                index=False
            )


        # ====================================================
        # 8 — SUMMARY
        # ====================================================

        summary_records = []

        for column in metrics_df.columns:

            if column == "Image":

                continue

            values = pd.to_numeric(
                metrics_df[column],
                errors="coerce"
            )

            values = values[
                np.isfinite(values)
            ]

            if len(values) == 0:

                continue

            summary_records.append({

                "Metric":
                    column,

                "Count":
                    int(len(values)),

                "Mean":
                    float(values.mean()),

                "Median":
                    float(values.median()),

                "Std":
                    float(
                        values.std(
                            ddof=1
                        )
                        if len(values) > 1
                        else 0.0
                    ),

                "Min":
                    float(values.min()),

                "Max":
                    float(values.max())
            })


        summary_df = pd.DataFrame(
            summary_records
        )

        if summary_df.empty:

            summary_df = pd.DataFrame({
                "Information": [
                    "No numeric metrics available."
                ]
            })


        summary_df.to_excel(
            writer,
            sheet_name="Summary",
            index=False
        )


        # ====================================================
        # FORMATTING
        # ====================================================

        workbook = writer.book

        for worksheet in workbook.worksheets:

            worksheet.freeze_panes = "A2"

            for column_cells in worksheet.columns:

                max_length = 0

                column_letter = (
                    column_cells[0]
                    .column_letter
                )

                for cell in column_cells:

                    try:

                        max_length = max(
                            max_length,
                            len(
                                str(
                                    cell.value
                                )
                            )
                        )

                    except Exception:

                        pass

                worksheet.column_dimensions[
                    column_letter
                ].width = min(
                    max(
                        max_length + 2,
                        10
                    ),
                    40
                )


# ============================================================
# CREATE
# ============================================================

create_security_dashboard(
    metrics_df=metrics_df,
    output_path=SECURITY_DASHBOARD_PATH
)


# ============================================================
# FINAL DISPLAY
# ============================================================

print()
print(
    f"Dashboard saved to:"
)

print(
    SECURITY_DASHBOARD_PATH
)

print()

print(
    "Sheets:"
)

for sheet_name in [

    "Per_Image_Metrics",
    "Statistical",
    "Differential",
    "Key_Sensitivity",
    "Attack",
    "Correctness",
    "Performance",
    "Summary"

]:

    print(
        f"    {sheet_name}"
    )

print()

print(
    "CELL 53 COMPLETE"
)

CELL 53 — SECURITY DASHBOARD

Metrics source : all_image_metrics
Images / rows  : 49
Columns        : 33

Available metrics:
    Image
    Height
    Width
    Block_Size
    Method_Version
    KDF
    KDF_Iterations
    Salt_Bytes
    Nonce_Bytes
    Feature_Descriptor_Bytes
    Authentication
    Controlled_NPCR_Nonce
    Plaintext_Changed_Pixels
    Differential_Salt_Shared
    Differential_Nonce_Shared
    Content_Digest_Changed
    Entropy_Original
    Entropy_Cipher
    Entropy_Decrypted
    Correlation_H
    Correlation_V
    Correlation_D
    NPCR_Percent
    UACI_Percent
    Changed_Cipher_Pixels
    Changed_Cipher_Percent
    Total_Cipher_Pixels
    MSE
    PSNR_dB
    Encryption_Time_Seconds
    Decryption_Time_Seconds
    Different_Pixels
    Exact_Reconstruction

Dashboard saved to:
/content/drive/MyDrive/DNA Adaptive RESULT/11_SUMMARY/security_dashboard.xlsx

Sheets:
    Per_Image_Metrics
    Statistical
    Differential
    Key_Sensitivity
    Attack
    Correctness
    

In [ ]:
# ============================================================
# CELL 54 — REPRODUCIBILITY REPORT
# ============================================================

import sys
import platform
import hashlib
import json
import datetime
import cv2
import numpy as np


def convert_to_json_serializable(
    value
):
    """
    Convert NumPy/path/config objects into JSON-compatible
    Python objects.
    """

    if isinstance(
        value,
        (np.integer,)
    ):
        return int(value)

    if isinstance(
        value,
        (np.floating,)
    ):
        return float(value)

    if isinstance(
        value,
        np.ndarray
    ):
        return value.tolist()

    if isinstance(
        value,
        Path
    ):
        return str(value)

    if isinstance(
        value,
        tuple
    ):
        return list(value)

    if isinstance(
        value,
        dict
    ):
        return {
            str(k):
            convert_to_json_serializable(v)
            for k, v in value.items()
        }

    if isinstance(
        value,
        list
    ):
        return [
            convert_to_json_serializable(v)
            for v in value
        ]

    return value


# ------------------------------------------------------------
# Collect reproducibility information
# ------------------------------------------------------------

reproducibility_report = {
    "Execution_Date_UTC": (
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat()
    ),

    "Python_Version": (
        sys.version
    ),

    "Platform": (
        platform.platform()
    ),

    "Processor": (
        platform.processor()
    ),

    "NumPy_Version": (
        np.__version__
    ),

    "OpenCV_Version": (
        cv2.__version__
    ),

    "Pandas_Version": (
        pd.__version__
    ),

    "Hash_Algorithm": (
        "SHA-256"
    ),

    "Image_Mode": (
        CONFIG.get(
            "IMAGE_MODE",
            "GRAYSCALE"
        )
    ),

    "Pixel_Depth": 8,

    "Block_Size": (
        CONFIG.get(
            "BLOCK_SIZE",
            16
        )
    ),

    "Chaotic_Map": (
        "Logistic"
    ),

    "Chaotic_Equation": (
        "x_(n+1) = r*x_n*(1-x_n)"
    ),

    "Initial_Condition_Derivation": (
        "SHA-256-derived"
    ),

    "Control_Parameter_Derivation": (
        "SHA-256-derived"
    ),

    "Chaotic_Parameter_Range": (
        "r in (3.9, 4.0)"
    ),

    "Chaos_Transient": (
        CONFIG.get(
            "CHAOS_TRANSIENT",
            1000
        )
    ),

    "DNA_Rules": (
        {
            str(k): v
            for k, v in DNA_RULES.items()
        }
        if "DNA_RULES" in globals()
        else {
            "1": "ATCG",
            "2": "ACGT",
            "3": "TGCA",
            "4": "TCGA",
            "5": "CTAG",
            "6": "CATG",
            "7": "GTAC",
            "8": "GATC"
        }
    ),

    "DNA_XOR_Table": {
        "A": {
            "A": "A",
            "T": "T",
            "C": "C",
            "G": "G"
        },
        "T": {
            "A": "T",
            "T": "A",
            "C": "G",
            "G": "C"
        },
        "C": {
            "A": "C",
            "T": "G",
            "C": "A",
            "G": "T"
        },
        "G": {
            "A": "G",
            "T": "C",
            "C": "T",
            "G": "A"
        }
    },

    "Adaptive_Rule_Equation": (
        "r_i = (uint32(Z_i[0:4]) mod 8) + 1"
    ),

    "Block_Hash": (
        "H_i = SHA256(B_i)"
    ),

    "Adaptive_State_Input": (
        "Z_i = SHA256(H_i || S_i)"
    ),

    "Feedback_Equation": (
        "M_i = SHA256(S_i || F_i || Z_i)"
    ),

    "State_Evolution": (
        "S_(i+1) = SHA256(S_i || C_i || H_i)"
    ),

    "Feedback_Initialization": (
        "F_0 = 0"
    ),

    "Feedback_Source": (
        "Previous ciphertext block"
    ),

    "Processing_All_Images": True,

    "Dataset_Path": (
        str(DATASET_ROOT)
        if "DATASET_ROOT" in globals()
        else "DNA Adaptive DATASET"
    ),

    "Result_Path": (
        str(RESULT_ROOT)
    ),

    "Dataset_Count": (
        len(dataset_files)
        if "dataset_files" in globals()
        else None
    ),

    "Dataset_Image_Size": (
        "256x256"
    ),

    "Dataset_Mode": (
        "Grayscale"
    ),

    "Random_Seed": (
        CONFIG.get(
            "RANDOM_SEED",
            42
        )
    ),

    "NumPy_Random_Seed": (
        CONFIG.get(
            "RANDOM_SEED",
            42
        )
    )
}


reproducibility_report = (
    convert_to_json_serializable(
        reproducibility_report
    )
)

REPRODUCIBILITY_PATH = (
    RESULT_ROOT /
    "11_SUMMARY" /
    "experiment_config.json"
)

REPRODUCIBILITY_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    REPRODUCIBILITY_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        reproducibility_report,
        file,
        indent=4,
        ensure_ascii=False
    )

print("=" * 80)
print("REPRODUCIBILITY REPORT")
print("=" * 80)

print(
    json.dumps(
        reproducibility_report,
        indent=2,
        ensure_ascii=False
    )
)

print()
print(
    f"Saved: {REPRODUCIBILITY_PATH}"
)

REPRODUCIBILITY REPORT
{
  "Execution_Date_UTC": "2026-10-03T17:32:00.298631+00:00",
  "Python_Version": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "Platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "Processor": "x86_64",
  "NumPy_Version": "2.1.3",
  "OpenCV_Version": "4.14.0",
  "Pandas_Version": "2.2.3",
  "Hash_Algorithm": "SHA-256",
  "Image_Mode": "GRAYSCALE",
  "Pixel_Depth": 8,
  "Block_Size": 16,
  "Chaotic_Map": "Logistic",
  "Chaotic_Equation": "x_(n+1) = r*x_n*(1-x_n)",
  "Initial_Condition_Derivation": "SHA-256-derived",
  "Control_Parameter_Derivation": "SHA-256-derived",
  "Chaotic_Parameter_Range": "r in (3.9, 4.0)",
  "Chaos_Transient": 1000,
  "DNA_Rules": {
    "1": "ATCG",
    "2": "ACGT",
    "3": "TGCA",
    "4": "TCGA",
    "5": "CTAG",
    "6": "CATG",
    "7": "GTAC",
    "8": "GATC"
  },
  "DNA_XOR_Table": {
    "A": {
      "A": "A",
      "T": "T",
      "C": "C",
      "G": "G"
    },
    "T": {
      "A": "T",
      "T": "A",
      "C": "G"

In [ ]:
# ============================================================
# CELL 55 — FINAL RESEARCH SUMMARY
# ============================================================

def safe_mean(
    dataframe,
    column
):
    """
    Return finite mean for a dataframe column.
    """

    if column not in dataframe.columns:
        return np.nan

    values = pd.to_numeric(
        dataframe[column],
        errors="coerce"
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return np.nan

    return float(
        values.mean()
    )


def format_metric(
    value,
    decimals=6
):
    """
    Format numeric summary values.
    """

    if value is None:
        return "N/A"

    try:

        if not np.isfinite(value):
            return "N/A"

        return f"{value:.{decimals}f}"

    except Exception:

        return "N/A"


# ------------------------------------------------------------
# Number of successfully reconstructed images
# ------------------------------------------------------------

if (
    "Exact_Reconstruction"
    in all_image_metrics.columns
):

    exact_series = (
        all_image_metrics[
            "Exact_Reconstruction"
        ]
    )

    exact_count = int(
        exact_series.astype(bool).sum()
    )

else:

    exact_count = 0


total_images = len(
    all_image_metrics
)

# ------------------------------------------------------------
# Mean metrics
# ------------------------------------------------------------

mean_cipher_entropy = safe_mean(
    all_image_metrics,
    "Entropy_Cipher"
)

mean_npcr = safe_mean(
    all_image_metrics,
    "NPCR_Percent"
)

mean_uaci = safe_mean(
    all_image_metrics,
    "UACI_Percent"
)

mean_corr_h = safe_mean(
    all_image_metrics,
    "Correlation_H"
)

mean_corr_v = safe_mean(
    all_image_metrics,
    "Correlation_V"
)

mean_corr_d = safe_mean(
    all_image_metrics,
    "Correlation_D"
)

mean_enc_time = safe_mean(
    all_image_metrics,
    "Encryption_Time_Seconds"
)

mean_dec_time = safe_mean(
    all_image_metrics,
    "Decryption_Time_Seconds"
)

# ------------------------------------------------------------
# Throughput
# ------------------------------------------------------------

image_pixels = (
    256 * 256
)

mean_enc_throughput = np.nan
mean_dec_throughput = np.nan

if (
    np.isfinite(mean_enc_time)
    and mean_enc_time > 0
):

    mean_enc_throughput = (
        image_pixels /
        mean_enc_time
    )

if (
    np.isfinite(mean_dec_time)
    and mean_dec_time > 0
):

    mean_dec_throughput = (
        image_pixels /
        mean_dec_time
    )

# ------------------------------------------------------------
# Final summary dictionary
# ------------------------------------------------------------

final_summary = {
    "Images_Processed": total_images,
    "Image_Size": "256 x 256",
    "Image_Mode": "Grayscale",
    "Pixel_Depth": "8-bit",
    "Block_Size": (
        CONFIG.get(
            "BLOCK_SIZE",
            16
        )
    ),
    "DNA_Rules": 8,
    "Hash": "SHA-256",
    "Chaotic_Map": "Logistic",
    "Feedback": "Enabled",

    "Mean_Cipher_Entropy": (
        mean_cipher_entropy
    ),

    "Mean_NPCR_Percent": (
        mean_npcr
    ),

    "Mean_UACI_Percent": (
        mean_uaci
    ),

    "Mean_Correlation_H": (
        mean_corr_h
    ),

    "Mean_Correlation_V": (
        mean_corr_v
    ),

    "Mean_Correlation_D": (
        mean_corr_d
    ),

    "Mean_Encryption_Time_Seconds": (
        mean_enc_time
    ),

    "Mean_Decryption_Time_Seconds": (
        mean_dec_time
    ),

    "Mean_Encryption_Throughput_Pixels_Per_Second": (
        mean_enc_throughput
    ),

    "Mean_Decryption_Throughput_Pixels_Per_Second": (
        mean_dec_throughput
    ),

    "Exact_Reconstruction": (
        f"{exact_count}/{total_images}"
    )
}


FINAL_SUMMARY_JSON = (
    RESULT_ROOT /
    "11_SUMMARY" /
    "final_research_summary.json"
)

with open(
    FINAL_SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        convert_to_json_serializable(
            final_summary
        ),
        file,
        indent=4
    )


# ------------------------------------------------------------
# Print final summary
# ------------------------------------------------------------

print()
print("=" * 80)
print("PROPOSED METHOD EXPERIMENT SUMMARY")
print("=" * 80)

print()
print(
    f"Images processed       : {total_images}"
)

print(
    "Image size             : 256 × 256"
)

print(
    "Image mode             : Grayscale"
)

print(
    "Block size             : "
    f"{CONFIG.get('BLOCK_SIZE', 16)} × "
    f"{CONFIG.get('BLOCK_SIZE', 16)}"
)

print(
    "DNA rules              : 8"
)

print(
    "Hash                   : SHA-256"
)

print(
    "Chaotic map            : Logistic"
)

print(
    "Feedback               : Enabled"
)

print()
print(
    "Mean Cipher Entropy    : "
    f"{format_metric(mean_cipher_entropy)}"
)

print(
    "Mean NPCR              : "
    f"{format_metric(mean_npcr)} %"
)

print(
    "Mean UACI              : "
    f"{format_metric(mean_uaci)} %"
)

print(
    "Mean Corr-H            : "
    f"{format_metric(mean_corr_h)}"
)

print(
    "Mean Corr-V            : "
    f"{format_metric(mean_corr_v)}"
)

print(
    "Mean Corr-D            : "
    f"{format_metric(mean_corr_d)}"
)

print()
print(
    "Mean Encryption Time   : "
    f"{format_metric(mean_enc_time)} s"
)

print(
    "Mean Decryption Time   : "
    f"{format_metric(mean_dec_time)} s"
)

print(
    "Mean Encryption Rate   : "
    f"{format_metric(mean_enc_throughput, 2)} "
    "pixels/s"
)

print(
    "Mean Decryption Rate   : "
    f"{format_metric(mean_dec_throughput, 2)} "
    "pixels/s"
)

print()
print(
    "Exact Reconstruction   : "
    f"{exact_count}/{total_images}"
)

print()
print("=" * 80)
print(
    "All final outputs have been written to:"
)
print(
    RESULT_ROOT
)
print("=" * 80)

print()
print(
    f"Final summary saved to: "
    f"{FINAL_SUMMARY_JSON}"
)


PROPOSED METHOD EXPERIMENT SUMMARY

Images processed       : 49
Image size             : 256 × 256
Image mode             : Grayscale
Block size             : 16 × 16
DNA rules              : 8
Hash                   : SHA-256
Chaotic map            : Logistic
Feedback               : Enabled

Mean Cipher Entropy    : 7.997175
Mean NPCR              : 99.613392 %
Mean UACI              : 33.479082 %
Mean Corr-H            : -0.000704
Mean Corr-V            : -0.000786
Mean Corr-D            : -0.001014

Mean Encryption Time   : 0.866953 s
Mean Decryption Time   : 0.866106 s
Mean Encryption Rate   : 75593.51 pixels/s
Mean Decryption Rate   : 75667.40 pixels/s

Exact Reconstruction   : 49/49

All final outputs have been written to:
/content/drive/MyDrive/DNA Adaptive RESULT

Final summary saved to: /content/drive/MyDrive/DNA Adaptive RESULT/11_SUMMARY/final_research_summary.json
